# Experiment 1: radius/similarity sensitivity

Reproduce the synthetic **ARI and AMI** curves for CLASSIX_T, DBSCAN, Taylor–Butina, BitBIRCH and BitBIRCH-Lean. This notebook generates the data, runs every clustering, verifies saved assignments, and renders the figure. It contains **only source code and experimental design**: no saved observations, labels, data, binaries or archives. No other notebook or scratch folder is required.

Use Windows x64, Python 3.12, Visual Studio C++ Build Tools and the Windows SDK. Setup creates an isolated environment and compiles the embedded C kernels. Package installation requires internet access. `RUN_MODE="full"` runs all **30 method/radius fits**; `"pilot"` is a small functional check, and `"check"` validates imports/design without fitting. Generated inputs, checkpoints, packages and records stay outside this notebook folder. Only a complete full run publishes figures under `plots/synthetic`.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys

RUN_MODE = "full"                         # "full", "pilot", or "check"
_candidates = [Path.cwd(), Path.cwd() / "experiment_1",
               Path.cwd() / "experiments" / "experiment_1",
               Path.cwd() / "classix_tanimoto_manhattan" / "experiments" / "experiment_1"]
NOTEBOOK_DIR = next((p.resolve() for p in _candidates
                     if (p / "03_radius_sensitivity.ipynb").is_file()), None)
if NOTEBOOK_DIR is None:
    raise RuntimeError("Start the kernel in this notebook folder or the repository root.")
WORK_DIR = NOTEBOOK_DIR.parent / "_experiment_1_work" / "radius_sensitivity_n10000_d1024"
PLOT_DIR = NOTEBOOK_DIR / "plots" / "synthetic"
PILOT_ROWS = 256
INSTALL_DEPENDENCIES = True
PYTHON_FOR_RUNS = None                    # optional existing Windows x64 Python 3.12 executable

assert RUN_MODE in {"full", "pilot", "check"}
WORK_DIR = WORK_DIR.expanduser().resolve()
assert WORK_DIR != NOTEBOOK_DIR and not WORK_DIR.is_relative_to(NOTEBOOK_DIR),     "Choose a work directory outside the notebook folder."
SOURCES = {}


## Fixed design

All six settings use the same generated and shuffled **10,000 × 1,024** binary dataset: 10 clusters of 1,000 points, seed 101, expected centre popcounts linearly spaced from 50 to 300, and the original `generate_data` function with `flip_prob=0.3`. For a centre with expected popcount `p`, each coordinate flips with probability `0.3*p/1024`; the realized centre popcounts are random. The shuffle continues the generator's seeded NumPy stream.

Tanimoto distance radius `r` takes **0.3, 0.4, 0.5, 0.6, 0.7, 0.8**. CLASSIX uses packed input, `minPts=1` and `mergeScale=1/sqrt(r)`; DBSCAN uses Jaccard distance, `eps=r`, `min_samples=1` and one thread; RDKit Taylor–Butina uses distance cutoff `r` and `reordering=False`. Both BitBIRCH versions use similarity threshold `1-r`, branching factor 50 and diameter merging. These numeric conversions do not imply identical merge rules. Original BitBIRCH receives int64 input; CLASSIX and Lean receive appropriately packed representations of the same bits.

There is one full fit per method/radius and no warmup: **this is a clustering-quality study, not a repeated runtime comparison**. ARI and AMI use all true labels and all assignments, without filtering; AMI uses the arithmetic normalizer. Any recorded noise remains its assigned category. Higher ARI/AMI is better. Fits use fresh processes and a 20 GiB private-memory limit, without a fit time cutoff. Source hashes, package versions, platform, generated-array hashes and assignment checks are written into the work directory.


## Frozen source and design

These readable source cells are the implementation used by the study. Byte hashes and line-ending metadata retain its exact source identity when materialized. CLASSIX's MIT license and BitBIRCH's LGPL-3.0 license accompany their sources. BitBIRCH is pinned to upstream commit `311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f`; its original module is loaded directly, avoiding incompatible upstream package constraints. BBLean 0.10.3 is installed from the hash-verified native Windows wheel below.


**`plan.json`**


In [ ]:
SOURCES['plan.json'] = r'''{
  "cases": [
    {
      "case_id": "radius_0.3_n10000_d1024",
      "study": "radius",
      "x": 0.3,
      "data_id": "49e1cc227160a986",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          300
        ],
        "d": 1024,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1024,
      "classix": {
        "radius": 0.3,
        "minPts": 1,
        "mergeScale": 1.8257418583505538
      },
      "bitbirch": {
        "threshold": 0.7,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.3,
        "min_samples": 1,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.3,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0.4_n10000_d1024",
      "study": "radius",
      "x": 0.4,
      "data_id": "49e1cc227160a986",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          300
        ],
        "d": 1024,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1024,
      "classix": {
        "radius": 0.4,
        "minPts": 1,
        "mergeScale": 1.5811388300841895
      },
      "bitbirch": {
        "threshold": 0.6,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.4,
        "min_samples": 1,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.4,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0.5_n10000_d1024",
      "study": "radius",
      "x": 0.5,
      "data_id": "49e1cc227160a986",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          300
        ],
        "d": 1024,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1024,
      "classix": {
        "radius": 0.5,
        "minPts": 1,
        "mergeScale": 1.414213562373095
      },
      "bitbirch": {
        "threshold": 0.5,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.5,
        "min_samples": 1,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.5,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0.6_n10000_d1024",
      "study": "radius",
      "x": 0.6,
      "data_id": "49e1cc227160a986",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          300
        ],
        "d": 1024,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1024,
      "classix": {
        "radius": 0.6,
        "minPts": 1,
        "mergeScale": 1.2909944487358056
      },
      "bitbirch": {
        "threshold": 0.4,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.6,
        "min_samples": 1,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.6,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0.7_n10000_d1024",
      "study": "radius",
      "x": 0.7,
      "data_id": "49e1cc227160a986",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          300
        ],
        "d": 1024,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1024,
      "classix": {
        "radius": 0.7,
        "minPts": 1,
        "mergeScale": 1.1952286093343936
      },
      "bitbirch": {
        "threshold": 0.3,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.7,
        "min_samples": 1,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.7,
        "reordering": false
      }
    },
    {
      "case_id": "radius_0.8_n10000_d1024",
      "study": "radius",
      "x": 0.8,
      "data_id": "49e1cc227160a986",
      "generator": {
        "num_clusters": 10,
        "pops": [
          50,
          300
        ],
        "d": 1024,
        "n": 1000,
        "flip_prob": 0.3,
        "seed": 101
      },
      "rows": 10000,
      "features": 1024,
      "classix": {
        "radius": 0.8,
        "minPts": 1,
        "mergeScale": 1.118033988749895
      },
      "bitbirch": {
        "threshold": 0.2,
        "branching_factor": 50,
        "merge_criterion": "diameter"
      },
      "dbscan": {
        "eps": 0.8,
        "min_samples": 1,
        "metric": "jaccard",
        "n_jobs": 1
      },
      "butina": {
        "cutoff": 0.8,
        "reordering": false
      }
    }
  ],
  "case_count": 6,
  "unique_datasets": 1,
  "methods": [
    "dbscan",
    "classix_packed",
    "bitbirch",
    "bblean",
    "butina"
  ],
  "measured_repeats": 1,
  "discarded_warmups": 0,
  "fit_timeout_seconds": null,
  "job_memory_limit_bytes": 21474836480,
  "source_notebook": "C:\\CLASSIX_Tanimoto_Manhattan_Paper\\classix_tanimoto_manhattan_code_repo\\experiments\\classix_t_blobs.ipynb",
  "source_notebook_sha256": "ad1bcf5778b07e442ee6ff1c294ab32fcd60828d2b6af4e949a0335ce6da3848",
  "interpretations": [
    "Same newly generated and shuffled seed-101 dataset at every radius: 10 clusters,1000 rows each,d1024,pops endpoints50 and300,flip_prob0.3.",
    "Radius grid0.3,0.4,0.5,0.6,0.7,0.8; BitBIRCH/Lean similarity=1-radius.",
    "CLASSIXminPts1,mergeScale1/sqrt(radius); DBSCANmin_samples1 explicitly confirmed by author.",
    "One fit per method/radius for deterministic quality curves; no runtime comparison or repetitions claimed.",
    "ARI and AMI use all10000 true labels, without filtering; no noise is expected with min_samples1.",
    "Branching50 and diameter criterion retained for both BitBIRCH versions; Taylor-Butina reordering disabled."
  ],
  "adjacency_storage": "One packed connection matrix plus sparse directed minPts reassignment markers; logical values 0/1/2.",
  "excluded_radius": {
    "radius": 0.1,
    "reason": "Author requested skipping this near-singleton aggregation case; initial dense-graph evidence is preserved separately."
  },
  "radius_sweep_similarity_threshold_range": [
    0.2,
    0.7
  ],
  "reuse_policy": "All seed-101 fits are new. No seed-0 timings or quality results reused.",
  "seed": 101,
  "butina_implementation": "standard_rdkit",
  "protocol_status": "radius_quality_n10000_d1024",
  "preparation_excluded": [
    "dataset generation",
    "loading",
    "conversion to each native input representation"
  ],
  "timed_operations": "estimator construction, all distance/neighborhood/tree/group construction, fit, and label assignment",
  "frozen_utc": "2026-09-30",
  "resource_choice_basis": "Author explicitly accepts long runs. The initial 20k-point DBSCAN fit took about five minutes, showing that a one-hour cutoff could censor feasible larger fits. The final suite has no runtime cutoff; the 20-GiB private-commit cap protects the 32-GiB host.",
  "generator_ast_matches_notebook": true,
  "butina_timing_note": "RDKit ExplicitBitVect construction from exact generated packed bits is input representation preparation. It is excluded, like input packing for CLASSIX/Lean. Full Python distance-list construction, unchanged RDKit clustering and assignment are timed. No Morgan embedding applies to synthetic data.",
  "results_directory": "runs/full/results",
  "initial_capped_exploration": {
    "plan": "C:\\CLASSIX_Tanimoto_Manhattan_Paper\\scratch\\final_benchmarks_seed101_20260921\\initial_capped_plan.json",
    "plan_sha256": "fe2df9bcbc6b3e85d8a399236d33eb7d480b3e7572a95e6043ebf752dd291fe4",
    "results": "C:\\CLASSIX_Tanimoto_Manhattan_Paper\\scratch\\final_benchmarks_seed101_20260921\\results",
    "use": "Preliminary evidence only. All final fits are rerun without a runtime cutoff; no capped-run timings reused."
  },
  "source_plan_sha256": "2f7689d62501a5db1fcd023f6286f8b2181d6a69e93384c48bfb317d06d4137f"
}
'''


**`records/repetition_amendment.json`**


In [ ]:
SOURCES['records/repetition_amendment.json'] = r'''{
  "description": "One measured full fit per method and radius; no warmup, quality-only study.",
  "schedules": {
    "radius_0.3_n10000_d1024": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        0
      ],
      "bitbirch": [
        0
      ],
      "bblean": [
        0
      ],
      "butina": [
        0
      ]
    },
    "radius_0.4_n10000_d1024": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        0
      ],
      "bitbirch": [
        0
      ],
      "bblean": [
        0
      ],
      "butina": [
        0
      ]
    },
    "radius_0.5_n10000_d1024": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        0
      ],
      "bitbirch": [
        0
      ],
      "bblean": [
        0
      ],
      "butina": [
        0
      ]
    },
    "radius_0.6_n10000_d1024": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        0
      ],
      "bitbirch": [
        0
      ],
      "bblean": [
        0
      ],
      "butina": [
        0
      ]
    },
    "radius_0.7_n10000_d1024": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        0
      ],
      "bitbirch": [
        0
      ],
      "bblean": [
        0
      ],
      "butina": [
        0
      ]
    },
    "radius_0.8_n10000_d1024": {
      "dbscan": [
        0
      ],
      "classix_packed": [
        0
      ],
      "bitbirch": [
        0
      ],
      "bblean": [
        0
      ],
      "butina": [
        0
      ]
    }
  }
}
'''


**`score_and_plot.py`**


In [ ]:
SOURCES['score_and_plot.py'] = r'''"""Verify saved radius fits and plot ARI/AMI; never fit clustering estimators."""
from pathlib import Path
import argparse
import hashlib
import json
import sys

HERE = Path(__file__).resolve().parent
sys.dont_write_bytecode = True
sys.path.insert(0, str(HERE / 'code'))
METHODS = {
    'classix_packed': ('CLASSIX_T', 'red', 'x', '--'),
    'dbscan': ('DBSCAN', 'blue', 'o', '-.'),
    'bitbirch': ('BitBIRCH', 'green', 'v', ':'),
    'bblean': ('BitBIRCH-Lean', '#9467bd', 's', '--'),
    'butina': ('Taylor-Butina', '#e68613', 'D', '-'),
}
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def read(path): return json.loads(path.read_text(encoding='utf-8-sig'))

def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--run-dir', type=Path, default=HERE / 'runs/full')
    parser.add_argument('--output-dir', type=Path, default=HERE)
    args = parser.parse_args()
    run_dir, output = args.run_dir.resolve(), args.output_dir.resolve()
    assert not (run_dir / 'run.lock').exists(), 'Wait for the fitting process to finish.'
    import numpy as np
    import sklearn
    from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
    from state_checks import array_hash
    from prepare_dataset import digest
    plan = read(HERE / 'plan.json')
    cases = sorted(plan['cases'], key=lambda case: case['x'])
    assert [case['x'] for case in cases] == [.3, .4, .5, .6, .7, .8]
    assert len({case['data_id'] for case in cases}) == 1
    generator = dict(num_clusters=10, pops=[50, 300], d=1024, n=1000, flip_prob=.3, seed=101)
    assert all(case['generator'] == generator for case in cases)
    directory = run_dir / 'data' / cases[0]['data_id']
    manifest = read(directory / 'manifest.json')
    assert manifest['shape'] == [10000, 1024] and manifest['parameters'] == generator
    for item in manifest['files'].values():
        array = np.load(directory / item['file'], mmap_mode='r', allow_pickle=False)
        assert digest(array) == item['sha256']
    truth = np.load(directory / 'truth.npy', allow_pickle=False)
    assert np.unique(truth, return_counts=True)[1].tolist() == [1000] * 10
    rows, evidence = [], []
    for case in cases:
        path = run_dir / 'results' / (case['case_id'] + '.json')
        raw = read(path)
        assert raw['completed'] and raw['pilot_rows'] is None and raw['case'] == case
        assert len(raw['runs']) == 5 and {run['variant'] for run in raw['runs']} == set(METHODS)
        for run in raw['runs']:
            assert run['status'] == 'ok' and not run['warmup'] and run['repeat'] == 0
            checked = run['validated']
            assert checked['logical_input_sha256'] == manifest['files']['int32']['sha256']
            assert run['ready']['rows'] == 10000 and run['ready']['features'] == 1024
            labels = np.load(run_dir / checked['output_relative_assignments_file'], allow_pickle=False)
            assert labels.shape == truth.shape and np.all(labels >= 0)
            mapping = {}
            canonical = np.array([mapping.setdefault(int(label), len(mapping)) for label in labels], dtype=np.int64)
            assert array_hash(canonical) == checked['canonical_partition_sha256']
            ari = float(adjusted_rand_score(truth, labels))
            ami = float(adjusted_mutual_info_score(truth, labels, average_method='arithmetic'))
            assert np.isclose(ari, checked['ari'], rtol=0, atol=1e-12)
            assert np.isfinite(ami)
            rows.append(dict(radius=case['x'], similarity=round(1-case['x'], 10), method=run['variant'],
                             ari=ari, ami=ami, clusters=int(len(np.unique(labels))), noise_points=0,
                             scored_rows=len(truth), seconds=run['fit_done']['clustering_seconds'],
                             canonical_partition_sha256=checked['canonical_partition_sha256'],
                             assignments_sha256=sha(run_dir / checked['output_relative_assignments_file'])))
        evidence.append(dict(path=str(path), sha256=sha(path)))
    assert len(rows) == 30
    output.mkdir(parents=True, exist_ok=True)
    result = dict(status='passed', generator=generator, radius_grid=[case['x'] for case in cases],
                  plan_sha256=sha(HERE / 'plan.json'), input_sha256=manifest['files']['int32']['sha256'],
                  sklearn_version=sklearn.__version__, ami_average_method='arithmetic',
                  all_rows_scored=True, run_records=evidence, records=rows)
    (output / 'scores.json').write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                        'axes.titlesize': 10, 'axes.labelsize': 9,
                        'xtick.labelsize': 9, 'ytick.labelsize': 9,
                        'axes.grid': False, 'svg.fonttype': 'none', 'pdf.fonttype': 42})
    fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.3), sharex=True, sharey=True)
    fig.subplots_adjust(left=.075, right=.98, bottom=.18, top=.73, wspace=.15)
    handles = []
    for method, (label, color, marker, linestyle) in METHODS.items():
        selected = sorted((row for row in rows if row['method'] == method), key=lambda row: row['radius'])
        x = [row['radius'] for row in selected]
        for ax, metric in zip(axes, ('ari', 'ami')):
            # Open markers help show methods whose quality scores coincide.
            line, = ax.plot(x, [row[metric] for row in selected], label=label,
                            color=color, marker=marker, linestyle=linestyle,
                            linewidth=1.4, markersize=5,
                            markerfacecolor='none',
                            markeredgewidth=1.1)
            np.testing.assert_array_equal(line.get_ydata(), [row[metric] for row in selected])
        handles.append(Line2D([0], [0], label=label, color=color, marker=marker, linestyle=linestyle,
                              linewidth=1.4, markersize=5,
                              markerfacecolor='none'))
    for ax, metric in zip(axes, ('ARI', 'AMI')):
        ax.set_title(metric, pad=5)
        ax.set_xlabel('Distance threshold r')
        ax.set_xlim(.28, .82)
        ax.set_xticks([.3, .4, .5, .6, .7, .8])
        ax.set_ylim(-.04, 1.04)
        ax.set_yticks([0, .25, .5, .75, 1])
    fig.legend(handles=handles, loc='upper left', bbox_to_anchor=(.065, .995),
               ncol=3, fontsize=8, frameon=False, columnspacing=1.7, handlelength=2.4)
    assert not fig.texts
    plots = output / 'plots'
    plots.mkdir(exist_ok=True)
    artifacts = []
    for extension in ('pdf', 'png', 'svg'):
        path = plots / ('radius_ari_ami.' + extension)
        fig.savefig(path, dpi=300, facecolor='white')
        artifacts.append(dict(path=str(path), sha256=sha(path)))
    plt.close(fig)
    pairs = []
    for radius in result['radius_grid']:
        selected = {row['method']: row for row in rows if row['radius'] == radius}
        pairs.append(dict(radius=radius, identical_bitbirch_partitions=
                          selected['bitbirch']['canonical_partition_sha256'] == selected['bblean']['canonical_partition_sha256']))
    verification = dict(status='passed', full_method_settings=30, new_clustering_fits_in_this_script=0,
                        scored_rows_per_fit=10000, all_assignment_and_input_hashes_verified=True,
                        ari_independently_verified=True, ami_computed_from_saved_assignments=True,
                        all_methods_complete=True, score_sha256=sha(output / 'scores.json'),
                        bitbirch_comparison=pairs, artifacts=artifacts)
    (output / 'verification.json').write_text(json.dumps(verification, indent=2) + '\n', encoding='utf-8')
    lines = ['| Radius | Similarity | Method | ARI | AMI | Clusters |', '|---|---|---|---:|---:|---:|']
    lines.extend(f"| {row['radius']:.1f} | {row['similarity']:.1f} | {METHODS[row['method']][0]} | {row['ari']:.4f} | {row['ami']:.4f} | {row['clusters']} |"
                 for row in rows)
    (output / 'scores.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    print(json.dumps({'status': 'passed', 'method_settings': len(rows), 'plot': str(plots / 'radius_ari_ami.pdf')}, indent=2))

if __name__ == '__main__':
    main()
'''


**`code/generate_data_snapshot.py`**


In [ ]:
SOURCES['code/generate_data_snapshot.py'] = r'''import numpy as np
import random

def generate_data(num_clusters, pops, d, n, flip_prob=0.1, seed=None):
    """
    Generate data for the CLASSIX_T dataset

        Parameters: \n
        num_clusters (int)  : number of clusters \n
        pops (list)         : the popcount of the different cluster centers. len(pops) must be equal to num_clusters \n
        d (int)             : dimension of the data \n
        n (int)             : number of total samples per cluster \n
        seed (int)          : seed for the random number generator \n


    """
    assert len(pops)>1, "You must provide at least 2 cluster mean populations in order to generate meaningful data"
    if len(pops) != num_clusters:
        try:
            print("The number of mean populations is not equal to the number of clusters. Trying to generate mean populations keeping in mind the minimum and maximum pop counts provided by the user")
            pops = np.linspace(min(pops), max(pops), num_clusters)
        except:
            raise ValueError("The number of cluster centers must be equal to the number of cluster populations")
    
    size_flag = [pops[x]<d for x in range(len(pops))]
    
    assert all(size_flag), "The number of dimensions must be greater than the pop counts of the clusters"
    
    clustc = []
    data = []
    if seed != None:
        np.random.seed(seed)
        random.seed(seed)

    labels = []

    for i in range(num_clusters):
        pop = pops[i]
        p1 = pop/d
        clustc.append(np.random.choice([0, 1], size=d, p=[1-p1, p1]))
        data.append(clustc[-1])
        labels.append(i)
        for j in range(n-1):
            labels.append(i)
            # Instead of flipping exactly pop*flip_prob bits, we flip each bit with a probability of flip_prob.
            flip_indices = np.random.choice([0, 1], size = d, p = [1-flip_prob*p1, flip_prob*p1])
            new_sample = np.logical_xor(clustc[-1], flip_indices).astype(np.int32)
            data.append(new_sample)
    return np.array(data).astype(np.int32), np.array(labels).astype(np.int32)
'''


**`code/prepare_dataset.py`**


In [ ]:
SOURCES['code/prepare_dataset.py'] = r'''"""Generate one shared dataset from the notebook function, outside fit workers."""
import argparse
import gc
import hashlib
import json
from pathlib import Path
import time
import numpy as np
from generate_data_snapshot import generate_data

HERE = Path(__file__).resolve().parent
ROOT = HERE.parent


def digest(array):
    result = hashlib.sha256()
    row_bytes = array.dtype.itemsize * int(np.prod(array.shape[1:], dtype=np.int64))
    rows = max(1, (8 * 1024 * 1024) // max(1, row_bytes))
    for start in range(0, len(array), rows):
        result.update(memoryview(np.ascontiguousarray(array[start:start + rows])).cast("B"))
    return result.hexdigest()


def require(condition, message):
    if not condition:
        raise ValueError(message)


def verify_recorded_data(directory, metadata, reference, case):
    """Check fresh or cached arrays against the frozen raw-array hashes."""
    require(metadata["data_id"] == reference["data_id"] == case["data_id"], "Dataset identity differs from recorded data")
    require(metadata["parameters"] == reference["parameters"] == case["generator"], "Generator parameters differ from recorded data")
    require(metadata["shape"] == reference["shape"] == [case["rows"], case["features"]], "Dataset shape differs from recorded data")
    require(metadata["generator_snapshot_sha256"] == reference["generator_snapshot_sha256"]
            == hashlib.sha256((HERE/"generate_data_snapshot.py").read_bytes()).hexdigest(),
            "Generator source differs from the recorded snapshot")
    require(metadata["source_notebook_sha256"] == reference["source_notebook_sha256"], "Dataset source provenance differs")
    names = {"int32", "int64", "packed_little", "packed_big", "truth", "permutation", "centers"}
    require(set(metadata["files"]) == set(reference["files"]) == names, "Dataset file inventory differs")
    n, d, k = case["rows"], case["features"], case["generator"]["num_clusters"]
    shapes = {"int32": (n, d), "int64": (n, d), "packed_little": (n, (d+7)//8),
              "packed_big": (n, (d+7)//8), "truth": (n,), "permutation": (n,), "centers": (k, d)}
    for name in sorted(names):
        record, expected = metadata["files"][name], reference["files"][name]
        require(all(record[key] == expected[key] for key in ("file", "dtype", "bytes", "sha256")),
                f"Manifest differs from recorded array: {name}")
        require(Path(record["file"]).name == record["file"], "Array filename must be local")
        array = np.load(directory/record["file"], allow_pickle=False, mmap_mode="r")
        require(array.shape == shapes[name] and str(array.dtype) == expected["dtype"]
                and array.nbytes == expected["bytes"], f"Actual array shape/dtype/bytes differ: {name}")
        require(digest(array) == expected["sha256"], f"Actual raw-array SHA256 differs from recorded data: {name}")
        del array
    return len(names)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("data_id")
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--plan", type=Path, default=ROOT/"plan.json")
    args = parser.parse_args()
    plan = json.loads(args.plan.read_text(encoding="utf-8-sig"))
    case = next(c for c in plan["cases"] if c["data_id"] == args.data_id)
    # New 10k datasets have no historical 25k reference hashes.
    directory = args.data_dir.resolve() / args.data_id
    if not directory.is_relative_to((ROOT/"runs").resolve()):
        raise ValueError("Generated data must stay under experiment_1A/runs")
    directory.mkdir(parents=True, exist_ok=True)
    manifest_path = directory/"manifest.json"
    if manifest_path.exists():
        prior = json.loads(manifest_path.read_text())
        verified = verify_recorded_data(directory, prior, prior, case)
        print(json.dumps({"reusing": args.data_id, "fresh_raw_array_hashes_verified": verified}), flush=True)
        return
    params = case["generator"]
    start = time.perf_counter()
    data, truth = generate_data(**params)
    centers = data[np.arange(params["num_clusters"])*params["n"]].copy()
    permutation = np.arange(len(data))
    np.random.shuffle(permutation)
    data, truth = np.ascontiguousarray(data[permutation]), truth[permutation]
    assert data.shape == (case["rows"], case["features"])
    generation_seconds = time.perf_counter()-start
    files = {}
    def save(name, array):
        np.save(directory/f"{name}.npy", array)
        files[name] = {"file": name+".npy", "bytes": array.nbytes,
                       "dtype": str(array.dtype), "sha256": digest(array)}
    save("int32", data)
    wide = data.astype(np.int64)
    save("int64", wide)
    del wide
    gc.collect()
    for order in ("little", "big"):
        packed = np.packbits(data, axis=1, bitorder=order)
        save("packed_"+order, packed)
    save("truth", truth)
    save("permutation", permutation)
    save("centers", centers)
    row_pops = data.sum(axis=1)
    metadata = {"data_id": args.data_id, "parameters": params, "shape": list(data.shape),
        "files": files, "zero_fraction": float(1-data.mean()), "zero_rows": int(np.sum(row_pops == 0)),
        "minimum_popcount": int(row_pops.min()), "maximum_popcount": int(row_pops.max()),
        "center_popcounts": centers.sum(axis=1).tolist(), "generation_seconds": generation_seconds,
        "source_notebook_sha256": plan["source_notebook_sha256"],
        "generator_snapshot_sha256": hashlib.sha256((HERE/"generate_data_snapshot.py").read_bytes()).hexdigest(),
        "shuffle": "np.random.shuffle after generator, continuing its seeded NumPy RNG"}
    verified = verify_recorded_data(directory, metadata, metadata, case)
    temporary = manifest_path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(metadata, indent=2)+"\n")
    temporary.replace(manifest_path)
    print(json.dumps({"generated": args.data_id, "shape": metadata["shape"], "zeros": metadata["zero_fraction"],
                      "fresh_raw_array_hashes_verified": verified}), flush=True)


if __name__ == "__main__":
    main()
'''


**`code/worker.py`**


In [ ]:
SOURCES['code/worker.py'] = r'''"""One fit with explicit pre-fit/post-fit checkpoints and separate validation."""
import os
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = "1"
import argparse
import gc
import json
from pathlib import Path
import sys
import traceback
from time import perf_counter
sys.dont_write_bytecode = True
HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
BASE = ROOT
PACKED = HERE / "classix"
from memory_win import snapshot


def send(event, **values):
    print("BENCH_JSON " + json.dumps({"event": event, **values}), flush=True)


def handshake(expected):
    received = sys.stdin.readline().strip()
    if received != expected:
        raise RuntimeError(f"Expected {expected}, received {received}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--case", required=True)
    parser.add_argument("--variant", required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--rows", type=int)
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--plan", type=Path, default=ROOT/"plan.json")
    args = parser.parse_args()
    import numpy as np
    from threadpoolctl import threadpool_limits, threadpool_info
    plan = json.loads(args.plan.read_text(encoding="utf-8-sig"))
    case = next(c for c in plan["cases"] if c["case_id"] == args.case)
    directory = args.data_dir/case["data_id"]
    manifest = json.loads((directory/"manifest.json").read_text())
    parameters = {}
    library = {}
    if args.variant.startswith("classix"):
        sys.path.insert(0, str(PACKED))
        if args.variant == "classix":
            import classix_t_reference_adjbits as module
            estimator = module.CLASSIX_T
            kind = "int32"
        else:
            import classix_t_packed_adjbits as module
            estimator = module.CLASSIX_T_PACKED
            kind = "packed_little"
        module.print = lambda *a, **k: None
        module.tqdm = lambda iterable, *a, **k: iterable
        parameters = case["classix"]
    elif args.variant == "bitbirch":
        sys.path.insert(0, str(BASE/"vendor"/"bitbirch"))
        from bitbirch import bitbirch as module
        module.set_merge("diameter")
        estimator = module.BitBirch
        parameters = {k: case["bitbirch"][k] for k in ("threshold", "branching_factor")}
        kind = "int64"
        library = {"module": module.__file__, "merge_criterion": "diameter"}
    elif args.variant == "bblean":
        import bblean
        import bblean.similarity as similarity
        assert similarity._jt_sim_arr_vec_packed.__module__ == "bblean._cpp_similarity"
        estimator = bblean.BitBirch
        parameters = case["bitbirch"]
        kind = "packed_big"
        library = {"version": bblean.__version__, "native_kernel": similarity._jt_sim_arr_vec_packed.__module__,
                   "logical_features": case["features"], "native_features": 8*((case["features"]+7)//8),
                   "zero_padding_bits": (-case["features"]) % 8}
    elif args.variant == "dbscan":
        import sklearn
        from sklearn.cluster import DBSCAN
        estimator = DBSCAN
        parameters = case["dbscan"]
        kind = "int32"
        library = {"version": sklearn.__version__, "algorithm": "auto", "metric": "jaccard",
                   "input_note": "Notebook int32 input; sklearn performs its own boolean conversion inside fit"}
    elif args.variant == "butina":
        import rdkit
        from rdkit import DataStructs
        from rdkit.ML.Cluster import Butina
        parameters = case["butina"]
        kind = "packed_little"
        library = {"version": rdkit.__version__, "implementation": plan["butina_implementation"],
                   "module": Butina.__file__, "logical_features": case["features"],
                   "native_features": 8*((case["features"]+7)//8),
                   "zero_padding_bits": (-case["features"]) % 8}
    else:
        raise ValueError(args.variant)
    with threadpool_limits(limits=1):
        pool_info = threadpool_info()
        gc.collect()
        after_import = snapshot()
        data = np.load(directory/manifest["files"][kind]["file"], allow_pickle=False)
        if args.rows is not None:
            data = data[:args.rows].copy()
        representation_started = perf_counter()
        fingerprints = None
        if args.variant == "butina":
            fingerprints = [DataStructs.CreateFromBinaryText(row.tobytes()) for row in data]
        representation_seconds = perf_counter() - representation_started
        gc.collect()
        before = snapshot()
        send("ready", pid=os.getpid(), variant=args.variant, case=args.case, rows=len(data), features=case["features"],
             input_kind=kind, input_bytes=data.nbytes, parameters=parameters, library=library,
             representation_seconds=representation_seconds,
             threadpools=pool_info, after_import=after_import, before_fit=before)
        handshake("GO")
        started = perf_counter()
        try:
            model = None if args.variant == "butina" else estimator(**parameters)
            if args.variant == "butina":
                distances = []
                for i in range(1, len(fingerprints)):
                    similarities = DataStructs.BulkTanimotoSimilarity(fingerprints[i], fingerprints[:i])
                    distances.extend([1 - value for value in similarities])
                clusters = Butina.ClusterData(distances, len(fingerprints), parameters["cutoff"],
                                               isDistData=True, reordering=parameters["reordering"])
                fit_end = perf_counter()
                labels = np.full(len(data), -1, dtype=np.int64)
                for cluster_id, members in enumerate(clusters):
                    labels[list(members)] = cluster_id
            elif args.variant == "dbscan":
                model.fit(data)
                fit_end = perf_counter()
                labels = model.labels_
            elif args.variant == "bblean":
                # Native 0.10.3 unpacking requires a byte-aligned dimension.
                # Existing packbits padding is zero and preserves similarity.
                model.fit(data, input_is_packed=True, n_features=8*data.shape[1])
                fit_end = perf_counter()
                labels = model.get_assignments()
            elif args.variant == "bitbirch":
                model.fit(data)
                fit_end = perf_counter()
                labels = model.get_assignments(len(data))
            else:
                model.fit(data)
                fit_end = perf_counter()
                labels = model.labels
            ended = perf_counter()
        except Exception as error:
            send("fit_error", error_type=type(error).__name__, message=str(error),
                 elapsed_seconds=perf_counter()-started, memory=snapshot(), traceback=traceback.format_exc())
            return
        after = snapshot()
        send("fit_done", clustering_seconds=ended-started, construction_and_fit_seconds=fit_end-started,
             assignment_seconds=ended-fit_end, after_fit=after)
        handshake("VALIDATE")
        try:
            from quality import adjusted_rand_index
            from state_checks import array_hash, logical_hash, state_fingerprint
            labels = np.asarray(labels, dtype=np.int64)
            assert labels.shape == (len(data),) and np.all(labels >= (-1 if args.variant == "dbscan" else 0))
            noise_points = int(np.sum(labels == -1))
            ids, counts = np.unique(labels[labels >= 0], return_counts=True)
            truth = np.load(directory/manifest["files"]["truth"]["file"], allow_pickle=False)[:len(data)]
            ari = adjusted_rand_index(truth, labels)
            mapping = {}
            canonical = np.empty_like(labels)
            for i, value in enumerate(labels):
                canonical[i] = mapping.setdefault(int(value), len(mapping))
            args.output.parent.mkdir(parents=True, exist_ok=True)
            np.save(args.output, labels)
            bitorder = "little" if kind == "packed_little" else "big" if kind == "packed_big" else None
            logical = logical_hash(data, case["features"], bitorder)
            if args.rows is None:
                assert logical == manifest["files"]["int32"]["sha256"], "Input changed"
            state_file = None
            if args.variant.startswith("classix"):
                state_file = args.output.with_suffix(".state.json")
                state_file.write_text(json.dumps(state_fingerprint(model, case["features"], args.variant == "classix_packed"), indent=2)+"\n")
            send("validated", ari=ari, clusters=len(ids), noise_points=noise_points,
                 singletons=int(np.sum(counts == 1)), largest_cluster=int(counts.max()) if len(counts) else 0,
                 aggregation_groups=len(model.splist) if args.variant.startswith("classix") else None,
                 adjacency_bytes=model.Adj.nbytes if args.variant.startswith("classix") else None,
                 adjacency_storage="connection bits + sparse minPts markers" if args.variant.startswith("classix") else None,
                 minpts_marker_count=model.Adj.marker_count if args.variant.startswith("classix") else None,
                 minpts_marker_storage_bytes=model.Adj.marker_storage_bytes if args.variant.startswith("classix") else None,
                 state_file=str(state_file) if state_file else None,
                 canonical_partition_sha256=array_hash(canonical), logical_input_sha256=logical,
                 input_unchanged=True if args.rows is None else None, assignments_file=str(args.output))
        except Exception as error:
            send("validation_error", error_type=type(error).__name__, message=str(error), traceback=traceback.format_exc())


if __name__ == "__main__":
    main()
'''


**`code/quality.py`**


In [ ]:
SOURCES['code/quality.py'] = r'''"""Adjusted Rand index from the contingency table, using Python integers."""
import numpy as np


def adjusted_rand_index(truth, labels):
    _, a = np.unique(truth, return_inverse=True)
    _, b = np.unique(labels, return_inverse=True)
    table = np.zeros((int(a.max())+1, int(b.max())+1), dtype=np.int64)
    np.add.at(table, (a, b), 1)
    def pairs(counts):
        return sum(int(x)*(int(x)-1)//2 for x in counts.flat)
    total = len(a)*(len(a)-1)//2
    cells, rows, cols = pairs(table), pairs(table.sum(axis=1)), pairs(table.sum(axis=0))
    denominator = total*(rows+cols)-2*rows*cols
    return 1.0 if denominator == 0 else 2*(total*cells-rows*cols)/denominator
'''


**`code/state_checks.py`**


In [ ]:
SOURCES['code/state_checks.py'] = r'''"""Validation after measurement, using bounded temporary buffers for hashes."""
import hashlib
import json
import numpy as np


def array_hash(array):
    array = np.ascontiguousarray(array)
    return hashlib.sha256(memoryview(array).cast("B")).hexdigest()


def logical_hash(data, n_features, packing=None):
    result = hashlib.sha256()
    for start in range(0, len(data), 256):
        block = data[start:start+256]
        if packing:
            block = np.unpackbits(block, axis=1, count=n_features, bitorder=packing)
        block = np.ascontiguousarray(block, dtype=np.int32)
        result.update(memoryview(block).cast("B"))
    return result.hexdigest()


def state_fingerprint(model, n_features, packed=False):
    fields = ("labels", "aggregation_labels", "splist", "group_sizes", "group_labels",
              "initial_merging_labels", "ind", "unsort_ind", "Adj")
    state = {}
    for name in fields:
        value = getattr(model, name)
        if name == "Adj" and hasattr(value, "iter_row_blocks"):
            digest = hashlib.sha256()
            for block in value.iter_row_blocks(rows=max(1, min(128, (8*1024*1024)//max(1, value.shape[1])))):
                digest.update(memoryview(np.ascontiguousarray(block)).cast("B"))
            state[name] = {"shape": list(value.shape), "dtype": "int8", "sha256": digest.hexdigest()}
            continue
        array = np.asarray(value)
        state[name] = {"shape": list(array.shape), "dtype": str(array.dtype), "sha256": array_hash(array)}
    state["logical_centers"] = logical_hash(model.spdata, n_features, "little" if packed else None)
    state["aggregation_dict"] = hashlib.sha256(json.dumps(model.aggregation_dict, sort_keys=True, default=lambda v: int(v)).encode()).hexdigest()
    return state
'''


**`code/memory_win.py`**


In [ ]:
SOURCES['code/memory_win.py'] = r'''"""Windows process counters; private commit is distinct from resident memory."""
import ctypes
from ctypes import wintypes


class Counters(ctypes.Structure):
    _fields_ = [("cb", wintypes.DWORD), ("PageFaultCount", wintypes.DWORD)] + [
        (name, ctypes.c_size_t) for name in (
            "PeakWorkingSetSize", "WorkingSetSize", "QuotaPeakPagedPoolUsage",
            "QuotaPagedPoolUsage", "QuotaPeakNonPagedPoolUsage",
            "QuotaNonPagedPoolUsage", "PagefileUsage", "PeakPagefileUsage", "PrivateUsage")]


kernel = ctypes.WinDLL("kernel32", use_last_error=True)
psapi = ctypes.WinDLL("psapi", use_last_error=True)
kernel.GetCurrentProcess.restype = wintypes.HANDLE
kernel.OpenProcess.argtypes = [wintypes.DWORD, wintypes.BOOL, wintypes.DWORD]
kernel.OpenProcess.restype = wintypes.HANDLE
kernel.CloseHandle.argtypes = [wintypes.HANDLE]
kernel.CloseHandle.restype = wintypes.BOOL
psapi.GetProcessMemoryInfo.argtypes = [wintypes.HANDLE, ctypes.POINTER(Counters), wintypes.DWORD]
psapi.GetProcessMemoryInfo.restype = wintypes.BOOL


def open_process(pid):
    handle = kernel.OpenProcess(0x1000, False, pid)
    if not handle:
        raise ctypes.WinError(ctypes.get_last_error())
    return handle


def close_process(handle):
    kernel.CloseHandle(handle)


def snapshot(handle=None):
    counters = Counters()
    counters.cb = ctypes.sizeof(counters)
    if not psapi.GetProcessMemoryInfo(handle or kernel.GetCurrentProcess(), ctypes.byref(counters), counters.cb):
        raise ctypes.WinError(ctypes.get_last_error())
    return {
        "private_commit_bytes": counters.PrivateUsage,
        "working_set_bytes": counters.WorkingSetSize,
        "lifetime_peak_private_commit_bytes": counters.PeakPagefileUsage,
        "lifetime_peak_working_set_bytes": counters.PeakWorkingSetSize,
    }
'''


**`code/job_limit.py`**


In [ ]:
SOURCES['code/job_limit.py'] = r'''"""Windows job-object commit limit for a paused benchmark process tree.

The controller must start its child with CREATE_NO_WINDOW and keep the actual
worker paused on a READY/GO handshake until assign_job returns. Pass the actual
worker PID from READY, not the outer Windows venv-launcher PID: an already-running
launcher can use a separate nested job for its child, preventing both processes
from being added to a common new job. Only that explicitly identified worker is
assigned; subsequently created children inherit the job. The worker must have
no pre-existing subprocesses that require bounding. Process-tree discovery is
deliberately disabled: Windows parent PID snapshots can identify unrelated old
processes after a parent PID is reused. No breakaway or host-wide limit is
enabled. The outer launcher holds no benchmark arrays.

The memory limit concerns committed memory, not working-set size. Assignment
does not retroactively reject allocations made before joining the job; attach
before substantial allocations when a strict whole-run bound is needed.

API:
    job = assign_job(pid=ready["worker_pid"], max_bytes=4 * 1024**3)
    ... send GO, wait for the child ...
    close_job(job)  # also kills any remaining assigned descendants

Alternatively pass an existing process handle as the first positional argument.
An integer positional argument is a HANDLE, never interpreted as a PID.

References:
https://learn.microsoft.com/en-us/windows/win32/api/winnt/ns-winnt-jobobject_extended_limit_information
https://learn.microsoft.com/en-us/windows/win32/api/jobapi2/nf-jobapi2-assignprocesstojobobject
"""
import ctypes
from ctypes import wintypes
import os

if os.name != "nt":
    raise OSError("This benchmark resource limiter requires Windows")

_kernel = ctypes.WinDLL("kernel32", use_last_error=True)
_SIZE_T = ctypes.c_size_t
_ULONG_PTR = ctypes.c_size_t
_PROCESS_RIGHTS = 0x0001 | 0x0100 | 0x1000  # TERMINATE, SET_QUOTA, QUERY_LIMITED
_JOB_INFO_EXTENDED = 9
_JOB_OBJECT_LIMIT_JOB_MEMORY = 0x00000200
_JOB_OBJECT_LIMIT_KILL_ON_JOB_CLOSE = 0x00002000


class _BasicLimit(ctypes.Structure):
    _fields_ = [
        ("PerProcessUserTimeLimit", ctypes.c_int64),
        ("PerJobUserTimeLimit", ctypes.c_int64),
        ("LimitFlags", wintypes.DWORD),
        ("MinimumWorkingSetSize", _SIZE_T),
        ("MaximumWorkingSetSize", _SIZE_T),
        ("ActiveProcessLimit", wintypes.DWORD),
        ("Affinity", _ULONG_PTR),
        ("PriorityClass", wintypes.DWORD),
        ("SchedulingClass", wintypes.DWORD),
    ]


class _IoCounters(ctypes.Structure):
    _fields_ = [(name, ctypes.c_uint64) for name in (
        "ReadOperationCount", "WriteOperationCount", "OtherOperationCount",
        "ReadTransferCount", "WriteTransferCount", "OtherTransferCount",
    )]


class _ExtendedLimit(ctypes.Structure):
    _fields_ = [
        ("BasicLimitInformation", _BasicLimit),
        ("IoInfo", _IoCounters),
        ("ProcessMemoryLimit", _SIZE_T),
        ("JobMemoryLimit", _SIZE_T),
        ("PeakProcessMemoryUsed", _SIZE_T),
        ("PeakJobMemoryUsed", _SIZE_T),
    ]


_kernel.CreateJobObjectW.argtypes = [ctypes.c_void_p, wintypes.LPCWSTR]
_kernel.CreateJobObjectW.restype = wintypes.HANDLE
_kernel.SetInformationJobObject.argtypes = [wintypes.HANDLE, ctypes.c_int, ctypes.c_void_p, wintypes.DWORD]
_kernel.SetInformationJobObject.restype = wintypes.BOOL
_kernel.QueryInformationJobObject.argtypes = [wintypes.HANDLE, ctypes.c_int, ctypes.c_void_p, wintypes.DWORD, ctypes.c_void_p]
_kernel.QueryInformationJobObject.restype = wintypes.BOOL
_kernel.AssignProcessToJobObject.argtypes = [wintypes.HANDLE, wintypes.HANDLE]
_kernel.AssignProcessToJobObject.restype = wintypes.BOOL
_kernel.IsProcessInJob.argtypes = [wintypes.HANDLE, wintypes.HANDLE, ctypes.POINTER(wintypes.BOOL)]
_kernel.IsProcessInJob.restype = wintypes.BOOL
_kernel.OpenProcess.argtypes = [wintypes.DWORD, wintypes.BOOL, wintypes.DWORD]
_kernel.OpenProcess.restype = wintypes.HANDLE
_kernel.GetProcessId.argtypes = [wintypes.HANDLE]
_kernel.GetProcessId.restype = wintypes.DWORD
_kernel.CloseHandle.argtypes = [wintypes.HANDLE]
_kernel.CloseHandle.restype = wintypes.BOOL


def _failure(operation):
    code = ctypes.get_last_error()
    detail = ctypes.FormatError(code).strip()
    return OSError(code, f"{operation}: {detail}; benchmark must not proceed without its job limit")


def _open_process(pid):
    handle = _kernel.OpenProcess(_PROCESS_RIGHTS, False, pid)
    if not handle:
        raise _failure(f"OpenProcess({pid})")
    return handle


def process_in_job(job, *, pid):
    """Check a READY worker PID is contained before sending GO."""
    handle = _open_process(pid)
    try:
        present = wintypes.BOOL()
        if not _kernel.IsProcessInJob(handle, job, ctypes.byref(present)):
            raise _failure(f"IsProcessInJob({pid})")
        return bool(present.value)
    finally:
        _kernel.CloseHandle(handle)


def assign_job(process_handle=None, max_bytes=4 * 1024**3, *, pid=None,
               include_existing_descendants=False):
    """Bound the explicit worker; its future children inherit the same job.

    Exactly one of process_handle and pid is required. The worker must have no
    existing subprocesses requiring this limit. include_existing_descendants=True
    is rejected before opening/assigning any process; PID-only tree discovery is
    unsafe when Windows reuses PIDs. A failure, including incompatible
    enclosing/nested jobs, raises; partially assigned processes are terminated.
    The caller owns only the returned job handle, not a passed process handle.
    """
    if (process_handle is None) == (pid is None):
        raise ValueError("Pass exactly one process_handle or keyword pid")
    if include_existing_descendants is not False:
        raise ValueError("Existing-descendant discovery is disabled; pass the actual READY worker with no pre-existing subprocesses")
    if not isinstance(max_bytes, int) or not 0 < max_bytes <= ctypes.c_size_t(-1).value:
        raise ValueError("max_bytes must be a positive SIZE_T integer")
    own_process_handle = process_handle is None
    if own_process_handle:
        process_handle = _open_process(pid)
    job = None
    try:
        root_pid = int(_kernel.GetProcessId(process_handle))
        if not root_pid:
            raise _failure("GetProcessId")
        if root_pid == os.getpid():
            raise ValueError("Refusing to assign the controller itself to a kill-on-close job")
        job = _kernel.CreateJobObjectW(None, None)
        if not job:
            raise _failure("CreateJobObjectW")
        limits = _ExtendedLimit()
        limits.BasicLimitInformation.LimitFlags = (
            _JOB_OBJECT_LIMIT_JOB_MEMORY | _JOB_OBJECT_LIMIT_KILL_ON_JOB_CLOSE
        )
        limits.JobMemoryLimit = max_bytes
        if not _kernel.SetInformationJobObject(job, _JOB_INFO_EXTENDED, ctypes.byref(limits), ctypes.sizeof(limits)):
            raise _failure("SetInformationJobObject")
        if not _kernel.AssignProcessToJobObject(job, process_handle):
            raise _failure(f"AssignProcessToJobObject({root_pid}); nested-job incompatibility is possible")
        # No enumeration: only the READY process is assigned. Windows ensures
        # subsequently created children inherit the same aggregate job limit.
        return job
    except BaseException:
        if job:
            _kernel.CloseHandle(job)
        raise
    finally:
        if own_process_handle:
            _kernel.CloseHandle(process_handle)


def job_snapshot(job):
    """Return diagnostic Windows job counters.

    Peak counters can include a subsequently denied allocation attempt, so they
    may exceed the configured quota; do not interpret them as resident memory.
    """
    limits = _ExtendedLimit()
    if not _kernel.QueryInformationJobObject(job, _JOB_INFO_EXTENDED, ctypes.byref(limits), ctypes.sizeof(limits), None):
        raise _failure("QueryInformationJobObject")
    return {
        "memory_limit_bytes": int(limits.JobMemoryLimit),
        "peak_job_commit_bytes": int(limits.PeakJobMemoryUsed),
        "peak_process_commit_bytes": int(limits.PeakProcessMemoryUsed),
        "limit_flags": int(limits.BasicLimitInformation.LimitFlags),
    }


def close_job(job):
    """Close a valid job handle; any still-running member processes are killed."""
    if job and not _kernel.CloseHandle(job):
        raise _failure("CloseHandle(job)")
'''


**`code/reproduce.py`**


In [ ]:
SOURCES['code/reproduce.py'] = r'''"""Fresh Windows reproductions; recorded evidence is never used as new results.

Default: inspect the plan, inputs to generation, dependencies and native imports.
Use --prepare to generate data, or --run to execute the selected cases. Every
write stays beneath experiment_1A/runs/<name>. PAUSE stops at the next fit
boundary; STOP_CURRENT interrupts the current attempt and leaves it retriable.
Remove the control file and repeat the same command to resume.
"""
import argparse
from contextlib import contextmanager
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import queue
import statistics
import subprocess
import sys
import threading
import time
import traceback

sys.dont_write_bytecode = True
HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
RUN_DIR = None
METHODS = ("dbscan", "classix_packed", "bitbirch", "bblean", "butina")
PACKAGES = ("numpy", "scipy", "scikit-learn", "joblib", "cloudpickle", "narwhals",
            "rdkit", "bblean", "threadpoolctl", "tqdm")
TERMINAL = {"timeout", "memory_limit", "algorithm_error", "worker_error",
            "startup_timeout", "validation_timeout", "validation_error"}


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save(value, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    for attempt in range(40):
        try:
            temporary.replace(path)
            break
        except PermissionError:
            if attempt == 39:
                raise
            time.sleep(.05)


def require(condition, message):
    if not condition:
        raise ValueError(message)


def load_design():
    plan = read(ROOT / "plan.json")
    policy = read(ROOT / "records" / "repetition_amendment.json")
    require(tuple(plan["methods"]) == METHODS, "Expected the five recorded methods")
    cases = plan["cases"]
    require(len(cases) == len({case["case_id"] for case in cases}) == 6,
            "Expected the six-setting radius plan")
    require(len({case["data_id"] for case in cases}) == 1, "Expected one shared dataset")
    require("sample_count_100000" not in {case["case_id"] for case in cases},
            "The cancelled 100k sample case must remain excluded")
    require(plan["fit_timeout_seconds"] is None and plan["job_memory_limit_bytes"] == 20 * 1024**3,
            "The recorded resource policy is no fit cutoff and a 20-GiB job limit")
    for case in cases:
        generator = case["generator"]
        require(generator["seed"] == 101 and case["rows"] == generator["n"] * generator["num_clusters"],
                "Dataset seed or total size differs")
        if case["study"] == "radius":
            require(case["rows"] == 10000 and .3 <= case["x"] <= .8, "Radius dataset differs")
        for method in METHODS:
            ids = policy["schedules"][case["case_id"]][method]
            require(ids == sorted(set(ids)) and ids and any(value >= 0 for value in ids)
                    and all(value in (-1, 0, 1, 2) for value in ids), "Invalid recorded repetition schedule")
    return plan, policy


def select_cases(plan, requested):
    known = {case["case_id"] for case in plan["cases"]}
    require(not requested or set(requested) <= known, "Unknown case ID; --check lists available cases")
    return [case for case in plan["cases"] if not requested or case["case_id"] in requested]


def output_root(value):
    root = Path(value).resolve()
    allowed = (ROOT / "runs").resolve()
    require(root != allowed and root.is_relative_to(allowed),
            "--output-dir must be a named folder beneath experiment_1A/runs; recorded evidence is read-only")
    return root


def environment_check(plan, policy, cases, methods):
    """Import dependencies and native bindings, without constructing estimators."""
    require(os.name == "nt", "The recorded native kernels and memory protocol require 64-bit Windows")
    require(platform.machine().lower() in ("amd64", "x86_64") and sys.maxsize > 2**32,
            "Use a 64-bit x86 Windows Python with the included native binaries")
    for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ[name] = "1"
    versions = {name: importlib.metadata.version(name) for name in PACKAGES}
    sys.path.insert(0, str(HERE / "classix"))
    sys.path.insert(0, str(ROOT / "vendor" / "bitbirch"))
    for name in ("numpy", "scipy", "sklearn.cluster", "rdkit.DataStructs", "rdkit.ML.Cluster.Butina",
                 "bblean", "classix_t_packed_adjbits", "bitbirch.bitbirch", "memory_win", "job_limit"):
        importlib.import_module(name)
    similarity = importlib.import_module("bblean.similarity")
    require(similarity._jt_sim_arr_vec_packed.__module__ == "bblean._cpp_similarity",
            "BBLean must use its native packed-bit kernel")
    native = importlib.import_module("packed_native")
    schedule_counts = {method: {
        "warmups": sum(-1 in policy["schedules"][case["case_id"]][method] for case in cases),
        "measured": sum(sum(value >= 0 for value in policy["schedules"][case["case_id"]][method]) for case in cases),
    } for method in methods}
    return {"status": "passed", "python": sys.version, "executable": sys.executable,
            "platform": platform.platform(), "machine": platform.machine(), "packages": versions,
            "hardware_popcount": bool(native.HARDWARE_POPCOUNT), "selected_cases": len(cases),
            "selected_methods": methods, "case_ids": [case["case_id"] for case in cases],
            "selected_datasets": len({case["data_id"] for case in cases}), "repetitions": schedule_counts,
            "fit_timeout_seconds": plan["fit_timeout_seconds"], "job_memory_limit_bytes": plan["job_memory_limit_bytes"],
            "note": "Imports only; no dataset generation or clustering fits were run."}


def protocol(plan, environment, pilot_rows=None):
    files = [HERE / name for name in ("reproduce.py", "worker.py", "prepare_dataset.py",
             "generate_data_snapshot.py", "quality.py", "state_checks.py", "memory_win.py", "job_limit.py")]
    files += [path for path in (HERE / "classix").rglob("*") if path.is_file() and path.suffix in (".py", ".dll", ".c")]
    files += [path for path in (ROOT / "vendor" / "bitbirch").rglob("*.py")]
    return {"kind": "standalone_fresh_reproduction", "schema": 1,
            "plan_sha256": sha(ROOT / "plan.json"),
            "repetition_amendment_sha256": sha(ROOT / "records" / "repetition_amendment.json"),
            "source_sha256": {path.relative_to(ROOT).as_posix(): sha(path) for path in sorted(set(files))},
            "packages": environment["packages"], "python": sys.version,
            "platform": platform.platform(), "fit_timeout_seconds": plan["fit_timeout_seconds"],
            "job_memory_limit_bytes": plan["job_memory_limit_bytes"], "threads": 1, "pilot_rows": pilot_rows,
            "repetitions": "one measured fit per method, no warmup; diagnostic pilot" if pilot_rows else "recorded per-case schedules",
            "timing_scope": plan["timed_operations"]}


def execute(case, variant, repeat, plan, output_directory, pilot_rows=None):
    labels_path = output_directory / f"{variant}_{repeat}.npy"
    command = [sys.executable, str(HERE/"worker.py"), "--case", case["case_id"], "--variant", variant, "--output", str(labels_path), "--data-dir", str(RUN_DIR/"data"), "--plan", str(ROOT/"plan.json")]
    if pilot_rows:
        command += ["--rows", str(pilot_rows)]
    environment = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTHONHASHSEED="0")
    for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        environment[variable] = "1"
    process = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, env=environment, creationflags=subprocess.CREATE_NO_WINDOW)
    messages = queue.Queue()
    def reader():
        for line in process.stdout:
            messages.put(line.rstrip())
        messages.put(None)
    threading.Thread(target=reader, daemon=True).start()
    result = {"case_id": case["case_id"], "variant": variant, "repeat": repeat, "warmup": repeat < 0,
              "status": "running", "logs": [], "limits": {"fit_seconds": plan["fit_timeout_seconds"], "job_memory_bytes": plan["job_memory_limit_bytes"]}}
    phase, phase_start = "startup", time.perf_counter()
    job, handle = None, None
    sampled = {"samples": 0, "target_interval_seconds": .02, "max_interval_seconds": 0,
               "peak_private_commit_bytes": 0, "peak_working_set_bytes": 0}
    last_sample_time = None
    def sample():
        nonlocal last_sample_time
        now, values = time.perf_counter(), snapshot(handle)
        if last_sample_time is not None:
            sampled["max_interval_seconds"] = max(sampled["max_interval_seconds"], now-last_sample_time)
        last_sample_time = now
        sampled["samples"] += 1
        for metric in ("private_commit", "working_set"):
            key = "peak_"+metric+"_bytes"
            sampled[key] = max(sampled[key], values[metric+"_bytes"])
    heartbeat = time.perf_counter()
    last_control_check = time.perf_counter() - 2
    try:
        while True:
            elapsed = time.perf_counter()-phase_start
            if job is not None and time.perf_counter() - last_control_check >= 1:
                last_control_check = time.perf_counter()
                if (RUN_DIR / "STOP_CURRENT").exists():
                    result["status"] = "interrupted"
                    result["interruption_phase"] = phase
                    result["elapsed_in_phase_seconds"] = elapsed
                    break
            if time.perf_counter() - heartbeat >= 15:
                save({"case": case["case_id"], "variant": variant, "repeat": repeat,
                      "phase": phase, "elapsed_in_phase_seconds": elapsed,
                      "sampled_memory": sampled, "updated_utc": datetime.now(timezone.utc).isoformat()}, RUN_DIR/"active_fit.json")
                heartbeat = time.perf_counter()
            deadline = plan["fit_timeout_seconds"] if phase == "fit" else 300
            if deadline is not None and elapsed > deadline:
                result["status"] = "timeout" if phase == "fit" else phase+"_timeout"
                result["timeout_phase"] = phase
                result["elapsed_in_phase_seconds"] = elapsed
                break
            if phase == "fit" and handle:
                try:
                    sample()
                except OSError:
                    pass  # Read queued final/error output if the worker just exited.
            try:
                line = messages.get(timeout=.02 if phase == "fit" else .1)
            except queue.Empty:
                if process.poll() is not None and not messages.qsize():
                    break
                continue
            if line is None:
                break
            if not line.startswith("BENCH_JSON "):
                result["logs"].append(line)
                continue
            event = json.loads(line[len("BENCH_JSON "):])
            kind = event.pop("event")
            result[kind] = event
            if kind == "ready":
                # Monitor/constrain the actual Python PID, not its venv launcher.
                actual_pid = event["pid"]
                job = assign_job(pid=actual_pid, max_bytes=plan["job_memory_limit_bytes"])
                handle = open_process(actual_pid)
                result["monitored_pid"] = actual_pid
                result["launcher_pid"] = process.pid
                sample()
                phase, phase_start = "fit", time.perf_counter()
                process.stdin.write("GO\n")
                process.stdin.flush()
            elif kind == "fit_done":
                sample()
                phase, phase_start = "validation", time.perf_counter()
                process.stdin.write("VALIDATE\n")
                process.stdin.flush()
            elif kind == "fit_error":
                is_memory = "MemoryError" in event["error_type"] or "Unable to allocate" in event["message"]
                result["status"] = "memory_limit" if is_memory else "algorithm_error"
            elif kind == "validation_error":
                result["status"] = "validation_error"
            elif kind == "validated":
                result["status"] = "ok"
        if result["status"] == "running":
            result["status"] = "worker_error"
    finally:
        if job is not None:
            close_job(job)
        if process.poll() is None:
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
        result["exit_code"] = process.wait(timeout=10)
        if handle:
            close_process(handle)
    if sampled["samples"]:
        result["sampled_memory"] = sampled
    if "fit_done" in result:
        result["native_peaks_attained_in_fit_window"] = {metric:
            result["fit_done"]["after_fit"]["lifetime_peak_"+metric+"_bytes"] > result["ready"]["before_fit"]["lifetime_peak_"+metric+"_bytes"]
            for metric in ("private_commit", "working_set")}
    save({"case": case["case_id"], "variant": variant, "repeat": repeat,
          "finished": True, "status": result["status"],
          "updated_utc": datetime.now(timezone.utc).isoformat()}, RUN_DIR/"active_fit.json")
    return result


def summarize(runs, methods):
    output = []
    for method in methods:
        all_runs = [r for r in runs if r["variant"] == method]
        failures = [r for r in all_runs if r["status"] != "ok"]
        measured = [r for r in all_runs if r["status"] == "ok" and not r["warmup"]]
        row = {"method": method, "status": failures[-1]["status"] if failures else "ok", "measured_repeats": len(measured),
               "warning_runs": sum(bool(r["logs"]) for r in all_runs)}
        if measured:
            values = [r["fit_done"]["clustering_seconds"] for r in measured]
            row.update(median_seconds=statistics.median(values), min_seconds=min(values), max_seconds=max(values), all_seconds=values,
                ari_values=sorted({r["validated"]["ari"] for r in measured}), cluster_counts=sorted({r["validated"]["clusters"] for r in measured}),
                aggregation_groups=sorted({r["validated"]["aggregation_groups"] for r in measured if r["validated"]["aggregation_groups"] is not None}),
                median_peak_private_bytes=statistics.median(r["fit_done"]["after_fit"]["lifetime_peak_private_commit_bytes"] for r in measured),
                median_peak_working_set_bytes=statistics.median(r["fit_done"]["after_fit"]["lifetime_peak_working_set_bytes"] for r in measured),
                private_peak_verified_in_fit_window=all(r["native_peaks_attained_in_fit_window"]["private_commit"] for r in measured),
                working_set_peak_verified_in_fit_window=all(r["native_peaks_attained_in_fit_window"]["working_set"] for r in measured))
            row["noise_counts"] = sorted({r["validated"]["noise_points"] for r in measured})
        output.append(row)
    return output


def saved_path(checked, kind):
    relative = checked.get("output_relative_" + kind)
    return RUN_DIR / relative if relative else Path(checked[kind])


def method_finished(runs, ids):
    if not runs:
        return False
    sequence = [run["repeat"] for run in runs]
    require(sequence == ids[:len(sequence)], "Saved repeats are not a unique ordered prefix of the recorded schedule")
    require(all(run["warmup"] == (run["repeat"] == -1) for run in runs), "Warmup flag differs")
    failed = [index for index, run in enumerate(runs) if run["status"] != "ok"]
    if failed:
        require(failed == [len(runs) - 1] and runs[-1]["status"] in TERMINAL,
                "Invalid terminal failure or additional attempts after failure")
        return True
    return sequence == ids


def verify_run(run, current_protocol, partitions, state_reference):
    require(run["protocol"] == current_protocol, "Existing run uses a different reproduction protocol; select a new output folder")
    if run["status"] != "ok":
        return state_reference
    checked = run["validated"]
    require(saved_path(checked, "assignments_file").is_file(), "Saved assignments are missing")
    partition = checked["canonical_partition_sha256"]
    require(partitions.setdefault(run["variant"], partition) == partition, "Partition changed between repetitions")
    if checked["state_file"]:
        state = read(saved_path(checked, "state_file"))
        if state_reference is None:
            state_reference = state
        require(state == state_reference, "CLASSIX state changed between repetitions")
        checked["classix_state_repeats_consistent"] = True
    return state_reference


def prepare(case):
    command = [sys.executable, str(HERE / "prepare_dataset.py"), case["data_id"],
               "--data-dir", str(RUN_DIR / "data"), "--plan", str(ROOT / "plan.json")]
    subprocess.run(command, check=True, cwd=ROOT, creationflags=subprocess.CREATE_NO_WINDOW,
                   env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"))


@contextmanager
def prevent_idle_sleep():
    """Match the historical runner's Windows power request outside fit timing."""
    import ctypes
    kernel = ctypes.WinDLL("kernel32", use_last_error=True)
    kernel.SetThreadExecutionState.argtypes = [ctypes.c_uint32]
    kernel.SetThreadExecutionState.restype = ctypes.c_uint32
    if not kernel.SetThreadExecutionState(0x80000001):
        raise ctypes.WinError(ctypes.get_last_error())
    try:
        yield
    finally:
        kernel.SetThreadExecutionState(0x80000000)


def controlled_pause(case_id=None):
    if any((RUN_DIR / name).exists() for name in ("PAUSE", "STOP_CURRENT")):
        save({"paused": True, "case": case_id, "utc": datetime.now(timezone.utc).isoformat()}, RUN_DIR / "live_progress.json")
        print("Paused. Remove PAUSE/STOP_CURRENT in this output folder and rerun the same command to resume.", flush=True)
        return True
    return False


def run_cases(plan, policy, cases, methods, current_protocol, pilot_rows=None):
    results_dir = RUN_DIR / "results"
    results_dir.mkdir(exist_ok=True)
    for case_index, case in enumerate(cases):
        case_id = case["case_id"]
        schedules = {method: [0] for method in METHODS} if pilot_rows else policy["schedules"][case_id]
        path = results_dir / (case_id + ".json")
        if path.exists():
            result = read(path)
            require(result["case"] == case and result["protocol"] == current_protocol,
                    "Saved case or protocol differs; use a new output folder")
        else:
            result = {"case": case, "runs": [], "pilot_rows": pilot_rows,
                      "measured_repeats_requested": 1 if pilot_rows else plan["measured_repeats"], "protocol": current_protocol,
                      "method_repeat_ids": schedules, "reproduction": True}
        partitions, state_reference = {}, None
        require(all(run["variant"] in METHODS for run in result["runs"]), "Unexpected saved method")
        for run in result["runs"]:
            state_reference = verify_run(run, current_protocol, partitions, state_reference)
        finished = {method: method_finished([run for run in result["runs"] if run["variant"] == method], schedules[method])
                    for method in METHODS}
        if all(finished[method] for method in methods):
            print("REUSE " + case_id, flush=True)
            continue
        if controlled_pause(case_id):
            return False
        prepare(case)
        result["data_manifest"] = str(RUN_DIR / "data" / case["data_id"] / "manifest.json")
        result["output_relative_data_manifest"] = (Path("data") / case["data_id"] / "manifest.json").as_posix()
        save(result, path)
        existing = {(run["variant"], run["repeat"]) for run in result["runs"]}
        for repeat in sorted({value for method in methods for value in schedules[method]}):
            offset = max(repeat, 0) % len(methods)
            for variant in methods[offset:] + methods[:offset]:
                if finished[variant] or repeat not in schedules[variant] or (variant, repeat) in existing:
                    continue
                if controlled_pause(case_id):
                    return False
                stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
                assignments = results_dir / (case_id + "_assignments") / f"{variant}_{repeat}_{stamp}"
                save({"case_index": case_index + 1, "case_total": len(cases), "case": case_id,
                      "variant": variant, "repeat": repeat, "started_utc": datetime.now(timezone.utc).isoformat()},
                     RUN_DIR / "live_progress.json")
                run = execute(case, variant, repeat, plan, assignments, pilot_rows)
                run["protocol"] = current_protocol
                if "validated" in run:
                    checked = run["validated"]
                    for kind in ("assignments_file", "state_file"):
                        if checked.get(kind):
                            checked["output_relative_" + kind] = Path(checked[kind]).relative_to(RUN_DIR).as_posix()
                if run["status"] == "interrupted":
                    attempts = RUN_DIR / "interrupted_attempts"
                    attempts.mkdir(exist_ok=True)
                    save(run, attempts / f"{case_id}_{variant}_{repeat}_{stamp}.json")
                    save({"paused": True, "interrupted_current_fit": True, "case": case_id,
                          "variant": variant, "repeat": repeat}, RUN_DIR / "live_progress.json")
                    print("Interrupted attempt preserved; rerun after removing the control file to retry.", flush=True)
                    return False
                try:
                    state_reference = verify_run(run, current_protocol, partitions, state_reference)
                except Exception as error:
                    run["status"] = "validation_error"
                    run["validation_error"] = {"error_type": type(error).__name__, "message": str(error)}
                result["runs"].append(run)
                existing.add((variant, repeat))
                finished[variant] = method_finished([value for value in result["runs"] if value["variant"] == variant], schedules[variant])
                result["summary"] = summarize(result["runs"], METHODS)
                result["completed"] = all(finished.values())
                save(result, path)
                print(json.dumps({"case": case_id, "variant": variant, "repeat": repeat,
                                  "status": run["status"], "seconds": run.get("fit_done", {}).get("clustering_seconds")}), flush=True)
                if run["status"] == "validation_error":
                    raise RuntimeError("Validation failed; preserved evidence requires review before continuing")
        result["summary"] = summarize(result["runs"], METHODS)
        result["completed"] = all(finished.values())
        save(result, path)
    save({"complete": True, "selected_cases": len(cases), "selected_methods": methods,
          "note": "Completion concerns this selection; per-case completed flags cover all five methods."}, RUN_DIR / "live_progress.json")
    return True


def main():
    global RUN_DIR, open_process, close_process, snapshot, assign_job, close_job
    parser = argparse.ArgumentParser(description=__doc__)
    mode = parser.add_mutually_exclusive_group()
    mode.add_argument("--check", action="store_true", help="Check dependencies and design without generating data or fitting; default")
    mode.add_argument("--prepare", action="store_true", help="Generate shared data for selected cases only")
    mode.add_argument("--run", action="store_true", help="Run/resume selected cases using the recorded repetition schedule")
    parser.add_argument("--output-dir", type=Path, default=ROOT / "runs" / "reproduction")
    parser.add_argument("--cases", nargs="+", help="Case IDs listed by --check")
    parser.add_argument("--methods", nargs="+", choices=METHODS)
    parser.add_argument("--pilot-rows", type=int, help="Diagnostic prefix only: one measured fit per method, no warmup; requires --run and --cases")
    args = parser.parse_args()
    plan, policy = load_design()
    cases = select_cases(plan, args.cases)
    if args.pilot_rows is not None:
        require(args.run and args.cases and 0 < args.pilot_rows <= min(case["rows"] for case in cases),
                "--pilot-rows requires --run, explicit --cases, and a positive prefix within each dataset")
    methods = [method for method in METHODS if not args.methods or method in args.methods]
    RUN_DIR = output_root(args.output_dir)
    environment = environment_check(plan, policy, cases, methods)
    if args.pilot_rows:
        environment["pilot_rows"] = args.pilot_rows
        environment["repetitions"] = {method: {"warmups": 0, "measured": len(cases)} for method in methods}
        environment["pilot_note"] = "Functional smoke check only. Full arrays load before slicing, so pilot memory does not describe a standalone small dataset."
    print(json.dumps(environment, indent=2), flush=True)
    if not (args.prepare or args.run):
        return 0
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    lock = RUN_DIR / "run.lock"
    descriptor = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    os.write(descriptor, json.dumps({"pid": os.getpid(), "utc": datetime.now(timezone.utc).isoformat()}).encode())
    os.close(descriptor)
    try:
        current_protocol = protocol(plan, environment, args.pilot_rows)
        protocol_file = RUN_DIR / ("preparation_protocol.json" if args.prepare else "protocol.json")
        if protocol_file.exists():
            require(read(protocol_file) == current_protocol, "Output folder uses a different protocol; choose a new run name")
        else:
            save(current_protocol, protocol_file)
        save(environment, RUN_DIR / "environment.json")
        if args.prepare:
            done = set()
            for case in cases:
                if controlled_pause(case["case_id"]):
                    return 0
                if case["data_id"] not in done:
                    prepare(case)
                    done.add(case["data_id"])
            return 0
        from memory_win import open_process, close_process, snapshot
        from job_limit import assign_job, close_job
        with prevent_idle_sleep():
            environment["prevents_idle_sleep_during_run"] = True
            save(environment, RUN_DIR / "environment.json")
            run_cases(plan, policy, cases, methods, current_protocol, args.pilot_rows)
        return 0
    except BaseException as error:
        save({"status": "controller_error", "error_type": type(error).__name__, "message": str(error),
              "traceback": traceback.format_exc(), "utc": datetime.now(timezone.utc).isoformat()}, RUN_DIR / "controller_error.json")
        raise
    finally:
        lock.unlink(missing_ok=True)


if __name__ == "__main__":
    raise SystemExit(main())
'''


**`code/classix/classix_t_packed_adjbits.py`**


In [ ]:
SOURCES['code/classix/classix_t_packed_adjbits.py'] = r'''# Temporary storage-only prototype; source and focused diff are recorded.
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent))
from packed_adjacency import PackedAdjacency
#%%
import numpy as np
from time import time
from tqdm import tqdm
from copy import deepcopy
from collections import deque
import scipy.sparse as sparse
from scipy import sparse
from fractions import Fraction
from packed_native import packed_submatxvec, row_popcounts

class CLASSIX_T_PACKED:
    def __init__(self, sorting="popcount", radius=0.3, minPts=1, group_merging="tanimoto_distance", norm=True, mergeScale=1.4, post_alloc=True, mergeTinyGroups=True, verbose=1, short_log_form=True, use_spmv=True):
        self.group_merging = group_merging
        self.mergeScale = mergeScale
        self.mergeTinyGroups = mergeTinyGroups
        self.sorting = sorting
        self.radius = radius
        self.minPts = minPts
        self.norm = norm

        self.post_alloc = post_alloc
        self.mergeTinyGroups = mergeTinyGroups
        self.verbose = verbose
        self.truncate = short_log_form


    
    def __str__(self):
        return f"CLASSIX_T(sorting={self.sorting}, radius={self.radius}, minPts={self.minPts}, group_merging={self.group_merging}, mergeScale={self.mergeScale}, mergeTinyGroups={self.mergeTinyGroups})"
    
    def __repr__(self):
        return f"CLASSIX_T(sorting={self.sorting}, radius={self.radius}, minPts={self.minPts}, group_merging={self.group_merging}, mergeScale={self.mergeScale}, mergeTinyGroups={self.mergeTinyGroups})"
    
    def fit(self, data, r=None, mergeScale=None, minPts=None):
        total_time = time()
        if r is not None:
            self.radius = r
        
        if mergeScale is not None:
            self.mergeScale = mergeScale

        if minPts is not None:
            self.minPts = minPts

        # Input is np.packbits(binary_rows, axis=1, bitorder='little').
        # Any padding bits in the final byte must be zero.
        data = np.asarray(data)
        if data.ndim != 2 or data.dtype != np.uint8:
            raise ValueError("Expected a 2D uint8 array of packed binary fingerprints")
        if not data.shape[0] or not data.shape[1]:
            raise ValueError("Packed input must have at least one row and one byte")
        data = np.ascontiguousarray(data)
        n, fdim = data.shape
        sort_vals = row_popcounts(data).astype(np.int64, copy=False)

        # print(sort_vals)

        self.ind = np.argsort(sort_vals, kind='stable')
        self.unsort_ind = np.argsort(self.ind, kind='stable')
        sort_vals = sort_vals[self.ind] 
        data = data[self.ind,:] # sort data

        # print(data)

        print("\nOWN AGGREGATION")


        ips_time = 0.0
        search_time = 0.0
        loop_time = 0.0
        conversion_time = 0.0
        
        st = time()
        lab = 0

        # labels of the data points. Initially all are -1 to keep track of unallocated points
        self.labels = np.array([-1]*n)
        nr_dist = 0
        # Indices of group starting pts, equivalent of the gc list in the original code
        self.splist = []
        rhs = 1/(1-self.radius)+1; # rhs of tanimoto distance inequality
        rhsi = 1/rhs

        rhs_n = 2 - self.radius
        rhs_d = 1 - self.radius

        # Do a small check to see if radius is a rational number:
        # If it is, we can use the rhs_n and rhs_d values to avoid floating point errors!

        self.group_sizes = [] # group sizes
        self.group_starting_pts = [] # group starting pts in the sorted array

        # Packed rows are used directly: no CSR construction is required.
        self.aggregation_dict = {}
        
        
        # Aggregation
        for i in tqdm(range(n)):
            
            if self.labels[i] >= 0:
                continue
                
            clustc = data[i,:]
            
            # What if we take the starting point as a slice of the sparse array instead?
            # clustc = datas[i,:]
            #
            self.labels[i] = lab

            # num_group = 1
            self.splist.append(i)
            self.group_sizes.append(1)
            
            search_radius = sort_vals[i]/(1-self.radius) # Only the right side limit is needed
            
            st1 = time()
            last_j = np.searchsorted(sort_vals, search_radius, side='right')
            self.aggregation_dict[i] = last_j

            search_time += time() - st1

            st1 = time()

            # ips = data[i+1:last_j,:]@clustc.T
            ips = packed_submatxvec(data, i+1, last_j, clustc)

            ips_time += time() - st1
            # Let's make the product with rhsi times sort_vals the condition.
            # lhs = sort_vals[i+1:last_j]
            
            nr_dist += last_j - i - 1
            
            st_c = time()

            vec = ips/(sort_vals[i+1:last_j]+sort_vals[i])
            conversion_time += time() - st_c
            
            st1 = time()

            """
            Create a custom C function that takes in a range of indices, a vector, and a scalar value.
            If the value of the vector at index i is greater than 0, it will check if the value is greater than the scalar value,
            
            Maybe just try boolean masks first?
            Note: self.labels has size = n
                  vec has size = last_j - i - 1

            The boolean mask of vec needs to padded with False values to match the size of self.labels

            Then we can take the intersection of the two boolean masks.

            ```
            vec_mask = vec >= rhsi
            assigned_mask = self.labels >= 0
            
            ```

            """

            notAssigned = self.labels < 0
            vec_mask = vec >= rhsi
            vec_mask = np.pad(vec_mask, (i+1, n - last_j), 'constant', constant_values=False)
            reassignMask = np.logical_and(notAssigned, vec_mask)
            self.group_sizes[-1] += np.sum(reassignMask)
            self.labels[reassignMask] = lab


            # for j in range(i+1, last_j):
            #     if self.labels[j] >= 0:
            #         continue

            #     # if sort_vals[j]+sort_vals[i]<=rhs*ips[j-i-1]:
            #     if (sort_vals[j]+sort_vals[i])*rhs_d<=rhs_n*ips[j-i-1]: # No division!
            #         # num_group += 1
            #         self.labels[j] = lab
            #         self.group_sizes[-1] += 1
                            
            loop_time += time() - st1

            lab += 1
            
        print("time for ips:", ips_time)
        print("time for search:", search_time)
        print("time for conversion:", conversion_time)
        print("time for loop:", loop_time)
        print("nr_dist:", nr_dist)

        self.aggregation_labels = deepcopy(np.array(self.labels))
        self.search_time = search_time
        self.ips_time = ips_time
        self.aggregation_time = time()-st
        
        # merging
        st = time()

        self.spdata = data[self.splist,:]
        # Group representatives also remain packed.
        spdata_len = len(self.splist)
        self.group_centers = self.spdata

        # Scores of the group starting points
        sort_vals_sp = sort_vals[self.splist]

        # Group labels of the group starting points
        label_sp = self.labels[self.splist]
        scale = self.mergeScale

        
        

        self.Adj = PackedAdjacency(len(self.splist))
        
        t1 = 0.0
        t2 = 0.0
        t3 = 0.0
        minPts = self.minPts

        for i in tqdm(range(len(self.splist))):
            if not self.mergeTinyGroups:
                if self.group_sizes[i] < minPts:
                    continue
                
            xi = self.spdata[i, :]
            search_radius = sort_vals_sp[i]/(1-self.mergeScale*self.radius)

            st1 = time()
            last_j = np.searchsorted(sort_vals_sp, search_radius, side='right')
            first_j = i
            ips = packed_submatxvec(self.spdata, first_j, last_j, xi)
            # ips = np.matmul(self.spdata[first_j:last_j,:],xi.T)
            tanimoto_distance = 1 - ips/(sort_vals_sp[i] + sort_vals_sp[first_j:last_j] - ips)
            # tanimoto_distance = 1 - tanimoto_score
            t1 += time() - st1

            st2 = time()
            inds = np.where(tanimoto_distance <= scale*self.radius)

            if not self.mergeTinyGroups:
                inds = inds[0][self.group_sizes[inds[0]] >= minPts]
                inds = i + inds
            
            else:
                inds = i + inds[0]



            self.Adj[i, inds] = 1
            self.Adj[inds, i] = 1

            spl = np.unique(label_sp[ inds ])

            minlab = np.min(spl)

            t2 += time() - st2

            st3 = time()
            for label in spl:
                label_sp[label_sp==label] = minlab
            t3 += time() - st3
            # continue

        self.initial_merging_labels = deepcopy(label_sp)

        print("  merging time:", time()-st)
        self.merging_time = time()-st
        
        # Cluster redistribution
        print(" minPts Merging")
        st = time()
        ############################################################################################################

        labels_sp_copy = deepcopy(label_sp)
        ul = np.unique(label_sp)
        cs = np.zeros(len(ul))
        self.group_sizes = np.array(self.group_sizes)
        for i in range(len(ul)):
            id = np.where(label_sp==ul[i])
            label_sp[id] = i
            cs[i] = np.sum(self.group_sizes[id])

        small_clusters = np.where(cs<self.minPts)[0]

        # print("small clusters", small_clusters)

        labels_sp_copy_2 = deepcopy(label_sp)

        for i in small_clusters:
            ii = np.where(labels_sp_copy_2==i)[0]
            for iii in ii:
                xi = self.spdata[iii, :]
                # d = Tanimoto distances to all other group centers
                # ips = np.matmul(self.spdata, xi.T)
                ips = packed_submatxvec(self.spdata, 0, spdata_len, xi)
                d = 1 - ips/(sort_vals_sp[iii] + sort_vals_sp - ips)
                
                o = np.argsort(d, kind='stable')
                # merge with the closest group that has more than minPts
                for j in o:
                    if cs[labels_sp_copy_2[j]]>=minPts:
                        label_sp[iii] = labels_sp_copy_2[j]
                        # Instead of having a separate Adjanceny matrix for the points merged with minPts criteria
                        # we can just update the old Adjacency matrix here. The issue is that the distance matrix will
                        # have some weird discrepancies.
                        self.Adj[iii, j] = 2
                        self.Adj[j, iii] = 2
                        break

        ul = np.unique(label_sp)
        cs = np.zeros(len(ul))
        for i in range(len(ul)):
            id = np.where(label_sp==ul[i])
            label_sp[id] = i
            cs[i] = np.sum(self.group_sizes[id])

        # print("final cluster sizes", cs)

        print("  minPts merging time:", time()-st)
        
        for idx, label in enumerate(self.labels):
            self.labels[idx] = label_sp[label]

        ############################################################################################################

        self.labels = self.labels[self.unsort_ind]
        # print(self.labels)
        self.aggregation_labels = self.aggregation_labels[self.unsort_ind]
        self.group_labels = label_sp
        self.group_centre_pts = self.spdata
        self.group_centers = self.splist
        print("Total time:", time()-total_time)
        return self

    def explain(self, ind1=None, ind2=None):
        if ind1 is None and ind2 is None:
            # If there are no specific points to explain, print the general information
            print(f"The data was clustered into {len(self.group_labels)} groups. These were further merged into {len(np.unique(self.group_labels))} clusters.")

        if ind1 is not None and ind2 is None:
            # If only one index is provided, print the cluster information for that index
            print(f"The data point at index {ind1} was assigned to cluster {self.labels[ind1]}")

        if ind1 is not None and ind2 is not None:
            # If two indices are provided, print the cluster information
            if self.labels[ind1] == self.labels[ind2]:
                # If the two data points belong to the same cluster, check if they are in the same group
                print(f"The data points at indices {ind1} and {ind2} belong to the same cluster.")
                agg_label_1 = self.aggregation_labels[ind1]
                agg_label_2 = self.aggregation_labels[ind2]
                if agg_label_1 == agg_label_2:
                    # If the two data points are in the same group, print the information
                    print(f"The data points at indices {ind1} and {ind2} are in the same group {agg_label_1}.")
                    return None

                else:
                    # If the two data points are in different groups, find the shortest connection path between the two groups
                    print(f"The data points at indices {ind1} and {ind2} are in different groups.")
                    connected_path = self.bfs_shortest_path(self.Adj, agg_label_1, agg_label_2)
                    if connected_path is not None:
                        # If a connection path is found, print the path
                        print(f'The connections between {ind1} and {ind2} are via this path: {connected_path[0]} ', end="")
                        for k in range(len(connected_path)-1):
                            if self.Adj[connected_path[k]-1, connected_path[k+1]-1] == 2:
                                print(f'(minPts) -> {connected_path[k+1]}', end="")
                            else:
                                print(f' -> {connected_path[k+1]}', end="")

                        return connected_path
                    
                    else:
                        # If no connection path is found, print that there is no connection, there must be something wrong with the code
                        print(f"No connection path found between {ind1} and {ind2}, although they belong to different groups in the same cluster. Please check the program for bugs!")
                        return None

            else:
                print(f"The data points at indices {ind1} and {ind2} belong to different clusters.")


    def bfs_shortest_path(self, adj_matrix, start, goal):
        # Convert start and goal from 1-based to 0-based indices
        start -= 1
        goal -= 1
        
        # Initialize the queue with the start node and a path containing only the start node
        queue = deque([(start, [start])])
        
        # Keep track of visited nodes to avoid cycles
        visited = set()
        
        while queue:
            # Dequeue a node and the path that led to it
            current_node, path = queue.popleft()
            
            # If the current node is the goal, return the path
            if current_node == goal:
                return [node + 1 for node in path]  # Convert back to 1-based indices
            
            # Mark the current node as visited
            visited.add(current_node)
            
            # Enqueue all unvisited neighbors
            for neighbor, connected in enumerate(adj_matrix[current_node]):
                if connected and neighbor not in visited:
                    queue.append((neighbor, path + [neighbor]))

        return None
# %%
'''


**`code/classix/classix_t_reference_adjbits.py`**


In [ ]:
SOURCES['code/classix/classix_t_reference_adjbits.py'] = r'''# Temporary storage-only prototype; source and focused diff are recorded.
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent))
from packed_adjacency import PackedAdjacency
#%%
import numpy as np
from time import time
from tqdm import tqdm
from copy import deepcopy
from collections import deque
import scipy.sparse as sparse
from scipy import sparse
from fractions import Fraction
from packed_native import original_spsubmatxvec as spsubmatxvec

class CLASSIX_T:
    def __init__(self, sorting="popcount", radius=0.3, minPts=1, group_merging="tanimoto_distance", norm=True, mergeScale=1.4, post_alloc=True, mergeTinyGroups=True, verbose=1, short_log_form=True, use_spmv=True):
        self.group_merging = group_merging
        self.mergeScale = mergeScale
        self.mergeTinyGroups = mergeTinyGroups
        self.sorting = sorting
        self.radius = radius
        self.minPts = minPts
        self.norm = norm

        self.post_alloc = post_alloc
        self.mergeTinyGroups = mergeTinyGroups
        self.verbose = verbose
        self.truncate = short_log_form


    
    def __str__(self):
        return f"CLASSIX_T(sorting={self.sorting}, radius={self.radius}, minPts={self.minPts}, group_merging={self.group_merging}, mergeScale={self.mergeScale}, mergeTinyGroups={self.mergeTinyGroups})"
    
    def __repr__(self):
        return f"CLASSIX_T(sorting={self.sorting}, radius={self.radius}, minPts={self.minPts}, group_merging={self.group_merging}, mergeScale={self.mergeScale}, mergeTinyGroups={self.mergeTinyGroups})"
    
    def fit(self, data, r=None, mergeScale=None, minPts=None):
        total_time = time()
        if r is not None:
            self.radius = r
        
        if mergeScale is not None:
            self.mergeScale = mergeScale

        if minPts is not None:
            self.minPts = minPts

        n, fdim = data.shape
        sort_vals = np.sum(data, axis=1)

        # print(sort_vals)

        self.ind = np.argsort(sort_vals, kind='stable')
        self.unsort_ind = np.argsort(self.ind, kind='stable')
        sort_vals = sort_vals[self.ind] 
        data = data[self.ind,:] # sort data

        # print(data)

        print("\nOWN AGGREGATION")


        ips_time = 0.0
        search_time = 0.0
        loop_time = 0.0
        conversion_time = 0.0
        
        st = time()
        lab = 0

        # labels of the data points. Initially all are -1 to keep track of unallocated points
        self.labels = np.array([-1]*n)
        nr_dist = 0
        # Indices of group starting pts, equivalent of the gc list in the original code
        self.splist = []
        rhs = 1/(1-self.radius)+1; # rhs of tanimoto distance inequality
        rhsi = 1/rhs

        rhs_n = 2 - self.radius
        rhs_d = 1 - self.radius

        # Do a small check to see if radius is a rational number:
        # If it is, we can use the rhs_n and rhs_d values to avoid floating point errors!

        self.group_sizes = [] # group sizes
        self.group_starting_pts = [] # group starting pts in the sorted array

        datas = sparse.csr_matrix(data)
        datas_data = datas.data
        datas_indices = datas.indices
        datas_indptr = datas.indptr
        self.aggregation_dict = {}
        
        
        # Aggregation
        for i in tqdm(range(n)):
            
            if self.labels[i] >= 0:
                continue
                
            clustc = data[i,:]
            
            # What if we take the starting point as a slice of the sparse array instead?
            # clustc = datas[i,:]
            #
            self.labels[i] = lab

            # num_group = 1
            self.splist.append(i)
            self.group_sizes.append(1)
            
            search_radius = sort_vals[i]/(1-self.radius) # Only the right side limit is needed
            
            st1 = time()
            last_j = np.searchsorted(sort_vals, search_radius, side='right')
            self.aggregation_dict[i] = last_j

            search_time += time() - st1

            st1 = time()

            # ips = data[i+1:last_j,:]@clustc.T
            ips = spsubmatxvec(datas_data, datas_indptr, datas_indices, i+1, last_j, clustc)

            ips_time += time() - st1
            # Let's make the product with rhsi times sort_vals the condition.
            # lhs = sort_vals[i+1:last_j]
            
            nr_dist += last_j - i - 1
            
            st_c = time()

            vec = ips/(sort_vals[i+1:last_j]+sort_vals[i])
            conversion_time += time() - st_c
            
            st1 = time()

            """
            Create a custom C function that takes in a range of indices, a vector, and a scalar value.
            If the value of the vector at index i is greater than 0, it will check if the value is greater than the scalar value,
            
            Maybe just try boolean masks first?
            Note: self.labels has size = n
                  vec has size = last_j - i - 1

            The boolean mask of vec needs to padded with False values to match the size of self.labels

            Then we can take the intersection of the two boolean masks.

            ```
            vec_mask = vec >= rhsi
            assigned_mask = self.labels >= 0
            
            ```

            """

            notAssigned = self.labels < 0
            vec_mask = vec >= rhsi
            vec_mask = np.pad(vec_mask, (i+1, n - last_j), 'constant', constant_values=False)
            reassignMask = np.logical_and(notAssigned, vec_mask)
            self.group_sizes[-1] += np.sum(reassignMask)
            self.labels[reassignMask] = lab


            # for j in range(i+1, last_j):
            #     if self.labels[j] >= 0:
            #         continue

            #     # if sort_vals[j]+sort_vals[i]<=rhs*ips[j-i-1]:
            #     if (sort_vals[j]+sort_vals[i])*rhs_d<=rhs_n*ips[j-i-1]: # No division!
            #         # num_group += 1
            #         self.labels[j] = lab
            #         self.group_sizes[-1] += 1
                            
            loop_time += time() - st1

            lab += 1
            
        print("time for ips:", ips_time)
        print("time for search:", search_time)
        print("time for conversion:", conversion_time)
        print("time for loop:", loop_time)
        print("nr_dist:", nr_dist)

        self.aggregation_labels = deepcopy(np.array(self.labels))
        self.search_time = search_time
        self.ips_time = ips_time
        self.aggregation_time = time()-st
        
        # merging
        st = time()

        self.spdata = data[self.splist,:]
        spdatas = sparse.csr_matrix(self.spdata)
        spdatas_data = spdatas.data
        spdatas_indices = spdatas.indices
        spdatas_indptr = spdatas.indptr
        spdata_len = len(self.splist)
        self.group_centers = self.spdata

        # Scores of the group starting points
        sort_vals_sp = sort_vals[self.splist]

        # Group labels of the group starting points
        label_sp = self.labels[self.splist]
        scale = self.mergeScale

        
        

        self.Adj = PackedAdjacency(len(self.splist))
        
        t1 = 0.0
        t2 = 0.0
        t3 = 0.0
        minPts = self.minPts

        for i in tqdm(range(len(self.splist))):
            if not self.mergeTinyGroups:
                if self.group_sizes[i] < minPts:
                    continue
                
            xi = self.spdata[i, :]
            search_radius = sort_vals_sp[i]/(1-self.mergeScale*self.radius)

            st1 = time()
            last_j = np.searchsorted(sort_vals_sp, search_radius, side='right')
            first_j = i
            ips = spsubmatxvec(spdatas_data, spdatas_indptr, spdatas_indices, first_j, last_j, xi)
            # ips = np.matmul(self.spdata[first_j:last_j,:],xi.T)
            tanimoto_distance = 1 - ips/(sort_vals_sp[i] + sort_vals_sp[first_j:last_j] - ips)
            # tanimoto_distance = 1 - tanimoto_score
            t1 += time() - st1

            st2 = time()
            inds = np.where(tanimoto_distance <= scale*self.radius)

            if not self.mergeTinyGroups:
                inds = inds[0][self.group_sizes[inds[0]] >= minPts]
                inds = i + inds
            
            else:
                inds = i + inds[0]



            self.Adj[i, inds] = 1
            self.Adj[inds, i] = 1

            spl = np.unique(label_sp[ inds ])

            minlab = np.min(spl)

            t2 += time() - st2

            st3 = time()
            for label in spl:
                label_sp[label_sp==label] = minlab
            t3 += time() - st3
            # continue

        self.initial_merging_labels = deepcopy(label_sp)

        print("  merging time:", time()-st)
        self.merging_time = time()-st
        
        # Cluster redistribution
        print(" minPts Merging")
        st = time()
        ############################################################################################################

        labels_sp_copy = deepcopy(label_sp)
        ul = np.unique(label_sp)
        cs = np.zeros(len(ul))
        self.group_sizes = np.array(self.group_sizes)
        for i in range(len(ul)):
            id = np.where(label_sp==ul[i])
            label_sp[id] = i
            cs[i] = np.sum(self.group_sizes[id])

        small_clusters = np.where(cs<self.minPts)[0]

        # print("small clusters", small_clusters)

        labels_sp_copy_2 = deepcopy(label_sp)

        for i in small_clusters:
            ii = np.where(labels_sp_copy_2==i)[0]
            for iii in ii:
                xi = self.spdata[iii, :]
                # d = Tanimoto distances to all other group centers
                # ips = np.matmul(self.spdata, xi.T)
                ips = spsubmatxvec(spdatas_data, spdatas_indptr, spdatas_indices, 0, spdata_len, xi)
                d = 1 - ips/(sort_vals_sp[iii] + sort_vals_sp - ips)
                
                o = np.argsort(d, kind='stable')
                # merge with the closest group that has more than minPts
                for j in o:
                    if cs[labels_sp_copy_2[j]]>=minPts:
                        label_sp[iii] = labels_sp_copy_2[j]
                        # Instead of having a separate Adjanceny matrix for the points merged with minPts criteria
                        # we can just update the old Adjacency matrix here. The issue is that the distance matrix will
                        # have some weird discrepancies.
                        self.Adj[iii, j] = 2
                        self.Adj[j, iii] = 2
                        break

        ul = np.unique(label_sp)
        cs = np.zeros(len(ul))
        for i in range(len(ul)):
            id = np.where(label_sp==ul[i])
            label_sp[id] = i
            cs[i] = np.sum(self.group_sizes[id])

        # print("final cluster sizes", cs)

        print("  minPts merging time:", time()-st)
        
        for idx, label in enumerate(self.labels):
            self.labels[idx] = label_sp[label]

        ############################################################################################################

        self.labels = self.labels[self.unsort_ind]
        # print(self.labels)
        self.aggregation_labels = self.aggregation_labels[self.unsort_ind]
        self.group_labels = label_sp
        self.group_centre_pts = self.spdata
        self.group_centers = self.splist
        print("Total time:", time()-total_time)
        return self

    def explain(self, ind1=None, ind2=None):
        if ind1 is None and ind2 is None:
            # If there are no specific points to explain, print the general information
            print(f"The data was clustered into {len(self.group_labels)} groups. These were further merged into {len(np.unique(self.group_labels))} clusters.")

        if ind1 is not None and ind2 is None:
            # If only one index is provided, print the cluster information for that index
            print(f"The data point at index {ind1} was assigned to cluster {self.labels[ind1]}")

        if ind1 is not None and ind2 is not None:
            # If two indices are provided, print the cluster information
            if self.labels[ind1] == self.labels[ind2]:
                # If the two data points belong to the same cluster, check if they are in the same group
                print(f"The data points at indices {ind1} and {ind2} belong to the same cluster.")
                agg_label_1 = self.aggregation_labels[ind1]
                agg_label_2 = self.aggregation_labels[ind2]
                if agg_label_1 == agg_label_2:
                    # If the two data points are in the same group, print the information
                    print(f"The data points at indices {ind1} and {ind2} are in the same group {agg_label_1}.")
                    return None

                else:
                    # If the two data points are in different groups, find the shortest connection path between the two groups
                    print(f"The data points at indices {ind1} and {ind2} are in different groups.")
                    connected_path = self.bfs_shortest_path(self.Adj, agg_label_1, agg_label_2)
                    if connected_path is not None:
                        # If a connection path is found, print the path
                        print(f'The connections between {ind1} and {ind2} are via this path: {connected_path[0]} ', end="")
                        for k in range(len(connected_path)-1):
                            if self.Adj[connected_path[k]-1, connected_path[k+1]-1] == 2:
                                print(f'(minPts) -> {connected_path[k+1]}', end="")
                            else:
                                print(f' -> {connected_path[k+1]}', end="")

                        return connected_path
                    
                    else:
                        # If no connection path is found, print that there is no connection, there must be something wrong with the code
                        print(f"No connection path found between {ind1} and {ind2}, although they belong to different groups in the same cluster. Please check the program for bugs!")
                        return None

            else:
                print(f"The data points at indices {ind1} and {ind2} belong to different clusters.")


    def bfs_shortest_path(self, adj_matrix, start, goal):
        # Convert start and goal from 1-based to 0-based indices
        start -= 1
        goal -= 1
        
        # Initialize the queue with the start node and a path containing only the start node
        queue = deque([(start, [start])])
        
        # Keep track of visited nodes to avoid cycles
        visited = set()
        
        while queue:
            # Dequeue a node and the path that led to it
            current_node, path = queue.popleft()
            
            # If the current node is the goal, return the path
            if current_node == goal:
                return [node + 1 for node in path]  # Convert back to 1-based indices
            
            # Mark the current node as visited
            visited.add(current_node)
            
            # Enqueue all unvisited neighbors
            for neighbor, connected in enumerate(adj_matrix[current_node]):
                if connected and neighbor not in visited:
                    queue.append((neighbor, path + [neighbor]))

        return None
# %%
'''


**`code/classix/LICENSE`**


In [ ]:
SOURCES['code/classix/LICENSE'] = r'''MIT License

Copyright (c) 2026 Kaustubh Roy

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
'''


**`code/classix/packed_adjacency.py`**


In [ ]:
SOURCES['code/classix/packed_adjacency.py'] = r'''"""CLASSIX adjacency with packed connections and sparse minPts edge markers.

Each logical entry is 0, 1, or 2. One bit records whether an edge exists; sparse
row-to-column sets mark edges whose logical value is 2. The connection buffer is
exactly G * ceil(G / 8) bytes for a G by G matrix. Sparse marker overhead is
separate and exposed as a Python-object size estimate. No dense graph is
allocated by construction, reads of individual edges, or writes. Decoding a row
returns an independent int8 array, not a writable view.

Supported writes match CLASSIX's existing operations:
    graph[i, columns] = 1
    graph[rows, j] = 1
    graph[i, j] = 2
The same forms accept 0 to clear edges. Repeated indices and different bits in
one byte are handled correctly using unbuffered NumPy ufunc.at updates.

Call iter_row_blocks(rows=...) for bounded logical hashing or validation.
Implicit conversion of the whole graph to a dense NumPy array is prohibited.
"""
import operator
import sys

import numpy as np


def _integer(value, name):
    if isinstance(value, (bool, np.bool_)):
        raise TypeError(f"{name} must be an integer, not a boolean")
    try:
        return operator.index(value)
    except TypeError as error:
        raise TypeError(f"{name} must be an integer") from error


class PackedAdjacency:
    """Logical int8 adjacency: one bit/entry plus sparse markers for value 2."""

    __slots__ = ("_n", "_row_bytes", "_connections", "_minpts")
    dtype = np.dtype(np.int8)
    ndim = 2

    def __init__(self, groups):
        groups = _integer(groups, "groups")
        if groups < 0:
            raise ValueError("groups must be nonnegative")
        self._n = groups
        self._row_bytes = (groups + 7) // 8
        self._connections = np.zeros((groups, self._row_bytes), dtype=np.uint8)
        self._minpts = {}

    @property
    def shape(self):
        return (self._n, self._n)

    @property
    def size(self):
        return self._n * self._n

    @property
    def nbytes(self):
        """Exact connection-buffer bytes; sparse marker objects are separate."""
        return self._connections.nbytes

    @property
    def connection_bits(self):
        """Read-only packed rows marking every connection, including minPts edges."""
        view = self._connections.view()
        view.flags.writeable = False
        return view

    @property
    def marker_count(self):
        """Number of directed edges marked as minPts edges."""
        return sum(len(columns) for columns in self._minpts.values())

    @property
    def marker_storage_bytes(self):
        """Estimated sparse-marker Python object bytes, for reporting only.

        Counts the dictionary, sets, row/column integers once by object identity.
        It excludes allocator overhead and may include shared/interned integers;
        therefore this is an estimate, not measured process memory. Cost is O(M)
        in stored directed markers, so avoid querying inside the clustering loop.
        """
        seen = set()
        def size_once(value):
            identity = id(value)
            if identity in seen:
                return 0
            seen.add(identity)
            return sys.getsizeof(value)
        total = size_once(self._minpts)
        for row, columns in self._minpts.items():
            total += size_once(row) + size_once(columns)
            total += sum(size_once(column) for column in columns)
        return total

    @property
    def logical_nbytes(self):
        """Bytes the equivalent dense int8 matrix would require."""
        return self.size

    def __len__(self):
        return self._n

    def __repr__(self):
        return f"PackedAdjacency(shape={self.shape}, nbytes={self.nbytes}, marker_count={self.marker_count})"

    def __array__(self, dtype=None, copy=None):
        raise TypeError("Whole-graph conversion is disabled; use iter_row_blocks(rows=...) or graph[i]")

    def _index(self, index):
        index = _integer(index, "index")
        if index < 0:
            index += self._n
        if not 0 <= index < self._n:
            raise IndexError("adjacency index out of bounds")
        return index

    def _selection(self, selection):
        if isinstance(selection, (int, np.integer)) and not isinstance(selection, (bool, np.bool_)):
            return self._index(selection)
        values = np.asarray(selection)
        if values.ndim != 1:
            raise TypeError("indices must be a scalar integer or a one-dimensional integer array")
        if values.size == 0:
            return np.empty(0, dtype=np.intp)
        if values.dtype.kind not in "iu":
            raise TypeError("indices must have an integer dtype")
        # Check unsigned values before casting to np.intp to prevent wraparound.
        if np.any(values >= self._n) or (values.dtype.kind == "i" and np.any(values < -self._n)):
            raise IndexError("adjacency index out of bounds")
        values = values.astype(np.intp, copy=False)
        if np.any(values < 0):
            values = np.where(values < 0, values + self._n, values)
        return values

    def _pair(self, key):
        if not isinstance(key, tuple) or len(key) != 2:
            raise IndexError("edge access requires exactly two indices")
        row, column = (self._selection(index) for index in key)
        if isinstance(row, np.ndarray) and isinstance(column, np.ndarray):
            raise TypeError("at most one index may be a vector")
        return row, column

    def __getitem__(self, key):
        if not isinstance(key, tuple):
            row = self._index(key)
            decoded = np.unpackbits(self._connections[row], bitorder="little", count=self._n).view(np.int8)
            markers = self._minpts.get(row)
            if markers:
                decoded[np.fromiter(markers, dtype=np.intp, count=len(markers))] = 2
            return decoded
        row, column = self._pair(key)
        byte_index = column >> 3
        offset = column & 7
        result = (self._connections[row, byte_index] >> offset) & 1
        if np.ndim(result) == 0:
            return np.int8(2 if column in self._minpts.get(row, ()) else result)
        result = np.asarray(result, dtype=np.int8)
        if not self._minpts:
            return result
        if isinstance(row, int):
            markers = self._minpts.get(row)
            if markers:
                mask = np.fromiter((int(c) in markers for c in column), dtype=bool, count=len(column))
                result[mask] = 2
        else:
            mask = np.fromiter((column in self._minpts.get(int(r), ()) for r in row), dtype=bool, count=len(row))
            result[mask] = 2
        return result

    def _write_marker(self, row, column, marked):
        if marked:
            markers = self._minpts.get(row)
            if markers is None:
                self._minpts[row] = {column}
            else:
                markers.add(column)
        else:
            markers = self._minpts.get(row)
            if markers is not None:
                markers.discard(column)
                if not markers:
                    del self._minpts[row]

    def __setitem__(self, key, value):
        value = _integer(value, "edge value")
        if not 0 <= value <= 2:
            raise ValueError("edge values must be 0, 1, or 2")
        row, column = self._pair(key)
        if isinstance(row, int) and isinstance(column, int):
            byte_index = column >> 3
            mask = np.uint8(1 << (column & 7))
            clear_mask = np.bitwise_not(mask)
            current = self._connections[row, byte_index]
            self._connections[row, byte_index] = current | mask if value else current & clear_mask
            self._write_marker(row, column, value == 2)
            return
        if isinstance(column, np.ndarray):
            byte_indices = column >> 3
            masks = np.left_shift(np.uint8(1), (column & 7).astype(np.uint8))
            target = (row, byte_indices)
        else:
            masks = np.uint8(1 << (column & 7))
            target = (row, column >> 3)
        if value:
            np.bitwise_or.at(self._connections, target, masks)
        else:
            np.bitwise_and.at(self._connections, target, np.bitwise_not(masks))
        if isinstance(row, int):
            if value == 2 and column.size:
                markers = self._minpts.get(row)
                if markers is None:
                    self._minpts[row] = set(map(int, column))
                else:
                    markers.update(map(int, column))
            elif row in self._minpts:
                markers = self._minpts[row]
                markers.difference_update(map(int, column))
                if not markers:
                    del self._minpts[row]
        elif value == 2 or self._minpts:
            # Normal vector writes during merging incur no Python per-edge loop
            # while the marker dictionary is empty. minPts writes are scalar in
            # CLASSIX; support vector marker writes for a complete tested API.
            for item in map(int, row):
                self._write_marker(item, column, value == 2)

    def iter_row_blocks(self, rows=64):
        """Yield dense int8 row chunks with shape at most (rows, G).

        One decoded int8 chunk is allocated. Callers should process chunks sequentially rather than
        collecting them, to keep memory bounded.
        """
        rows = _integer(rows, "rows")
        if rows <= 0:
            raise ValueError("rows must be positive")
        for start in range(0, self._n, rows):
            stop = min(start + rows, self._n)
            decoded = np.unpackbits(self._connections[start:stop], axis=1,
                                    bitorder="little", count=self._n).view(np.int8)
            for row in range(start, stop):
                markers = self._minpts.get(row)
                if markers:
                    decoded[row - start, np.fromiter(markers, dtype=np.intp, count=len(markers))] = 2
            yield decoded
'''


**`code/classix/packed_kernel.c`**


In [ ]:
SOURCES['code/classix/packed_kernel.c'] = r'''/* Isolated binary-fingerprint prototype. Does not modify the original source.
 * Input: row-major np.packbits(..., axis=1, bitorder="little") uint8 arrays.
 * Padding bits must be zero. memcpy avoids alignment/aliasing assumptions.
 */
#include <stdint.h>
#include <stddef.h>
#include <string.h>
#include <intrin.h>

#define EXPORT __declspec(dllexport)

EXPORT int hardware_popcount_available(void) {
    int regs[4];
    __cpuid(regs, 1);
    return (regs[2] & (1 << 23)) != 0;
}

static __forceinline unsigned int portable_popcount64(uint64_t value) {
    value -= (value >> 1) & UINT64_C(0x5555555555555555);
    value = (value & UINT64_C(0x3333333333333333)) +
            ((value >> 2) & UINT64_C(0x3333333333333333));
    value = (value + (value >> 4)) & UINT64_C(0x0f0f0f0f0f0f0f0f);
    return (unsigned int)((value * UINT64_C(0x0101010101010101)) >> 56);
}

static __forceinline unsigned int popcount64(uint64_t value, int hardware) {
    return hardware ? (unsigned int)__popcnt64(value) : portable_popcount64(value);
}

EXPORT void row_popcounts(const uint8_t *packed, int nrows, int nbytes,
                          int32_t *output) {
    const int hardware = hardware_popcount_available();
    for (int row = 0; row < nrows; ++row) {
        const uint8_t *current = packed + (size_t)row * (size_t)nbytes;
        int32_t count = 0;
        int column = 0;
        for (; column + 8 <= nbytes; column += 8) {
            uint64_t word;
            memcpy(&word, current + column, sizeof(word));
            count += (int32_t)popcount64(word, hardware);
        }
        for (; column < nbytes; ++column) {
            count += (int32_t)popcount64(current[column], hardware);
        }
        output[row] = count;
    }
}

EXPORT void packed_submatxvec(const uint8_t *packed, int nbytes,
                             int start_row, int end_row,
                             const uint8_t *vector, int32_t *output) {
    const int hardware = hardware_popcount_available();
    for (int row = start_row; row < end_row; ++row) {
        const uint8_t *current = packed + (size_t)row * (size_t)nbytes;
        int32_t intersection = 0;
        int column = 0;
        for (; column + 8 <= nbytes; column += 8) {
            uint64_t lhs, rhs;
            memcpy(&lhs, current + column, sizeof(lhs));
            memcpy(&rhs, vector + column, sizeof(rhs));
            intersection += (int32_t)popcount64(lhs & rhs, hardware);
        }
        for (; column < nbytes; ++column) {
            intersection += (int32_t)popcount64(current[column] & vector[column], hardware);
        }
        output[row - start_row] = intersection;
    }
}
'''


**`code/classix/packed_native.py`**


In [ ]:
SOURCES['code/classix/packed_native.py'] = r'''"""Strict ctypes interfaces for the isolated packed-bit trial.

The original interface deliberately matches the original Python wrapper's hot
path and assumes valid immutable SciPy CSR arrays and row/vector bounds. Call
validate_original_csr once before timing; ctypes argtypes still enforce dtype,
dimensionality, and contiguity. Extra per-call validation would introduce
artificial overhead absent from the published wrapper.
"""
from ctypes import CDLL, c_int
from pathlib import Path

import numpy as np

_ROOT = Path(__file__).resolve().parent
_packed = CDLL(str(_ROOT / "packed_kernel.dll"))
_original = CDLL(str(_ROOT / "original_spmv.dll"))
_int32_1d = np.ctypeslib.ndpointer(dtype=np.int32, ndim=1, flags="C_CONTIGUOUS")
_uint8_1d = np.ctypeslib.ndpointer(dtype=np.uint8, ndim=1, flags="C_CONTIGUOUS")
_uint8_2d = np.ctypeslib.ndpointer(dtype=np.uint8, ndim=2, flags="C_CONTIGUOUS")
_packed.hardware_popcount_available.argtypes = []
_packed.hardware_popcount_available.restype = c_int
_packed.row_popcounts.argtypes = [_uint8_2d, c_int, c_int, _int32_1d]
_packed.row_popcounts.restype = None
_packed.packed_submatxvec.argtypes = [
    _uint8_2d, c_int, c_int, c_int, _uint8_1d, _int32_1d
]
_packed.packed_submatxvec.restype = None
_original.multiply.argtypes = [
    _int32_1d, _int32_1d, _int32_1d, c_int, c_int, _int32_1d, _int32_1d
]
_original.multiply.restype = None
HARDWARE_POPCOUNT = bool(_packed.hardware_popcount_available())
_INT_MAX = np.iinfo(np.int32).max


def _array(value, dtype, ndim, name):
    if not isinstance(value, np.ndarray):
        raise TypeError(f"{name} must be a numpy array")
    if value.dtype != dtype or value.ndim != ndim or not value.flags.c_contiguous:
        raise TypeError(f"{name} must be a C-contiguous {ndim}D {np.dtype(dtype)} array")
    return value


def _bounds(start, end, nrows):
    if not isinstance(start, (int, np.integer)) or not isinstance(end, (int, np.integer)):
        raise TypeError("row bounds must be integers")
    if not 0 <= start <= end <= nrows <= _INT_MAX:
        raise ValueError("invalid row bounds or too many rows for native int32 API")


def _packed_shape(data):
    _array(data, np.uint8, 2, "data")
    nrows, nbytes = data.shape
    if nrows > _INT_MAX or nbytes > _INT_MAX // 8:
        raise ValueError("packed shape exceeds native int32 API")
    return nrows, nbytes


def row_popcounts(data):
    nrows, nbytes = _packed_shape(data)
    output = np.empty(nrows, dtype=np.int32)
    _packed.row_popcounts(data, nrows, nbytes, output)
    return output


def packed_submatxvec(data, start, end, vector):
    nrows, nbytes = _packed_shape(data)
    _bounds(start, end, nrows)
    _array(vector, np.uint8, 1, "vector")
    if vector.size != nbytes:
        raise ValueError("packed vector length differs from packed matrix row width")
    output = np.empty(end - start, dtype=np.int32)
    _packed.packed_submatxvec(data, nbytes, start, end, vector, output)
    return output


def validate_original_csr(values, indptr, indices, nfeatures):
    """Validate full CSR index contents once, outside timing boundaries."""
    _array(values, np.int32, 1, "values")
    _array(indptr, np.int32, 1, "indptr")
    _array(indices, np.int32, 1, "indices")
    if indptr.size == 0 or values.size != indices.size:
        raise ValueError("invalid CSR array lengths")
    if indptr[0] != 0 or indptr[-1] != values.size or np.any(indptr[1:] < indptr[:-1]):
        raise ValueError("invalid CSR row pointers")
    if indices.size and (indices.min() < 0 or indices.max() >= nfeatures):
        raise ValueError("CSR column indices exceed vector bounds")
    if indptr.size - 1 > _INT_MAX:
        raise ValueError("too many CSR rows")


def original_spsubmatxvec(values, indptr, indices, start, end, vector):
    """Original hot path; callers supply valid SciPy CSR arrays and bounds."""
    output = np.zeros(end - start).astype(np.int32)
    _original.multiply(values, indptr, indices, start, end, vector, output)
    return output
'''


**`code/classix/spsubmatxvec.c`**


In [ ]:
SOURCES['code/classix/spsubmatxvec.c'] = r'''void multiply(
    const int *m_data, const int *m_indptr, const int *m_indices,
    int start_row, int end_row, const int *v_data, int *result) {
    
    int j = 0;
    int val = 0;
    int num_vals_matrix = 0;
    int idx = 0;
    int m_indptri = 0;
    
    for (int i = start_row; i < end_row; i++) {
        m_indptri = m_indptr[i];
        num_vals_matrix = m_indptr[i+1] - m_indptri;
        val = 0;
        for (j = m_indptri; j < m_indptri + num_vals_matrix; j++) { 
            idx = m_indices[j];
            val += m_data[j] * v_data[idx];
        }
        result[i-start_row] = val;
    }
}'''


**`vendor/bitbirch/bitbirch/__init__.py`**


In [ ]:
SOURCES['vendor/bitbirch/bitbirch/__init__.py'] = r''''''


**`vendor/bitbirch/bitbirch/bitbirch.py`**


In [ ]:
SOURCES['vendor/bitbirch/bitbirch/bitbirch.py'] = r'''# BitBIRCH is an open-source clustering module based on iSIM
#
# Please, cite the BitBIRCH paper: https://www.biorxiv.org/content/10.1101/2024.08.10.607459v1
#
# BitBIRCH is free software; you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, version 3.
#
# BitBIRCH is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU Lesser General Public License for more details.
#
# BitBIRCH authors (PYTHON): Ramon Alain Miranda Quintana <ramirandaq@gmail.com>, <quintana@chem.ufl.edu>
#                            Vicky (Vic) Jung <jungvicky@ufl.edu>
#                            Kenneth Lopez Perez <klopezperez@chem.ufl.edu>
#                            Kate Huddleston <kdavis2@chem.ufl.edu>
#
# BitBIRCH License: LGPL-3.0 https://www.gnu.org/licenses/lgpl-3.0.en.html#license-text
#
### Part of the tree-management code was derived from https://scikit-learn.org/stable/modules/generated/sklearn.cluster.Birch.html
### Authors: Manoj Kumar <manojkumarsivaraj334@gmail.com>
###          Alexandre Gramfort <alexandre.gramfort@telecom-paristech.fr>
###          Joel Nothman <joel.nothman@gmail.com>
### License: BSD 3 clause

import numpy as np
from scipy import sparse
from bitbirch.pruning import lazyPrune

def set_merge(merge_criterion, tolerance=0.05):
    """
    Sets merge_accept function for merge_subcluster, based on user specified merge_criteria. 

    Radius: merge subcluster based on comparison to centroid of the cluster
    Diameter: merge subcluster based on instant Tanimoto similarity of cluster
    Tolerance: applies tolerance threshold to diameter merge criteria, which will merge subcluster with stricter threshold for newly added molecules

    Parameters:
    -----------
    merge_criterion: str(); 
                        radius, diameter or tolerance
    tolerance: float; 
                        sets penalty value for similarity threshold when callng tolerance merge criteria

    Returns:
    --------
    merge_accept(): function 
                        if cluster is accepted to merge, merge the cluster based on the criteria specified
    """
    if merge_criterion == 'radius':
        def merge_accept(threshold, new_ls, new_centroid, new_n, old_ls, nom_ls, old_n, nom_n):
            jt_sim = jt_isim(new_ls + new_centroid, new_n + 1) * (new_n + 1) - jt_isim(new_ls, new_n) * (new_n - 1)
            return jt_sim >= threshold*2
    elif merge_criterion == 'diameter':
        def merge_accept(threshold, new_ls, new_centroid, new_n, old_ls, nom_ls, old_n, nom_n): 
            jt_radius = jt_isim(new_ls, new_n)
            return jt_radius >= threshold
    elif merge_criterion == 'tolerance_tough':
        def merge_accept(threshold, new_ls, new_centroid, new_n, old_ls, nom_ls, old_n, nom_n):
            jt_radius = jt_isim(new_ls, new_n)
            if jt_radius < threshold:
                return False
            else:
                if old_n == 1 and nom_n == 1:
                    return True
                elif nom_n == 1:
                    return (jt_isim(old_ls + nom_ls, old_n + 1) * (old_n + 1) - jt_isim(old_ls, old_n) * (old_n - 1))/2 >= jt_isim(old_ls, old_n) - tolerance and (jt_radius >= threshold)
                else:
                    return (jt_isim(old_ls + nom_ls, old_n + nom_n) * (old_n + nom_n) * (old_n + nom_n - 1) 
                    - jt_isim(old_ls, old_n) * old_n * (old_n - 1)
                    - jt_isim(nom_ls, nom_n) * nom_n * (nom_n - 1))/(2 * old_n * nom_n) >= jt_isim(old_ls, old_n) - tolerance and (jt_radius >= threshold)
    elif merge_criterion == 'tolerance':
        def merge_accept(threshold, new_ls, new_centroid, new_n, old_ls, nom_ls, old_n, nom_n):
            jt_radius = jt_isim(new_ls, new_n)
            if jt_radius < threshold:
                return False
            else:
                if old_n == 1 and nom_n == 1:
                    return True
                elif nom_n == 1:
                    return (jt_isim(old_ls + nom_ls, old_n + 1) * (old_n + 1) - jt_isim(old_ls, old_n) * (old_n - 1))/2 >= jt_isim(old_ls, old_n) - tolerance and (jt_radius >= threshold)
                else:
                    return True
    globals()['merge_accept'] = merge_accept

def jt_isim(c_total, n_objects):
    """iSIM Tanimoto calculation
    
    https://pubs.rsc.org/en/content/articlelanding/2024/dd/d4dd00041b
    
    Parameters
    ----------
    c_total : np.ndarray
              Sum of the elements column-wise
              
    n_objects : int
                Number of elements
                
    Returns
    ----------
    isim : float
           iSIM Jaccard-Tanimoto value
    """
    sum_kq = np.sum(c_total)
    sum_kqsq = np.dot(c_total, c_total)
    a = (sum_kqsq - sum_kq)/2

    return a/(a + n_objects * sum_kq - sum_kqsq)

def max_separation(X):
    """Finds two objects in X that are very separated
    This is an approximation (not guaranteed to find
    the two absolutely most separated objects), but it is
    a very robust O(N) implementation. Quality of clustering
    does not diminish in the end.
    
    Algorithm:
    a) Find centroid of X
    b) mol1 is the molecule most distant from the centroid
    c) mol2 is the molecule most distant from mol1
    
    Returns
    -------
    (mol1, mol2) : (int, int)
                   indices of mol1 and mol2
    1 - sims_mol1 : np.ndarray
                   Distances to mol1
    1 - sims_mol2: np.ndarray
                   Distances to mol2
    These are needed for node1_dist and node2_dist in _split_node
    """
    # Get the centroid of the set
    n_samples = len(X)
    linear_sum = np.sum(X, axis = 0)
    centroid = calc_centroid(linear_sum, n_samples)

    # Get the similarity of each molecule to the centroid
    pop_counts = np.sum(X, axis = 1)
    a_centroid = np.dot(X, centroid)
    sims_med = a_centroid / (pop_counts + np.sum(centroid) - a_centroid)

    # Get the least similar molecule to the centroid
    mol1 = np.argmin(sims_med)

    # Get the similarity of each molecule to mol1
    a_mol1 = np.dot(X, X[mol1])
    sims_mol1 = a_mol1 / (pop_counts + pop_counts[mol1] - a_mol1)

    # Get the least similar molecule to mol1
    mol2 = np.argmin(sims_mol1)

    # Get the similarity of each molecule to mol2
    a_mol2 = np.dot(X, X[mol2])
    sims_mol2 = a_mol2 / (pop_counts + pop_counts[mol2] - a_mol2)
    
    return (mol1, mol2), sims_mol1, sims_mol2

def calc_centroid(linear_sum, n_samples):
    """Calculates centroid
    
    Parameters
    ----------
    
    linear_sum : np.ndarray
                 Sum of the elements column-wise
    n_samples : int
                Number of samples
                
    Returns
    -------
    centroid : np.ndarray
               Centroid fingerprints of the given set
    """
    return np.where(linear_sum >= n_samples * 0.5 , 1, 0)

def _iterate_sparse_X(X):
    """This little hack returns a densified row when iterating over a sparse
    matrix, instead of constructing a sparse matrix for every row that is
    expensive.
    """
    n_samples, n_features = X.shape
    X_indices = X.indices
    X_data = X.data
    X_indptr = X.indptr

    for i in range(n_samples):
        row = np.zeros(n_features)
        startptr, endptr = X_indptr[i], X_indptr[i + 1]
        nonzero_indices = X_indices[startptr:endptr]
        row[nonzero_indices] = X_data[startptr:endptr]
        yield row

def _split_node(node, threshold, branching_factor, singly):
    """The node has to be split if there is no place for a new subcluster
    in the node.
    1. Two empty nodes and two empty subclusters are initialized.
    2. The pair of distant subclusters are found.
    3. The properties of the empty subclusters and nodes are updated
       according to the nearest distance between the subclusters to the
       pair of distant subclusters.
    4. The two nodes are set as children to the two subclusters.
    """
    new_subcluster1 = _BFSubcluster()
    new_subcluster2 = _BFSubcluster()
    new_node1 = _BFNode(
        threshold=threshold,
        branching_factor=branching_factor,
        is_leaf=node.is_leaf,
        n_features=node.n_features,
        dtype=node.init_centroids_.dtype,
    )
    new_node2 = _BFNode(
        threshold=threshold,
        branching_factor=branching_factor,
        is_leaf=node.is_leaf,
        n_features=node.n_features,
        dtype=node.init_centroids_.dtype,
    )
    new_subcluster1.child_ = new_node1
    new_subcluster2.child_ = new_node2

    if node.is_leaf:
        if node.prev_leaf_ is not None:
            node.prev_leaf_.next_leaf_ = new_node1
        new_node1.prev_leaf_ = node.prev_leaf_
        new_node1.next_leaf_ = new_node2
        new_node2.prev_leaf_ = new_node1
        new_node2.next_leaf_ = node.next_leaf_
        if node.next_leaf_ is not None:
            node.next_leaf_.prev_leaf_ = new_node2  
    
    # O(N) implementation of max separation
    farthest_idx, node1_dist, node2_dist = max_separation(node.centroids_)    
    # Notice that max_separation is returning similarities and not distances
    node1_closer = node1_dist > node2_dist
    # Make sure node1 is closest to itself even if all distances are equal.
    # This can only happen when all node.centroids_ are duplicates leading to all
    # distances between centroids being zero.
    node1_closer[farthest_idx[0]] = True

    for idx, subcluster in enumerate(node.subclusters_):
        if node1_closer[idx]:
            new_node1.append_subcluster(subcluster)
            new_subcluster1.update(subcluster)
            if not singly:
                subcluster.parent_ = new_subcluster1
        else:
            new_node2.append_subcluster(subcluster)
            new_subcluster2.update(subcluster)
            if not singly:
                subcluster.parent_ = new_subcluster2
    return new_subcluster1, new_subcluster2


class _BFNode:
    """Each node in a BFTree is called a BFNode.

    The BFNode can have a maximum of branching_factor
    number of BFSubclusters.

    Parameters
    ----------
    threshold : float
        Threshold needed for a new subcluster to enter a BFSubcluster.

    branching_factor : int
        Maximum number of BF subclusters in each node.

    is_leaf : bool
        We need to know if the BFNode is a leaf or not, in order to
        retrieve the final subclusters.

    n_features : int
        The number of features.

    Attributes
    ----------
    subclusters_ : list
        List of subclusters for a particular BFNode.

    prev_leaf_ : _BFNode
        Useful only if is_leaf is True.

    next_leaf_ : _BFNode
        next_leaf. Useful only if is_leaf is True.
        the final subclusters.

    init_centroids_ : ndarray of shape (branching_factor + 1, n_features)
        Manipulate ``init_centroids_`` throughout rather than centroids_ since
        the centroids are just a view of the ``init_centroids_`` .

    centroids_ : ndarray of shape (branching_factor + 1, n_features)
        View of ``init_centroids_``.

    """

    def __init__(self, *, threshold, branching_factor, is_leaf, n_features, dtype):
        self.threshold = threshold
        self.branching_factor = branching_factor
        self.is_leaf = is_leaf
        self.n_features = n_features

        # The list of subclusters, centroids and squared norms
        # to manipulate throughout.
        self.subclusters_ = []
        self.init_centroids_ = np.zeros((branching_factor + 1, n_features), dtype=dtype)
        self.prev_leaf_ = None
        self.next_leaf_ = None

    def append_subcluster(self, subcluster):
        n_samples = len(self.subclusters_)
        self.subclusters_.append(subcluster)
        self.init_centroids_[n_samples] = subcluster.centroid_
        
        # Keep centroids as views. In this way
        # if we change init_centroids, it is sufficient
        self.centroids_ = self.init_centroids_[: n_samples + 1, :]
        
    def update_split_subclusters(self, subcluster, new_subcluster1, new_subcluster2, singly):
        """Remove a subcluster from a node and update it with the
        split subclusters.
        """
        if not singly:
            new_subcluster1.parent_ = self.subclusters_[0].parent_
            new_subcluster2.parent_ = self.subclusters_[0].parent_

        ind = self.subclusters_.index(subcluster)
        self.subclusters_[ind] = new_subcluster1
        self.init_centroids_[ind] = new_subcluster1.centroid_
        self.centroids_[ind] = new_subcluster1.centroid_
        self.append_subcluster(new_subcluster2)

    def insert_bf_subcluster(self, subcluster, set_bits, ps, singly):
        """Insert a new subcluster into the node."""
        if not self.subclusters_:
            self.append_subcluster(subcluster)
            return False

        threshold = self.threshold
        branching_factor = self.branching_factor
        # We need to find the closest subcluster among all the
        # subclusters so that we can insert our new subcluster.
        a = np.dot(self.centroids_, subcluster.centroid_)
        sim_matrix = a / (np.sum(self.centroids_, axis = 1) + set_bits - a)
        closest_index = np.argmax(sim_matrix)
        closest_subcluster = self.subclusters_[closest_index]

        # If the subcluster has a child, we need a recursive strategy.
        if closest_subcluster.child_ is not None:
            ps = closest_subcluster
            split_child = closest_subcluster.child_.insert_bf_subcluster(subcluster, set_bits, ps, singly)

            if not split_child:
                # If it is determined that the child need not be split, we
                # can just update the closest_subcluster
                closest_subcluster.update(subcluster)
                self.init_centroids_[closest_index] = self.subclusters_[closest_index].centroid_
                self.centroids_[closest_index] = self.subclusters_[closest_index].centroid_
                return False

            # things not too good. we need to redistribute the subclusters in
            # our child node, and add a new subcluster in the parent
            # subcluster to accommodate the new child.
            else:
                new_subcluster1, new_subcluster2 = _split_node(
                    closest_subcluster.child_,
                    threshold,
                    branching_factor,
                    singly
                )
                self.update_split_subclusters(
                    closest_subcluster, new_subcluster1, new_subcluster2, singly
                )

                if len(self.subclusters_) > self.branching_factor:
                    return True
                return False

        # good to go!
        else:
            merged = closest_subcluster.merge_subcluster(subcluster, self.threshold)
            if merged:
                self.centroids_[closest_index] = closest_subcluster.centroid_
                self.init_centroids_[closest_index] = closest_subcluster.centroid_
                if not singly:
                    closest_subcluster.parent_ = ps
                return False

            # not close to any other subclusters, and we still
            # have space, so add.
            elif len(self.subclusters_) < self.branching_factor:
                self.append_subcluster(subcluster)
                if not singly:
                    closest_subcluster.parent_ = ps
                return False

            # We do not have enough space nor is it closer to an
            # other subcluster. We need to split.
            else:
                self.append_subcluster(subcluster)
                return True


class _BFSubcluster:
    """Each subcluster in a BFNode is called a BFSubcluster.

    A BFSubcluster can have a BFNode has its child.

    Parameters
    ----------
    linear_sum : ndarray of shape (n_features,), default=None
        Sample. This is kept optional to allow initialization of empty
        subclusters.

    Attributes
    ----------
    n_samples_ : int
        Number of samples that belong to each subcluster.

    linear_sum_ : ndarray
        Linear sum of all the samples in a subcluster. Prevents holding
        all sample data in memory.

    centroid_ : ndarray of shape (branching_factor + 1, n_features)
        Centroid of the subcluster. Prevent recomputing of centroids when
        ``BFNode.centroids_`` is called.
    
    mol_indices : list, default=[]
        List of indices of molecules included in the given cluster.

    child_ : _BFNode
        Child Node of the subcluster. Once a given _BFNode is set as the child
        of the _BFNode, it is set to ``self.child_``.
    """

    def __init__(self, *, linear_sum = None, mol_indices = []):
        if linear_sum is None:
            self.n_samples_ = 0
            self.centroid_ = self.linear_sum_ = 0
            self.mol_indices = []
        else:
            self.n_samples_ = 1
            self.centroid_ = self.linear_sum_ = linear_sum
            self.mol_indices = mol_indices
        
        self.child_ = None
        self.parent_ = None

    def update(self, subcluster):
        self.n_samples_ += subcluster.n_samples_
        self.linear_sum_ += subcluster.linear_sum_
        self.mol_indices += subcluster.mol_indices
        self.centroid_ = calc_centroid(self.linear_sum_, self.n_samples_)

    def merge_subcluster(self, nominee_cluster, threshold):
        """Check if a cluster is worthy enough to be merged. If
        yes then merge.
        """
        new_ls = self.linear_sum_ + nominee_cluster.linear_sum_
        new_n = self.n_samples_ + nominee_cluster.n_samples_
        new_centroid = calc_centroid(new_ls, new_n)
        
        if merge_accept(threshold, new_ls, new_centroid, new_n, self.linear_sum_, nominee_cluster.linear_sum_, self.n_samples_, nominee_cluster.n_samples_):
            (
                self.n_samples_,
                self.linear_sum_,
                self.centroid_,
                self.mol_indices
            ) = (new_n, new_ls, new_centroid, self.mol_indices + nominee_cluster.mol_indices)
            return True
        return False


class BitBirch():
    """Implements the BitBIRCH clustering algorithm.
    
    BitBIRCH paper: 

    Memory- and time-efficient, online-learning algorithm.
    It constructs a tree data structure with the cluster centroids being read off the leaf.
    
    Parameters
    ----------
    threshold : float, default=0.5
        The similarity radius of the subcluster obtained by merging a new sample and the
        closest subcluster should be greater than the threshold. Otherwise a new
        subcluster is started. Setting this value to be very low promotes
        splitting and vice-versa.

    branching_factor : int, default=50
        Maximum number of BF subclusters in each node. If a new samples enters
        such that the number of subclusters exceed the branching_factor then
        that node is split into two nodes with the subclusters redistributed
        in each. The parent subcluster of that node is removed and two new
        subclusters are added as parents of the 2 split nodes.

    Attributes
    ----------
    root_ : _BFNode
        Root of the BFTree.

    dummy_leaf_ : _BFNode
        Start pointer to all the leaves.

    subcluster_centers_ : ndarray
        Centroids of all subclusters read directly from the leaves.

    Notes
    -----
    The tree data structure consists of nodes with each node consisting of
    a number of subclusters. The maximum number of subclusters in a node
    is determined by the branching factor. Each subcluster maintains a
    linear sum, mol_indices and the number of samples in that subcluster.
    In addition, each subcluster can also have a node as its child, if the
    subcluster is not a member of a leaf node.

    For a new point entering the root, it is merged with the subcluster closest
    to it and the linear sum, mol_indices and the number of samples of that
    subcluster are updated. This is done recursively till the properties of
    the leaf node are updated.
    """


    def __init__(
        self,
        *,
        threshold=0.5,
        branching_factor=50,
    ):
        self.threshold = threshold
        self.branching_factor = branching_factor
        self.index_tracker = 0
        self.first_call = True

    def fit(self, X, singly=True):
        """
        Build a BF Tree for the input data.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Input data.

        Returns
        -------
        self
            Fitted estimator.
        """

        # TODO: Add input verification

        return self._fit(X, singly)

    def _fit(self, X, singly=True):
        threshold = self.threshold
        branching_factor = self.branching_factor

        n_features = X.shape[1]
        d_type = X.dtype

        # If partial_fit is called for the first time or fit is called, we
        # start a new tree.
        if self.first_call:
            # The first root is the leaf. Manipulate this object throughout.
            self.root_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )
    
            # To enable getting back subclusters.
            self.dummy_leaf_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )
            self.dummy_leaf_.next_leaf_ = self.root_
            self.root_.prev_leaf_ = self.dummy_leaf_

        # Cannot vectorize. Enough to convince to use cython.
        if not sparse.issparse(X):
            iter_func = iter
        else:
            iter_func = _iterate_sparse_X

        for sample in iter_func(X):
            set_bits = np.sum(sample)
            subcluster = _BFSubcluster(linear_sum=sample, mol_indices = [self.index_tracker])
            split = self.root_.insert_bf_subcluster(subcluster, set_bits,subcluster.parent_, singly)

            if split:
                new_subcluster1, new_subcluster2 = _split_node(
                    self.root_, threshold, branching_factor, singly
                )
                del self.root_
                self.root_ = _BFNode(
                    threshold=threshold,
                    branching_factor=branching_factor,
                    is_leaf=False,
                    n_features=n_features,
                    dtype=d_type,
                )
                self.root_.append_subcluster(new_subcluster1)
                self.root_.append_subcluster(new_subcluster2)

                if not singly:
                    for i in new_subcluster1.child_.subclusters_:
                        i.parent_ = new_subcluster1
                    for i in new_subcluster2.child_.subclusters_:
                        i.parent_ = new_subcluster2

            self.index_tracker += 1

        centroids = np.concatenate([leaf.centroids_ for leaf in self._get_leaves()])
        self.subcluster_centers_ = centroids
        self._n_features_out = self.subcluster_centers_.shape[0]
        
        self.first_call = False
        return self
    
    def fit_reinsert(self, X, reinsert_indices, singly=False):
        """X corresponds to only the molecules that will be reinserted into the tree
        reinsert indices are the indices of the molecules that will be reinserted into the tree"""
        threshold = self.threshold
        branching_factor = self.branching_factor

        n_features = X.shape[1]
        d_type = X.dtype

        # If partial_fit is called for the first time or fit is called, we
        # start a new tree.
        if self.first_call:
            # The first root is the leaf. Manipulate this object throughout.
            self.root_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )
    
            # To enable getting back subclusters.
            self.dummy_leaf_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )
            self.dummy_leaf_.next_leaf_ = self.root_
            self.root_.prev_leaf_ = self.dummy_leaf_

        # Cannot vectorize. Enough to convince to use cython.
        if not sparse.issparse(X):
            iter_func = iter
        else:
            iter_func = _iterate_sparse_X

        for sample, mol_ind in zip(iter_func(X), reinsert_indices):
            set_bits = np.sum(sample)
            subcluster = _BFSubcluster(linear_sum=sample, mol_indices = [mol_ind])
            split = self.root_.insert_bf_subcluster(subcluster, set_bits, subcluster.parent_, False)
            if split:
                new_subcluster1, new_subcluster2 = _split_node(
                    self.root_, threshold, branching_factor, False
                )
                del self.root_
                self.root_ = _BFNode(
                    threshold=threshold,
                    branching_factor=branching_factor,
                    is_leaf=False,
                    n_features=n_features,
                    dtype=d_type,
                )
                self.root_.append_subcluster(new_subcluster1)
                self.root_.append_subcluster(new_subcluster2)

                if not singly:
                    for i in new_subcluster1.child_.subclusters_:
                        i.parent_ = new_subcluster1
                    for i in new_subcluster2.child_.subclusters_:
                        i.parent_ = new_subcluster2

        centroids = np.concatenate([leaf.centroids_ for leaf in self._get_leaves()])
        self.subcluster_centers_ = centroids
        self._n_features_out = self.subcluster_centers_.shape[0]
        
        self.first_call = False
        return self
    
    def fit_BFs(self, X):
        """
        Method to fit a BitBirch model with the given BitFeatyres.

        Parameters:
        -----------
        X : list of BitFeatures

        Returns:
        --------
        self
        """

        # Check that the input is a list of BitFeatures
        if type(X) != list or len(X[0]) != 3:
            raise ValueError('The input must be a list of BitFeatures')
        
        threshold = self.threshold
        branching_factor = self.branching_factor

        n_features = len(X[0][1])
        d_type = X[0][1].dtype

        # If partial_fit is called for the first time or fit is called, we
        # start a new tree.
        if self.first_call:
            # The first root is the leaf. Manipulate this object throughout.
            self.root_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )
    
            # To enable getting back subclusters.
            self.dummy_leaf_ = _BFNode(
                threshold=threshold,
                branching_factor=branching_factor,
                is_leaf=True,
                n_features=n_features,
                dtype=d_type,
            )
            self.dummy_leaf_.next_leaf_ = self.root_
            self.root_.prev_leaf_ = self.dummy_leaf_

        for sample in iter(X):

            cluster = _BFSubcluster()
            cluster.n_samples_, cluster.linear_sum_, cluster.mol_indices = sample[0], sample[1], sample[2]
            cluster.centroid_ = calc_centroid(cluster.linear_sum_, cluster.n_samples_)

            set_bits = np.sum(cluster.centroid_)
            split = self.root_.insert_bf_subcluster(cluster, set_bits, cluster.parent_, True)

            if split:
                new_subcluster1, new_subcluster2 = _split_node(
                    self.root_, threshold, branching_factor, True
                )
                del self.root_
                self.root_ = _BFNode(
                    threshold=threshold,
                    branching_factor=branching_factor,
                    is_leaf=False,
                    n_features=n_features,
                    dtype=d_type,
                )
                self.root_.append_subcluster(new_subcluster1)
                self.root_.append_subcluster(new_subcluster2)
            self.index_tracker += 1

        centroids = np.concatenate([leaf.centroids_ for leaf in self._get_leaves()])
        self.subcluster_centers_ = centroids
        self._n_features_out = self.subcluster_centers_.shape[0]
        
        self.first_call = False
        return self


    def _get_leaves(self):
        """
        Retrieve the leaves of the BF Node.

        Returns
        -------
        leaves : list of shape (n_leaves,)
            List of the leaf nodes.
        """
        leaf_ptr = self.dummy_leaf_.next_leaf_
        leaves = []
        while leaf_ptr is not None:
            leaves.append(leaf_ptr)
            leaf_ptr = leaf_ptr.next_leaf_
        return leaves
    
    def get_centroids_mol_ids(self):
        """Method to return a dictionary containing the centroids and mol indices of the leaves"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')
        
        centroids = []
        mol_ids = []
        for leaf in self._get_leaves():
            for subcluster in leaf.subclusters_:
                centroids.append(subcluster.centroid_)
                mol_ids.append(subcluster.mol_indices)

        dict_centroids_mol_ids = {'centroids': centroids, 'mol_ids': mol_ids}

        return dict_centroids_mol_ids
    
    def get_centroids(self):
        """Method to return a list of Numpy arrays containing the centroids' fingerprints"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')
        
        centroids = []
        for leaf in self._get_leaves():
            for subcluster in leaf.subclusters_:
                centroids.append(subcluster.centroid_)

        return centroids
    
    def get_cluster_mol_ids(self):
        """Method to return the indices of molecules in each cluster"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')
        
        clusters_mol_id = []
        for leaf in self._get_leaves():
            for subcluster in leaf.subclusters_:
                clusters_mol_id.append(subcluster.mol_indices)

        # Sort the clusters by the number of samples in the cluster
        clusters_mol_id = sorted(clusters_mol_id, key = lambda x: len(x), reverse = True)

        return clusters_mol_id
    
    def _get_BFs(self):
        """Method to return the BitFeatures of the leaves"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')
        
        BFs = []
        for leaf in self._get_leaves():
            for subcluster in leaf.subclusters_:
                BFs.append(subcluster)

        # Sort the BitFeatures by the number of samples in the cluster
        BFs = sorted(BFs, key = lambda x: x.n_samples_, reverse = True)

        return BFs
         
    def prepare_data_BFs(self, fps, initial_mol = 0):
        """Method to prepare the BitFeatures of the largest cluster and the rest of the clusters"""
        if self.first_call:
            raise ValueError('The model has not been fitted yet.')
        
        BFs = self._get_BFs()
        big, rest = BFs[0], BFs[1:]

        data = []
        for BF in rest:
            data.append([BF.n_samples_, BF.linear_sum_.astype(np.int64), BF.mol_indices])

        bigs = []
        for mol in big.mol_indices:
            bigs.append([1, fps[mol - initial_mol].astype(np.int64), [mol]])

        return data, bigs
    
    def get_assignments(self, n_mols):
        clustered_ids = self.get_cluster_mol_ids()

        assignments = np.full(n_mols, -1, dtype=int)
        for i, cluster in enumerate(clustered_ids):
            assignments[cluster] = i + 1

        # Check that there are no unassigned molecules
        assert np.all(assignments != -1)

        return assignments
    
#Refinement functionality 

    def _get_prune_indices(self):
        """Method to return the indices of molecules in the largest cluster, specifically to be used in fit_reinsert."""
        largest_cluster = max(
            (
                (leaf_idx, cluster_idx, len(cluster.mol_indices))
                for leaf_idx, leaf in enumerate(self._get_leaves())
                for cluster_idx, cluster in enumerate(leaf.subclusters_)
            ),
            key=lambda x: x[2],  # Sort by the cluster size
            default=(None, None, 0),  # Default if no clusters are found
        )
        prune_indices = lazyPrune(largest_cluster[0], largest_cluster[1], self) 
        return prune_indices

    def prune(self, X):
        """
        Retrieves the molecules in the largest cluster and "prunes" the tree by redistributing these molecules. 

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Input data.

        Returns
        -------
        self
            Fitted estimator.
        """
        mol_indices = self._get_prune_indices()
        cluster_fps = X[mol_indices]
        self.fit_reinsert(cluster_fps, mol_indices)
        return self 

    def _calc_centroid(self, X, cluster):
        """Calculates centroid
    
        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
                Input data.
        cluster: np.array of the molecule indices within the cluster
                
        Returns
        -------
        centroid : np.ndarray
               Centroid fingerprints of the given set
    """

        full_fp = X[cluster]
        linear_sum = np.sum(full_fp, axis=0)
        n_samples = len(full_fp)
        return calc_centroid(linear_sum, n_samples)

    def _get_top_cluster_params(self, X, top):
        """Method to recieve the cluster mol indices, centroids, and fingerprints of the top user-specified clusters.
        
        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Input data.
        
        top: int
            default: 20; specifies number of top largest clusters to reassign

        Returns
        -------
        top_clusters: sorted list
            list of len top; containing the cluster mol ids of given clusters

        centroids: np.array; shape (top, n_features)
            centroids of the top clusters

        mol_indices: list of indices with the top clusters; len of number of molecules in all top clusters

        data_top_clusters: np.array; shape (n_molecules, n_features)
            fingerprints of the molecules in the top clusters
        """
        top_clusters = sorted(self.get_cluster_mol_ids(), key=len, reverse=True)[:top]
        centroids = np.array([self._calc_centroid(X, c) for c in top_clusters])
        mol_indices = [i for c in top_clusters for i in c]
        data_top_clusters = np.array([X[i] for i in mol_indices])
        assert data_top_clusters.shape[0] == len(mol_indices)

        return top_clusters, centroids, mol_indices, data_top_clusters

    def _get_sim_matrix(self, data_top_clusters, centroids):
        """Method to receive the similarity matrix of the user-sepcified top clusters for reassign.
        
        Parameters
        ----------
        data_top_clusters: np.array; shape (n_molecules, n_features)
            fingerprints of the molecules in the top clusters

        centroids: np.array; shape (top, n_features)
            centroids of the top clusters

        Returns
        -------
        row_max: np.array; shape(n_molecules)
            Each entry is the index of the cluster centroid to which the molecule was closest
        """
        pop_counts = np.sum(centroids, axis=1)
        pop_mols = np.sum(data_top_clusters, axis=1).reshape(-1,1)
        ei = np.einsum("ij,kj->ik", data_top_clusters, centroids)
        tanis = ei/(pop_counts + pop_mols -ei)
        row_max = np.argmax(tanis, axis=1)
        assert row_max.shape == data_top_clusters.shape[0:1]
        
        return row_max

        
    def reassign(self, X, top=20, quick=False):
        """
        Reassign molecules across the user-specified (default 20) top largest clusters in the tree based on the tanimoto similarity matrix. 

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Input data.

        top: int
            default: 20; specifies number of top largest clusters to reassign

        quick: boolean
            default: False; if quick specifed, the whole tree will not be return, but a dictionary of top cluster data will be returned

        Return
        ------
        quick return: dict
            dictionary sorted by largest to smallest newly reassigned top clusters

        self
            Fitted estimator.
        """
        top_clusters, centroids, mol_indices, data_top_clusters = self._get_top_cluster_params(X, top)
        row_max = self._get_sim_matrix(data_top_clusters, centroids)


        final_clusters = {i: [] for i in np.unique(row_max)}
        for mol, cluster_idx in zip(mol_indices, row_max):
            final_clusters[cluster_idx].append(mol)
        
        if quick:
            return dict(sorted(final_clusters.items(), key=lambda item: len(item[1]), reverse=True))    #Return dict sorted by largest to smallest newly reassigned top clusters
        
        else:
            sub_clusters = [sc for leaf in self._get_leaves() for sc in leaf.subclusters_]
            assert len(sub_clusters) == len(self.get_cluster_mol_ids())
           
            top_sub_clusters = sorted(sub_clusters, key=lambda c: c.n_samples_, reverse=True)[:top]
   
            for sc, tc in zip(top_sub_clusters, top_clusters):
                assert sc.n_samples_ == len(tc)

            for cluster, c_ids in zip(top_sub_clusters, final_clusters.values()):
                cluster.n_samples_ = len(c_ids)
                cluster.linear_sum = np.sum(X[c_ids], axis=0)
                cluster.mol_indices = c_ids
                cluster.centroid_ = calc_centroid(cluster.linear_sum_, cluster.n_samples_)
        return self
            
'''


**`vendor/bitbirch/bitbirch/cluster_analysis.py`**


In [ ]:
SOURCES['vendor/bitbirch/bitbirch/cluster_analysis.py'] = r'''from bitbirch.cluster_control import chi, dbi, dunn, birch_analysis, radius, jt_isim
import numpy as np
from rdkit.Chem.Scaffolds import MurckoScaffold
import pandas as pd
from iSIM.sampling import outlier_sampling, medoid_sampling
from iSIM.utils import binary_fps

class ClusterAnalysis:
    def __init__(self, brc, fingerprints, smiles, 
            double=False):
        self.brc = brc
        self.fps = fingerprints
        self.smiles = smiles
        self.double = double

    def clustering_metrics(self, min_size=10):
        cluster_analysis = birch_analysis(self.brc, self.fps, min_size)
        return chi(cluster_analysis), dbi(cluster_analysis), dunn(cluster_analysis)

    def get_biggest_cluster(self):
        clusters = sorted(self.brc.get_cluster_mol_ids(), key=lambda x: len(x), reverse=True)
        biggest_cluster = np.array(clusters[0])
        fps_biggest_cluster = self.fps[biggest_cluster]
        return biggest_cluster, fps_biggest_cluster

    def biggest_cluster_metrics(self, biggest_cluster, fps_biggest_cluster, out_percentage=5, med_percentage=5):
        n_biggest_cluster = len(biggest_cluster)

        #Get iSIM of the biggest cluster
        isim_biggest_cluster = jt_isim(np.sum(fps_biggest_cluster, axis=0), n_biggest_cluster)

        #Get iSIM of the outliers, percentage defaulted to 5%
        outliers = outlier_sampling(fps_biggest_cluster, percentage=out_percentage, n_ary="JT")
        outliers_fps = fps_biggest_cluster[outliers]
        isim_outliers = jt_isim(np.sum(outliers_fps, axis=0), len(outliers))

        #Get iSIM of the medoids, percentage defaulted to 5%
        medoids = medoid_sampling(fps_biggest_cluster, percentage=med_percentage, n_ary="JT")
        medoids_fps = fps_biggest_cluster[medoids]
        isim_medoids = jt_isim(np.sum(medoids_fps, axis=0), len(medoids))

        #Get radius of biggest_cluster
        radius_value = radius(fps_biggest_cluster)

        return n_biggest_cluster, isim_biggest_cluster, isim_outliers, isim_medoids, radius_value

    def scaffold_analysis(self, biggest_cluster):
        smiles_biggest_cluster = self.smiles[biggest_cluster]
        scaffolds = [MurckoScaffold.MurckoScaffoldSmilesFromSmiles(smile) for smile in smiles_biggest_cluster]
        unique_scaffolds = set(scaffolds)
        n_unique = len(unique_scaffolds)
        scaffolds_fps = binary_fps(unique_scaffolds, fp_type = 'RDKIT')
        scaffolds_iSIM = jt_isim(np.sum(scaffolds_fps, axis = 0), n_unique)
        return n_unique, scaffolds_iSIM #Returning number of unique scaffolds and the iSIM

    def get_clusters(self):
        clusters = self.brc.get_cluster_mol_ids()
        return clusters

    def forward(self):
        print('Running Cluster Analysis: \n')
        print('Computing clustering metrics...')
        self.chi_val, self.dbi_val, self.dunn_val = self.clustering_metrics()
        print(f'Chi:{self.chi_val}')
        print(f'DBI:{self.dbi_val}')
        print(f'Dunn:{self.dunn_val}\n')

        print('Analyzing the biggest cluster...')
        biggest_cluster, fps_biggest_cluster = self.get_biggest_cluster()
        (self.n_biggest_cluster, self.isim_biggest_cluster, 
        self.isim_outliers, self.isim_medoids, self.radius_value) = self.biggest_cluster_metrics(biggest_cluster, fps_biggest_cluster)
        print(f'Size of biggest cluster: {self.n_biggest_cluster}')
        print(f'iSIM: {self.isim_biggest_cluster}')
        print(f'iSIM of the outliers: {self.isim_outliers}')
        print(f'iSIM of the medoids: {self.isim_medoids}')
        print(f'Radius of biggest_cluster: {self.radius_value} \n')

        print('Looking for unique scaffolds...')
        self.unique_scaffolds, self.scaffolds_iSIM = self.scaffold_analysis(biggest_cluster)
        print(f'Number of unique scaffolds: {self.unique_scaffolds}')
        print(f'iSIM of unique scaffolds: {self.scaffolds_iSIM} \n')
        
        print('Getting total clusters...')
        clusters = self.get_clusters()
        self.total_clusters = len(clusters)
        self.larger_clusters = [x for x in [len(cluster) > 10 for cluster in clusters] if x]
        print(f'Total number of clusters: {self.total_clusters}')
        print(f'Total number of clusters with more than 10 molecules: {len(self.larger_clusters)} \n')
        
        print('Analysis Complete!')
        
        return clusters
        #return chi_val, dbi_val, dunn_val, n_biggest_cluster, isim_biggest_cluster, isim_outliers, isim_medoids, radius_value


def create_pandas_dataframe(cluster_analysis_instance, method):
    data = {
        "Chi": cluster_analysis_instance.chi_val,
        "DBI": cluster_analysis_instance.dbi_val,
        "Dunn": cluster_analysis_instance.dunn_val,
        "Biggest Cluster Size": cluster_analysis_instance.n_biggest_cluster,
        "Biggest Cluster iSIM": cluster_analysis_instance.isim_biggest_cluster,
        "Outliers iSIM": cluster_analysis_instance.isim_outliers,
        "Medoids iSIM": cluster_analysis_instance.isim_medoids,
        "Cluster Radius": cluster_analysis_instance.radius_value,
        "Number of Unique Scaffolds": cluster_analysis_instance.unique_scaffolds,
        "Scaffolds iSIM": cluster_analysis_instance.scaffolds_iSIM,
        "Total Clusters": cluster_analysis_instance.total_clusters, 
        "Total Clusters > 10 mols": len(cluster_analysis_instance.larger_clusters)
    }

    # Convert to a DataFrame
    df = pd.DataFrame([data], index=[method])

    return df


def check_cluster_structure(brc):
    total_mol = 0
    mol_j_vals = []
    for index, node in enumerate(brc._get_leaves()):
        for c in node.subclusters_:
            mol_j = c.mol_indices
            total_mol+=c.n_samples_
            mol_j_vals.append(mol_j)
            assert c.n_samples_ == len(mol_j), f"Fatal! N_j:{c.n_samples_} and len(mol_j):{len(mol_j)} are incompatible."
    print('Check to ensure all N_j values are identical to all len(mol_j) values passed.')


    print('Total number of molecules:', total_mol)

    mol_j_vals = [i for cl in mol_j_vals for i in cl]

    unique = []
    for m in mol_j_vals:
        if m in unique:
            print(f'Fatal! Index {m} is repeating!')
        else:
            msg = 'There are no repeated indices.'
        unique.append(m)
    if msg:
        print(msg)

    print(f'The range of molecular indices is {min(mol_j_vals)} thru {max(mol_j_vals)}')
'''


**`vendor/bitbirch/bitbirch/cluster_control.py`**


In [ ]:
SOURCES['vendor/bitbirch/bitbirch/cluster_control.py'] = r'''import numpy as np

def jt_isim(c_total, n_objects):
    """iSIM Tanimoto calculation"""
    sum_kq = np.sum(c_total)
    sum_kqsq = np.dot(c_total, c_total)
    a = (sum_kqsq - sum_kq)/2
    return a/(a + n_objects * sum_kq - sum_kqsq)

def jt_pair(mol1, mol2):
    """Tanimoto between two molecules"""
    a = np.dot(mol1, mol2)
    return a/(np.sum(mol1) + np.sum(mol2) - a)

def jt_one_to_many(mol, mol_set):
    """Tanimoto similarities between a molecule and a set of molecules
    
    mol: np.array of a single molecule
    mol_set: np.array containing the fingerprints of a set of molecules
    """
    pop_counts = np.sum(mol_set, axis = 1)
    a = np.dot(mol_set, mol)
    sims = a / (np.sum(mol) + pop_counts - a)
    return sims

def calculate_centroid(linear_sum, n_samples):
    """Calculates centroid"""
    return np.floor(linear_sum / n_samples + 0.5)

def calculate_comp_sim(data):
    """Returns vector of complementary similarities"""
    n_objects = len(data) - 1
    c_total = np.sum(data, axis = 0)
    comp_matrix = c_total - data
    a = comp_matrix * (comp_matrix - 1)/2
    comp_sims = np.sum(a, axis = 1)/np.sum((a + comp_matrix * (n_objects - comp_matrix)), axis = 1)
    return comp_sims

def calculate_medoid(data):
    """Returns index of medoid"""
    return np.argmin(calculate_comp_sim(data))

def remove_singles(clusters, min_size = 1):
    """Remove singletons
    
    clusters : list of np arrays.
    min_size : int size below which clusters will be ignored
    """
    curated_clusters = []
    for c in clusters:
        if len(c) <= min_size:
            pass
        else:
            curated_clusters.append(c)
    return curated_clusters

def intra_sim(clust1, clust2):
    """Similarity between clusters from the iSIM of their union"""
    n1 = len(clust1)
    n2 = len(clust2)
    combined = np.sum(clust1, axis=0) + np.sum(clust2, axis=0)
    return jt_isim(combined, n1 + n2)

def inter_sim(clust1, clust2):
    """Similarity between clusters from the average distance between their elements"""
    n1 = len(clust1)
    n2 = len(clust2)
    n = n1 + n2
    c_total1 = np.sum(clust1, axis=0)
    c_total2 = np.sum(clust2, axis=0)
    combined = c_total1 + c_total2
    return (jt_isim(combined, n1 + n2) * n * (n-1) - (jt_isim(c_total1) * n1 * (n1 - 1) + jt_isim(c_total2) * n2 * (n2 - 1)))/(2 * n1 * n2)

def chi(clusters, reps = False, rep_type = "centroid", curated = False, min_size = 1):
    """Calinski-Harabasz index
    
    clusters : list of np.arrays containing the clusters
    
    reps : {bool, list} indicates if cluster representatives are given
    
    rep_type : type of representative, medoid or centroid
    
    curated : bool indicates if singletons have been removed
    
    min_size : int size below which clusters will be ignored
    
    Note
    ----
    Higher values are better
    
    """
    # n : total number of points
    n = 0
    
    # k : number of clusters
    k = len(clusters)
    
    # wcss : within-cluster sum of squares
    wcss = 0
    
    # bcss : between-cluster sum of squares
    bcss = 0
    
    if not curated:
        clusters = remove_singles(clusters, min_size)
    
    total_data = []
    for clust in clusters:
        for mol in clust:
            total_data.append(mol)
    total_data = np.array(total_data)
    if rep_type == 'centroid':
        linear_sum = np.sum(total_data, axis=0)
        n_samples = len(total_data)
        c = calculate_centroid(linear_sum, n_samples)
    elif rep_type == 'medoid':
        medoid = calculate_medoid(total_data)
        c = total_data[medoid]
    
    if not reps:
        for clust in clusters:
            n_samples = len(clust)
            n += n_samples
            if rep_type == 'centroid':
                linear_sum = np.sum(clust, axis=0)
                rep = calculate_centroid(linear_sum, n_samples)
            elif rep_type == 'medoid':
                medoid = calculate_medoid(clust)
                rep = clust[medoid]
            distances = 1 - jt_one_to_many(rep, clust)
            wcss += np.dot(distances, distances)
            bcss += n_samples * (1 - jt_pair(c, rep))**2
    else:
        for i, clust in enumerate(clusters):
            n_samples = len(clust)
            n += n_samples
            bcss += n_samples * (1 - jt_pair(c, reps[i]))**2
            distances = 1 - jt_one_to_many(reps[i], clust)
            wcss += np.dot(distances, distances)
    
    try:
        value = bcss * (n - k)/(wcss * (k - 1))
    except:
        value = 0
    return value

def dbi(clusters, reps = False, rep_type = "centroid", curated = False, min_size = 1):
    """Davies-Bouldin index
    
    clusters : list of np.arrays containing the clusters
    
    reps : {bool, list} indicates if cluster representatives are given
    
    rep_type : type of representative, medoid or centroid
    
    curated : bool indicates if singletons have been removed
    
    min_size : int size below which clusters will be ignored
    
    Note
    ----
    Lower values are better
    
    """
    if not curated:
        clusters = remove_singles(clusters, min_size)
    
    n = 0
    
    S = []
    
    if not reps:
        reps  = []
        for clust in clusters:
            n_samples = len(clust)
            n += n_samples
            if rep_type == 'centroid':
                linear_sum = np.sum(clust, axis=0)
                rep = calculate_centroid(linear_sum, n_samples)
            elif rep_type == 'medoid':
                medoid = calculate_medoid(clust)
                rep = clust[medoid]
            reps.append(rep)
            S.append(np.sum(1 - jt_one_to_many(rep, clust))/n_samples)
    else:
        for i, clust in enumerate(clusters):
            n_samples = len(clust)
            n += n_samples
            S.append(np.sum(1 - jt_one_to_many(reps[i], clust))/n_samples)
    
    db  = 0
    
    for i, clust in enumerate(clusters):
        d = []
        for j, other_clust in enumerate(clusters):
            if i == j:
                d.append(-1)
            else:
                Mij = 1 - jt_pair(reps[i], reps[j])
                Rij = (S[i] + S[j])/Mij
                d.append(Rij)
        db += max(d)
    
    try:
        value = db/n
    except:
        value = 0

    return value 

def dunn(clusters, within_sim = 'isim', cluster_sim = 'intra', reps = False, rep_type = 'centroid', curated = False, min_size = 1):
    """Dunn index
    
    within_sim : {'isim', 'mean'} type of intra cluster similarity
        isim : isim of the cluster
        mean : mean distances to the cluster representative
        
    cluster_sim : {'intra', 'inter'} type of similarity between clusters
        intra : intra_sim
        inter : inter_sim
    
    clusters : list of np.arrays containing the clusters
    
    reps : {bool, list} indicates if cluster representatives are given
    
    rep_type : type of representative, medoid or centroid
    
    curated : bool indicates if singletons have been removed
    
    min_size : int size below which clusters will be ignored
    
    Note
    ----
    Higher values are better
    
    """
    if not curated:
        clusters = remove_singles(clusters, min_size)
    
    D = []
    
    if within_sim == 'isim':
        for clust in clusters:
            n_samples = len(clust)
            linear_sum = np.sum(clust, axis=0)
            D.append(jt_isim(linear_sum, n_samples))
    elif intra_sim == 'mean':
        if not reps:
            for clust in clusters:
                n_samples = len(clust)
                if rep_type == 'centroid':
                    linear_sum = np.sum(clust, axis=0)
                    rep = calculate_centroid(linear_sum, n_samples)
                elif rep_type == 'medoid':
                    medoid = calculate_medoid(clust)
                    rep = clust[medoid]
                d = 1 - (jt_isim(linear_sum + rep, n_samples + 1) * (n_samples + 1) - jt_isim(linear_sum, n_samples) * (n_samples - 1))/2
                D.append(d)
        else:
            for i, clust in enumerate(clusters):
                n_samples = len(clust)
                d = 1 - (jt_isim(linear_sum + reps[i], n_samples + 1) * (n_samples + 1) - jt_isim(linear_sum, n_samples) * (n_samples - 1))/2
                D.append(d)
    
    if len(D) == 0:
        Dm = 0
    else:
        Dm = max(D)
    
    # initial min_d value could be any number > 1
    min_d = 3.08
    
    for i, clust1 in enumerate(clusters):
        for j, clust2 in enumerate(clusters):
            if i == j:
                pass
            else:
                if cluster_sim == 'intra':
                    dij = 1 - intra_sim(clust1, clust2)
                elif cluster_sim == 'inter':
                    dij = 1 - inter_sim(clust1, clust2)
                if dij < min_d:
                    min_d = dij
    
    try:
        value = min_d/Dm
    except:
        value = 0

    return value

def clust_dispersion(clusters, reps = False, rep_type = "centroid", curated = False, min_size = 1):
    """Cluster dispersion
    
    cd = isim(representatives)/<isim(clusters)>
    
    clusters : list of np.arrays containing the clusters
    
    reps : {bool, list} indicates if cluster representatives are given
    
    rep_type : type of representative, medoid or centroid
    
    curated : bool indicates if singletons have been removed
    
    min_size : int size below which clusters will be ignored
    
    Note
    ----
    Lower values are better TODO: DOUBLE CHECK THIS!!!!
    """
    if not curated:
        clusters = remove_singles(clusters, min_size)
    
    n_clusters = len(clusters)
    
    if n_clusters == 1:
        return -1
    
    isim_clusts = 0
    
    if not reps:
        representatives = []
    else:
        representatives = reps
    
    for clust in clusters:
        n_samples = len(clust)
        linear_sum = np.sum(clust, axis=0)
        isim_clusts += jt_isim(linear_sum, n_samples)
        if not reps:
            if rep_type == 'centroid':
                representatives.append(calculate_centroid(linear_sum, n_samples))
            elif rep_type == 'medoid':
                medoid = calculate_medoid(clust)
                representatives.append(clust[medoid])
    
    av_isim_clusts = isim_clusts/n_clusters
    
    representatives = np.array(representatives)
    rep_linear_sum = np.sum(representatives, axis=0)
    rep_isim = jt_isim(rep_linear_sum, n_clusters)

    try:
        value =  rep_isim/av_isim_clusts
    except:
        value = 0
    
    return value    

def birch_analysis(birch, fp_np, min_size = 10):
    # Minimum number of elements in a set
    min_size = min_size

    # List that will contain the BIRCH results
    birch_clusters = []
    
    # Getting leaves from the BIRCH tree
    leaves = birch._get_leaves()
    
    # Populating BIRCH results' list
    for leave in leaves:
        for subcluster in leave.subclusters_:
            if len(subcluster.mol_indices) <= min_size:
                pass
            else:
                birch_clusters.append(fp_np[subcluster.mol_indices])

    return birch_clusters

def radius(fps):
    c_sum = np.sum(fps, axis = 0)
    n = len(fps)
    centroid = calculate_centroid(c_sum, n)

    jt = jt_isim(c_sum, n_objects=n)
    jt_c = jt_isim(np.sum(fps, axis = 0) + centroid, n_objects=n+1)

    r = 1 - ((n+1)*jt_c-(n-1)*jt)/2

    return r
'''


**`vendor/bitbirch/bitbirch/plotting_utils.py`**


In [ ]:
SOURCES['vendor/bitbirch/bitbirch/plotting_utils.py'] = r'''import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import plotly.graph_objects as go
from bitbirch.bitbirch import jt_isim, BitBirch
from bitbirch.cluster_analysis import ClusterAnalysis
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from rdkit.Chem import Draw
from rdkit import Chem

def tsne_plot(brc, fps, method='', title=None):
    clusters = brc.get_cluster_mol_ids()
    sort_clusters = sorted(clusters, key=lambda x: len(x), reverse=True)[:20]
    # #     # Map each mol ID to its cluster ID
    n_molecules = sum([len(x) for x in sort_clusters])


    cluster_labels = [0] * n_molecules

    fps_clusters = []#[0] * n_molecules
    cluster_labels = []#[0] * n_molecules

    for cluster_id, cluster in enumerate(sort_clusters):
        for idx in cluster:
            fps_clusters.append(fps[idx])
            cluster_labels.append(int(cluster_id))


    scaler = StandardScaler()
    fps_scaled = scaler.fit_transform(fps_clusters)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    fps_tsne = tsne.fit_transform(fps_scaled)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(fps_tsne[:, 0], fps_tsne[:, 1], c=cluster_labels, cmap='tab20', alpha=0.9)
    cbar = plt.colorbar(scatter, label="Cluster ID")
    cbar.set_ticks(np.arange(20))  # Set ticks at the center of each color
    cbar.set_ticklabels(np.arange(1, 21))  # Set labels from 1 to 20
    plt.xlabel("t-SNE Component 1")
    plt.ylabel("t-SNE Component 2")
    if title:
        plt.title(f"t-SNE of Top 20 Largest Clusters for {title}")
    else:
        plt.title(f"t-SNE of Top 20 Largest Clusters for {method[0].upper() + method[1:]}")
    plt.show()

def mol_relocation_plots(brc_1, brc_2, top_n = 2, title=None):
    
    clusters_1 = brc_1.get_cluster_mol_ids()
    clusters_2 = brc_2.get_cluster_mol_ids()

    list_1 = sorted(clusters_1, key=lambda x: len(x), reverse=True)[:top_n]
    list_2 = sorted(clusters_2, key=lambda x: len(x), reverse=True)[:top_n]

    # Map elements to clusters
    cluster_map_1 = {elem: f"Cluster 1-{i}" for i, cluster in enumerate(list_1) for elem in cluster}
    cluster_map_2 = {elem: f"Cluster 2-{i}" for i, cluster in enumerate(list_2) for elem in cluster}

    # Track flow of elements between clusters
    flows = {}
    for elem in cluster_map_1:
        source = cluster_map_1[elem]
        target = cluster_map_2.get(elem, "Other smaller cluster")
        flow_key = (source, target)
        flows[flow_key] = flows.get(flow_key, 0) + 1

    # Prepare data for Sankey diagram
    labels = list(set([src for src, _ in flows] + [tgt for _, tgt in flows]))
    source_indices = [labels.index(src) for src, _ in flows]
    target_indices = [labels.index(tgt) for _, tgt in flows]
    values = list(flows.values())

    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=labels
        ),
        link=dict(
            source=source_indices,
            target=target_indices,
            value=values
        )
    )])

    # Update layout
    fig.update_layout(
        title_text=title,
        font_size=10
    )
    fig.show()

def get_cluster_cai(brc, data, smiles):
        cai = ClusterAnalysis(brc, data, smiles)
        clusters = cai.get_clusters()
        return clusters, cai


def get_plot_metrics(clusters, cai, fps, smiles, top=20):
    sort_clusters = sort_clusters = sorted(clusters, key=lambda x: len(x), reverse=True)
    biggest_clusters = sort_clusters[:top]
    fps_biggest_clusters = [[fps[i] for i in c] for c in biggest_clusters]

    n_mol_bcs = [len(x) for x in biggest_clusters] # #molecules


    n_scaff = [cai.scaffold_analysis(c)[0] for c in biggest_clusters]
    isim_scaff = [cai.scaffold_analysis(c)[1] for c in biggest_clusters]


    isim_clusters = [jt_isim(np.sum(fpsc, axis=0), n) for fpsc,n in zip(fps_biggest_clusters, n_mol_bcs)]

    return n_mol_bcs, n_scaff, isim_scaff, isim_clusters


def save_metrics_to_csv(brc, fps, smiles, top=20, method=''):
    clusters, cai = get_cluster_cai(brc, fps, smiles)

    # Extract metrics
    n_mol_bcs, n_scaff, isim_scaff, isim_clusters  = get_plot_metrics(clusters, cai, fps, smiles, top=top)

    # Create DataFrame
    df = pd.DataFrame({
        "Cluster": list(range(1, top + 1)),
        "Number of Molecules": n_mol_bcs,
        "Number of Unique Scaffolds": n_scaff,
        "iSIM of Unique Scaffolds": isim_scaff,
        "iSIM": isim_clusters,
    })

    # Ensure output directory exists
    output_dir = "data/"
    os.makedirs(output_dir, exist_ok=True)

    # Save to CSV
    output_file = os.path.join(output_dir, f"{method}_cluster_metrics.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved cluster metrics for {method} to {output_file}")

def init_plot(method, title=None):
    csv_file = f"data/{method}_cluster_metrics.csv"

    if not os.path.exists(csv_file):
        print(f"Error: CSV file not found at {csv_file}")
    else:
        # Read CSV
        df = pd.read_csv(csv_file)

        # Ensure Cluster is treated as a categorical variable
        df["Cluster"] = df["Cluster"].astype(str)

        # Create figure and axes
        plt.figure(figsize=(10, 5))

        # Plot the number of molecules
        plt.bar(df["Cluster"], df["Number of Molecules"], color="blue", alpha=1, label="Molecules")
        
        # Annotate the number of molecules
        for i, mol in enumerate(df["Number of Molecules"]):
            plt.text(i, mol, f"{mol}", ha="center", va="bottom", fontsize=10, color="black")

        # Plot the number of unique scaffolds
        plt.bar(df["Cluster"], df["Number of Unique Scaffolds"], color="orange", alpha=1, label="Scaffolds")

        # Annotate the number of unique scaffolds
        for i, scaf in enumerate(df["Number of Unique Scaffolds"]):
            plt.text(i, scaf, f"{scaf}", ha="center", va="bottom", fontsize=8, color="white")

        # Labels
        plt.xlabel("Cluster", fontsize=12)
        plt.ylabel("Count", fontsize=12)
        plt.xticks(range(len(df["Cluster"])))
        plt.legend(loc="upper right")
     
     
        # Plot iSIM
        plt.twinx()
        plt.plot(df["Cluster"], df["iSIM"], color="green", marker="o", label="iSIM", linewidth=2)
        plt.ylabel("iSIM", fontsize=12)
        plt.yticks(np.arange(0, 1.1, 0.1))
        plt.ylim(0, 1)

        plt.tight_layout()

        plt.legend(loc="upper right", bbox_to_anchor=(0.80, 1))

        # Title
        if title:
            plt.title(f'Top 20 Cluster Metrics for {title}', fontsize=14)
        else:
            plt.title(f'Top 20 Cluster Metrics for {method[0].upper() + method[1:]}', fontsize=14)


        plt.show() 

def mol_visualization(smiles, brc, cluster = 0):
    clustered_ids = brc.get_cluster_mol_ids()

    mols = [Chem.MolFromSmiles(smiles[i]) for i in clustered_ids[cluster-1]]
    print('Number of molecules:', len(mols))

    for i in range(0, len(mols), 30):

        img_1 = Draw.MolsToGridImage(mols[i:i+30], molsPerRow=5)

        # Save the images from the .data attribute
        with open(f'cluster_{cluster}_{i}.png', 'wb') as f:
            f.write(img_1.data)'''


**`vendor/bitbirch/bitbirch/pruning.py`**


In [ ]:
SOURCES['vendor/bitbirch/bitbirch/pruning.py'] = (
    '# BitBIRCH is an open-source clustering module based on iSIM\n'
    '#\n'
    '# Please, cite the BitBIRCH paper: https://www.biorxiv.org/content/10.1101/2024.08.10.607459v1\n'
    '#\n'
    '# BitBIRCH is free software; you can redistribute it and/or modify\n'
    '# it under the terms of the GNU Lesser General Public License as published by\n'
    '# the Free Software Foundation, version 3.\n'
    '#\n'
    '# BitBIRCH is distributed in the hope that it will be useful,\n'
    '# but WITHOUT ANY WARRANTY; without even the implied warranty of\n'
    '# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the\n'
    '# GNU Lesser General Public License for more details.\n'
    '#\n'
    '# BitBIRCH authors (PYTHON): Ramon Alain Miranda Quintana <ramirandaq@gmail.com>, <quintana@chem.ufl.edu>\n'
    '#                            Vicky (Vic) Jung <jungvicky@ufl.edu>\n'
    '#                            Kenneth Lopez Perez <klopezperez@chem.ufl.edu>\n'
    '#                            Kate Huddleston <kdavis2@chem.ufl.edu>\n'
    '#\n'
    '# BitBIRCH License: LGPL-3.0 https://www.gnu.org/licenses/lgpl-3.0.en.html#license-text\n'
    '\n'
    'import numpy as np\n'
    'import copy\n'
    '\n'
    '# when deleting from init_centroids, we need to make sure that the elements are shifted\n'
    '# to the left to fill the space and make sure the empty space on the right is 0 because\n'
    '# init_centroids is an array with a pre-defined size\n'
    'def shift_delete(array, index):\n'
    '    for i in range(index, len(array)-1):\n'
    '        array[i]=array[i+1]\n'
    '    array[-1]=0\n'
    '    \n'
    'def recursivelyTraverse(node, children):\n'
    '    if node is None:\n'
    '        return\n'
    '    \n'
    '    for i in node.subclusters_:\n'
    '        children.append(i)\n'
    '        recursivelyTraverse(i.child_, children)\n'
    '\n'
    'def pSubPath(subcluster, pSubs, brc):\n'
    '    # figure out the path of subclusters from leaf level to root\n'
    '\n'
    '    if subcluster is None:\n'
    '        return \n'
    '    \n'
    '    if subcluster.parent_ is None:\n'
    '        pSubs.append(brc.root_.subclusters_.index(subcluster))\n'
    '    else:\n'
    '        pSubs.append(subcluster.parent_.child_.subclusters_.index(subcluster))\n'
    '    \n'
    '    pSubPath(subcluster.parent_, pSubs, brc)\n'
    '\n'
    '# in order to preserve the insertion order of pNodes, make sure python is updated to 3.7+\n'
    'def pNodePath(pSubs, pNodes, root):\n'
    '    temp=root\n'
    '\n'
    '    for i in pSubs[:0:-1]:\n'
    '        index=temp.subclusters_.index(i)\n'
    '        pNodes[temp]=index\n'
    '        temp=temp.subclusters_[index].child_\n'
    '\n'
    '\n'
    'def calc_centroid(linear_sum, n_samples):\n'
    '    """Calculates centroid\n'
    '    \n'
    '    Parameters\n'
    '    ----------\n'
    '    \n'
    '    linear_sum : np.ndarray\n'
    '                 Sum of the elements column-wise\n'
    '    n_samples : int\n'
    '                Number of samples\n'
    '                \n'
    '    Returns\n'
    '    -------\n'
    '    centroid : np.ndarray\n'
    '               Centroid fingerprints of the given set\n'
    '    """\n'
    '    return np.where(linear_sum >= n_samples * 0.5 , 1, 0)\n'
    '\n'
    'def lazyPrune(leaf_index, subc_index, brc):\n'
    '    #from bitbirch_tools import bitbirch_double_link_tolerance as bitbirch\n'
    '    # path from leaf level to root, indices are marked \n'
    '    pSubs=[]\n'
    '    pSubPath(brc._get_leaves()[leaf_index].subclusters_[subc_index], pSubs, brc)\n'
    '\n'
    '    # traverse all the way down to the leaf node\n'
    '    node=brc.root_\n'
    '    for index in pSubs[:0:-1]:\n'
    '        node=node.subclusters_[index].child_\n'
    '\n'
    '    \n'
    '    # information needed to update all preceding subclusters\n'
    '    cluster_ls = node.subclusters_[subc_index].linear_sum_\n'
    '    cluster_n = node.subclusters_[subc_index].n_samples_\n'
    '    cluster_mol_indices = node.subclusters_[subc_index].mol_indices\n'
    '    \n'
    "    '''# IMPORTANT: tree must have at least three levels, at least that is what is assumed when inputting dataset\n"
    '    parent_sub = node.subclusters_[0].parent_       # needed to traverse node upwards\n'
    "    parent_node = parent_sub.parent_.child_         # needed to reassign leaf nodes'''\n"
    '\n'
    '    parent_sub = node.subclusters_[0].parent_\n'
    '    if parent_sub.parent_ is None:\n'
    '        parent_node = brc.root_\n'
    '    else:\n'
    '        parent_node = parent_sub.parent_.child_\n'
    '    \n'
    '    # eliminate the biggest subcluster from the leaf node\n'
    '    node.subclusters_.pop(subc_index)\n'
    '    node.centroids_ = np.delete(node.centroids_, subc_index, 0)\n'
    '    shift_delete(node.init_centroids_, subc_index)\n'
    '\n'
    '    if len(node.subclusters_) == 0:\n'
    '        parent_sub.child_ = None\n'
    '\n'
    '    # rearrange leaf pointers if need to\n'
    '\n'
    '    # if the leaf node disappears, we assign parent node to replace leaf node\n'
    '    num_childs=0\n'
    '    for i in parent_node.subclusters_:\n'
    '        if i.child_ is not None:\n'
    '            num_childs+=1\n'
    '    if num_childs == 0:\n'
    '        parent_node.is_leaf = True\n'
    '        if node.prev_leaf_ is not None:\n'
    '            node.prev_leaf_.next_leaf_ = parent_node\n'
    '        parent_node.prev_leaf_ = node.prev_leaf_\n'
    '        parent_node.next_leaf_ = node.next_leaf_\n'
    '        if node.next_leaf_ is not None:\n'
    '            node.next_leaf_.prev_leaf_ = parent_node\n'
    '\n'
    '    # leaf node disappears and there are no reassignments\n'
    '    elif len(node.subclusters_) == 0:\n'
    '        if node.prev_leaf_ is not None:\n'
    '            node.prev_leaf_.next_leaf_ = node.next_leaf_\n'
    '        if node.next_leaf_ is not None:\n'
    '            node.next_leaf_.prev_leaf_ = node.prev_leaf_\n'
    '\n'
    '\n'
    '    if parent_sub.parent_ is None:\n'
    '        node = brc.root_\n'
    '    else:\n'
    '        node = parent_sub.parent_.child_ \n'
    '\n'
    '  \n'
    '    # here, we update all the preceding subclusters all the way up to the root\n'
    '    for index in pSubs[1:]:\n'
    '        sub=node.subclusters_[index]\n'
    ' \n'
    '        sub.linear_sum_ -= cluster_ls\n'
    '        sub.n_samples_ -= cluster_n\n'
    '        sub.centroid_ = calc_centroid(sub.linear_sum_, sub.n_samples_)\n'
    '        sub.mol_indices = [x for x in sub.mol_indices if x not in cluster_mol_indices]\n'
    '        \n'
    '        node.centroids_[index] = sub.centroid_\n'
    '        node.init_centroids_[index] = sub.centroid_\n'
    '\n'
    '        if sub.parent_ is None:\n'
    '            break\n'
    '        elif sub.parent_.parent_ is None:\n'
    '            node=brc.root_\n'
    '        else:\n'
    '            node=sub.parent_.parent_.child_\n'
    '\n'
    '    return cluster_mol_indices\n'
    '\n'
)


**`vendor/bitbirch/bitbirch/refinement.py`**


In [ ]:
SOURCES['vendor/bitbirch/bitbirch/refinement.py'] = r'''# BitBIRCH is an open-source clustering module based on iSIM
#
# Please, cite the BitBIRCH paper: https://www.biorxiv.org/content/10.1101/2024.08.10.607459v1
#
# BitBIRCH is free software; you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, version 3.
#
# BitBIRCH is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU Lesser General Public License for more details.
#
# BitBIRCH authors (PYTHON): Ramon Alain Miranda Quintana <ramirandaq@gmail.com>, <quintana@chem.ufl.edu>
#                            Vicky (Vic) Jung <jungvicky@ufl.edu>
#                            Kenneth Lopez Perez <klopezperez@chem.ufl.edu>
#                            Kate Huddleston <kdavis2@chem.ufl.edu>
#
# BitBIRCH License: LGPL-3.0 https://www.gnu.org/licenses/lgpl-3.0.en.html#license-text

import bitbirch.bitbirch as bb


##Functions##
#add docstrings to functions
def radius(fps, threshold = 0.65, branching_factor=50):
    """Runnign radius birch; classic birch"""
    bb.set_merge('radius')
    brc = bb.BitBirch(branching_factor=branching_factor, threshold=threshold)
    brc.fit(fps)
    return brc

def diameter(fps, threshold = 0.65):
    bb.set_merge('diameter')
    brc = bb.BitBirch(branching_factor=50, threshold=threshold)
    brc.fit(fps)
    return brc

def diameter_prune(fps, threshold = 0.65):
    bb.set_merge('diameter')
    brc = bb.BitBirch(branching_factor=50, threshold=threshold)
    brc.fit(fps, singly=False)
    brc.prune(fps)
    return brc

def diameter_prune_tolerance(fps, tol=0.05, threshold = 0.65):
    bb.set_merge('diameter')
    brc = bb.BitBirch(branching_factor=50, threshold=threshold)
    brc.fit(fps, singly=False)
    bb.set_merge('tolerance', tolerance=tol)
    brc.prune(fps)
    return brc

def diameter_prune_reassign(fps, threshold = 0.65):
    bb.set_merge('diameter')
    brc = bb.BitBirch(branching_factor=50, threshold=threshold)  
    brc.fit(fps, singly=False)
    brc.prune(fps)
    brc.reassign(fps)
    return brc

def diameter_prune_tolerance_reassign(fps, tol=0.05, threshold = 0.65):
    bb.set_merge('diameter')
    brc = bb.BitBirch(branching_factor=50, threshold=threshold)
    brc.fit(fps, singly=False)
    bb.set_merge('tolerance', tolerance=tol)
    brc.prune(fps)
    brc.reassign(fps)
    return brc

def BFs_reclustering(fps, init_threshold = 0.65, second_threshold = 0.7, second_tolerance = 0.0):
    # Do the diameter clustering
    bb.set_merge('diameter')
    brc = bb.BitBirch(branching_factor=50, threshold=init_threshold)
    brc.fit(fps, singly=True)

    # Extract the BFs for the second clustering
    BFs = brc.prepare_BFs(fps)

    # Do the second clustering
    bb.set_merge('tolerance', tolerance=second_tolerance)
    brc = bb.BitBirch(branching_factor=50, threshold=second_threshold)
    brc.fit_BFs(BFs) # Note that we fit the BFs, not the fps

    return brc'''


**`vendor/bitbirch/LICENSE`**


In [ ]:
SOURCES['vendor/bitbirch/LICENSE'] = r'''GNU LESSER GENERAL PUBLIC LICENSE
                       Version 3, 29 June 2007

 Copyright (C) 2007 Free Software Foundation, Inc. <https://fsf.org/>
 Everyone is permitted to copy and distribute verbatim copies
 of this license document, but changing it is not allowed.


  This version of the GNU Lesser General Public License incorporates
the terms and conditions of version 3 of the GNU General Public
License, supplemented by the additional permissions listed below.

  0. Additional Definitions.

  As used herein, "this License" refers to version 3 of the GNU Lesser
General Public License, and the "GNU GPL" refers to version 3 of the GNU
General Public License.

  "The Library" refers to a covered work governed by this License,
other than an Application or a Combined Work as defined below.

  An "Application" is any work that makes use of an interface provided
by the Library, but which is not otherwise based on the Library.
Defining a subclass of a class defined by the Library is deemed a mode
of using an interface provided by the Library.

  A "Combined Work" is a work produced by combining or linking an
Application with the Library.  The particular version of the Library
with which the Combined Work was made is also called the "Linked
Version".

  The "Minimal Corresponding Source" for a Combined Work means the
Corresponding Source for the Combined Work, excluding any source code
for portions of the Combined Work that, considered in isolation, are
based on the Application, and not on the Linked Version.

  The "Corresponding Application Code" for a Combined Work means the
object code and/or source code for the Application, including any data
and utility programs needed for reproducing the Combined Work from the
Application, but excluding the System Libraries of the Combined Work.

  1. Exception to Section 3 of the GNU GPL.

  You may convey a covered work under sections 3 and 4 of this License
without being bound by section 3 of the GNU GPL.

  2. Conveying Modified Versions.

  If you modify a copy of the Library, and, in your modifications, a
facility refers to a function or data to be supplied by an Application
that uses the facility (other than as an argument passed when the
facility is invoked), then you may convey a copy of the modified
version:

   a) under this License, provided that you make a good faith effort to
   ensure that, in the event an Application does not supply the
   function or data, the facility still operates, and performs
   whatever part of its purpose remains meaningful, or

   b) under the GNU GPL, with none of the additional permissions of
   this License applicable to that copy.

  3. Object Code Incorporating Material from Library Header Files.

  The object code form of an Application may incorporate material from
a header file that is part of the Library.  You may convey such object
code under terms of your choice, provided that, if the incorporated
material is not limited to numerical parameters, data structure
layouts and accessors, or small macros, inline functions and templates
(ten or fewer lines in length), you do both of the following:

   a) Give prominent notice with each copy of the object code that the
   Library is used in it and that the Library and its use are
   covered by this License.

   b) Accompany the object code with a copy of the GNU GPL and this license
   document.

  4. Combined Works.

  You may convey a Combined Work under terms of your choice that,
taken together, effectively do not restrict modification of the
portions of the Library contained in the Combined Work and reverse
engineering for debugging such modifications, if you also do each of
the following:

   a) Give prominent notice with each copy of the Combined Work that
   the Library is used in it and that the Library and its use are
   covered by this License.

   b) Accompany the Combined Work with a copy of the GNU GPL and this license
   document.

   c) For a Combined Work that displays copyright notices during
   execution, include the copyright notice for the Library among
   these notices, as well as a reference directing the user to the
   copies of the GNU GPL and this license document.

   d) Do one of the following:

       0) Convey the Minimal Corresponding Source under the terms of this
       License, and the Corresponding Application Code in a form
       suitable for, and under terms that permit, the user to
       recombine or relink the Application with a modified version of
       the Linked Version to produce a modified Combined Work, in the
       manner specified by section 6 of the GNU GPL for conveying
       Corresponding Source.

       1) Use a suitable shared library mechanism for linking with the
       Library.  A suitable mechanism is one that (a) uses at run time
       a copy of the Library already present on the user's computer
       system, and (b) will operate properly with a modified version
       of the Library that is interface-compatible with the Linked
       Version.

   e) Provide Installation Information, but only if you would otherwise
   be required to provide such information under section 6 of the
   GNU GPL, and only to the extent that such information is
   necessary to install and execute a modified version of the
   Combined Work produced by recombining or relinking the
   Application with a modified version of the Linked Version. (If
   you use option 4d0, the Installation Information must accompany
   the Minimal Corresponding Source and Corresponding Application
   Code. If you use option 4d1, you must provide the Installation
   Information in the manner specified by section 6 of the GNU GPL
   for conveying Corresponding Source.)

  5. Combined Libraries.

  You may place library facilities that are a work based on the
Library side by side in a single library together with other library
facilities that are not Applications and are not covered by this
License, and convey such a combined library under terms of your
choice, if you do both of the following:

   a) Accompany the combined library with a copy of the same work based
   on the Library, uncombined with any other library facilities,
   conveyed under the terms of this License.

   b) Give prominent notice with the combined library that part of it
   is a work based on the Library, and explaining where to find the
   accompanying uncombined form of the same work.

  6. Revised Versions of the GNU Lesser General Public License.

  The Free Software Foundation may publish revised and/or new versions
of the GNU Lesser General Public License from time to time. Such new
versions will be similar in spirit to the present version, but may
differ in detail to address new problems or concerns.

  Each version is given a distinguishing version number. If the
Library as you received it specifies that a certain numbered version
of the GNU Lesser General Public License "or any later version"
applies to it, you have the option of following the terms and
conditions either of that published version or of any later version
published by the Free Software Foundation. If the Library as you
received it does not specify a version number of the GNU Lesser
General Public License, you may choose any version of the GNU Lesser
General Public License ever published by the Free Software Foundation.

  If the Library as you received it specifies that a proxy can decide
whether future versions of the GNU Lesser General Public License shall
apply, that proxy's public statement of acceptance of any version is
permanent authorization for you to choose that version for the
Library.
'''


**`vendor/bitbirch/README.md`**


In [ ]:
SOURCES['vendor/bitbirch/README.md'] = r'''# BitBirch: Efficient Clustering of Large Molecular Libraries

**Miranda-Quintana Group**  
*University of Florida*

![BitBirch Logo](data/BitBIRCH.png)

**NOTE**: For our cutting-edge, API-compatible, fast and memory-efficient implementation
of the BitBIRCH algorithm for huge molecular libraries check out the [BitBirch-Lean
(bblean)](https://github.com/mqcomplab/bblean) repository.

## Reference  

[DOI: 10.1039/D5DD00030K](https://doi.org/10.1039/D5DD00030K)

## Installation  

```bash
git clone https://github.com/mqcomplab/bitbirch.git
cd bitbirch
pip install -e .
```
'''


**`vendor/bitbirch/pyproject.toml`**


In [ ]:
SOURCES['vendor/bitbirch/pyproject.toml'] = r'''[project]
name = "bitbirch"
license = "GPL-3.0-only"
description = "BitBirch: efficient clustering of large molecular libraries."
requires-python = ">=3.8"
version = "0.2"
dependencies = [
    "numpy<2,>=1.20",
    "matplotlib",
    "pandas",
    "rdkit",
    "scipy",
    "seaborn",
    "scikit-learn",
    "plotly",
    "iSIM @ git+https://github.com/mqcomplab/iSIM.git",
    "useful_rdkit_utils @ git+https://github.com/PatWalters/useful_rdkit_utils.git"
]
readme = "README.md"

[tool.setuptools.dynamic]
readme = {file = ["README.md"]}

[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project.urls]
homepage = "https://github.com/mqcomplab/bitbirch.git"
repository = "https://github.com/mqcomplab/bitbirch.git"
documentation = "https://github.com/mqcomplab/bitbirch.git"

[tool.setuptools.packages.find]
where = ["."]
include = ["bitbirch*"]
'''


**`vendor/bitbirch/provenance.json`**


In [ ]:
SOURCES['vendor/bitbirch/provenance.json'] = r'''{
    "repository":  "https://github.com/mqcomplab/bitbirch",
    "commit":  "311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f",
    "commit_date":  "2025-10-21T18:48:53Z",
    "retrieved_at_utc":  "2026-09-21T14:12:40.5579062Z",
    "files":  [
                  {
                      "path":  "LICENSE",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/LICENSE",
                      "sha256":  "6557ADC4BC1D59232E27567A185DD70CF2ECCA3154018B0BE61EA29CF9E814E2"
                  },
                  {
                      "path":  "README.md",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/README.md",
                      "sha256":  "1F5C9627B1D3E81767E91EEFC54E3AD56542CD145EEC1BB4DFF57E88634735BB"
                  },
                  {
                      "path":  "pyproject.toml",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/pyproject.toml",
                      "sha256":  "3EA9D413D619C538FD243FC384E17D7A3A31EA31854A74FB3DEABF30F859CF68"
                  },
                  {
                      "path":  "bitbirch/__init__.py",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/bitbirch/__init__.py",
                      "sha256":  "E3B0C44298FC1C149AFBF4C8996FB92427AE41E4649B934CA495991B7852B855"
                  },
                  {
                      "path":  "bitbirch/bitbirch.py",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/bitbirch/bitbirch.py",
                      "sha256":  "FF88E988306854FF5A45EDCD9E656A6AE79F398DD61A4B60ADE89F61D6D33B22"
                  },
                  {
                      "path":  "bitbirch/cluster_analysis.py",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/bitbirch/cluster_analysis.py",
                      "sha256":  "23B8AAC5CA5403AB80EE29EF8A4359A448A311929711875CA669440E7C40D0DC"
                  },
                  {
                      "path":  "bitbirch/cluster_control.py",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/bitbirch/cluster_control.py",
                      "sha256":  "D681B94D75ECEA6F9D064582A31361EF609613356F5C33974BE4EB8EB6EB2C62"
                  },
                  {
                      "path":  "bitbirch/plotting_utils.py",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/bitbirch/plotting_utils.py",
                      "sha256":  "D62BC65DAD554D61A2F594FD276AB2AE0BE2C1E347C9148E58985B574D9F6812"
                  },
                  {
                      "path":  "bitbirch/pruning.py",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/bitbirch/pruning.py",
                      "sha256":  "9EB65EB920112A10767D516ABC5CA6064EEE578B5969FF1489370EF1719CE0F5"
                  },
                  {
                      "path":  "bitbirch/refinement.py",
                      "url":  "https://raw.githubusercontent.com/mqcomplab/bitbirch/311fbf7dcc8ba019a53f58e75d40ebce6a3bc76f/bitbirch/refinement.py",
                      "sha256":  "BA79958E0E568DCCC6B18D2294EA32E59497BD3E549F8E2095058E9AD4FCA5B9"
                  }
              ]
}
'''


In [ ]:
SOURCE_METADATA = json.loads(r'''{
  "plan.json": {
    "sha256": "6c4ad080b28d476b6d50c1610f74f01504612647390fba26e76c3addc7a1692f",
    "bytes": 8514,
    "utf8_bom": false,
    "crlf_lines": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48,
      49,
      50,
      51,
      52,
      53,
      54,
      55,
      56,
      57,
      58,
      59,
      60,
      61,
      62,
      63,
      64,
      65,
      66,
      67,
      68,
      69,
      70,
      71,
      72,
      73,
      74,
      75,
      76,
      77,
      78,
      79,
      80,
      81,
      82,
      83,
      84,
      85,
      86,
      87,
      88,
      89,
      90,
      91,
      92,
      93,
      94,
      95,
      96,
      97,
      98,
      99,
      100,
      101,
      102,
      103,
      104,
      105,
      106,
      107,
      108,
      109,
      110,
      111,
      112,
      113,
      114,
      115,
      116,
      117,
      118,
      119,
      120,
      121,
      122,
      123,
      124,
      125,
      126,
      127,
      128,
      129,
      130,
      131,
      132,
      133,
      134,
      135,
      136,
      137,
      138,
      139,
      140,
      141,
      142,
      143,
      144,
      145,
      146,
      147,
      148,
      149,
      150,
      151,
      152,
      153,
      154,
      155,
      156,
      157,
      158,
      159,
      160,
      161,
      162,
      163,
      164,
      165,
      166,
      167,
      168,
      169,
      170,
      171,
      172,
      173,
      174,
      175,
      176,
      177,
      178,
      179,
      180,
      181,
      182,
      183,
      184,
      185,
      186,
      187,
      188,
      189,
      190,
      191,
      192,
      193,
      194,
      195,
      196,
      197,
      198,
      199,
      200,
      201,
      202,
      203,
      204,
      205,
      206,
      207,
      208,
      209,
      210,
      211,
      212,
      213,
      214,
      215,
      216,
      217,
      218,
      219,
      220,
      221,
      222,
      223,
      224,
      225,
      226,
      227,
      228,
      229,
      230,
      231,
      232,
      233,
      234,
      235,
      236,
      237,
      238,
      239,
      240,
      241,
      242,
      243,
      244,
      245,
      246,
      247,
      248,
      249,
      250,
      251,
      252,
      253,
      254,
      255,
      256,
      257,
      258,
      259,
      260,
      261,
      262,
      263,
      264,
      265,
      266,
      267,
      268,
      269,
      270,
      271,
      272,
      273,
      274,
      275,
      276,
      277,
      278,
      279,
      280,
      281,
      282,
      283,
      284,
      285,
      286,
      287,
      288,
      289,
      290,
      291
    ]
  },
  "records/repetition_amendment.json": {
    "sha256": "6c4d8486fa85b5c2cff77c6902b1cc884e4309e329a4c4fb509ac7cc261245af",
    "bytes": 1631,
    "utf8_bom": false,
    "crlf_lines": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48,
      49,
      50,
      51,
      52,
      53,
      54,
      55,
      56,
      57,
      58,
      59,
      60,
      61,
      62,
      63,
      64,
      65,
      66,
      67,
      68,
      69,
      70,
      71,
      72,
      73,
      74,
      75,
      76,
      77,
      78,
      79,
      80,
      81,
      82,
      83,
      84,
      85,
      86,
      87,
      88,
      89,
      90,
      91,
      92,
      93,
      94,
      95,
      96,
      97,
      98,
      99,
      100,
      101,
      102,
      103,
      104,
      105,
      106
    ]
  },
  "score_and_plot.py": {
    "sha256": "57b89a67d5202dd9c9bc4366a84a3588b54fd4b564c848c483a424eeb66107a4",
    "bytes": 8436,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/generate_data_snapshot.py": {
    "sha256": "519fd5bad59501c493e8189d3dc13a6f62c5abe917a9cc90e9cc82d85cd14d0d",
    "bytes": 2131,
    "utf8_bom": false,
    "crlf_lines": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48
    ]
  },
  "code/prepare_dataset.py": {
    "sha256": "3164e88918693c8b2cf955d9d81ab30f55ad5664f2e7bbe1ee4f0bf0f8623727",
    "bytes": 6215,
    "utf8_bom": false,
    "crlf_lines": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48,
      49,
      50,
      51,
      52,
      53,
      54,
      55,
      56,
      57,
      58,
      59,
      60,
      61,
      62,
      63,
      64,
      65,
      66,
      67,
      68,
      69,
      70,
      71,
      72,
      73,
      74,
      75,
      76,
      77,
      78,
      79,
      80,
      81,
      82,
      83,
      84,
      85,
      86,
      87,
      88,
      89,
      90,
      91,
      92,
      93,
      94,
      95,
      96,
      97,
      98,
      99,
      100,
      101,
      102,
      103,
      104,
      105,
      106,
      107,
      108,
      109,
      110,
      111,
      112,
      113,
      114,
      115,
      116
    ]
  },
  "code/worker.py": {
    "sha256": "5b29c32d03708880e3c29532545c7f9d860af0be16a90e16e3105029be45ec0c",
    "bytes": 9979,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/quality.py": {
    "sha256": "411c4127263a6bb7b1b5abb369dad0f031dfc069498cc5e28c18d4a9f33b2a9f",
    "bytes": 673,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/state_checks.py": {
    "sha256": "0fd7929b44d87b2ac9fd44d2fad248d8569f2c87e229aeca4ffbfd28093ff61c",
    "bytes": 1815,
    "utf8_bom": false,
    "crlf_lines": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38
    ]
  },
  "code/memory_win.py": {
    "sha256": "4df105252a835e9abf9a858956d377bcb88742c3d4945520d5b85d072b675c99",
    "bytes": 1815,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/job_limit.py": {
    "sha256": "f31f41203dcea0f607ccc90fed2e70f9b3a37f5695289f871147dd2d28ca58d3",
    "bytes": 8452,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/reproduce.py": {
    "sha256": "190f35150c9c640c9cea0470c0510dbc327f4c9a9ef0f40bf27becc4cf2380be",
    "bytes": 28050,
    "utf8_bom": false,
    "crlf_lines": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48,
      49,
      50,
      51,
      52,
      53,
      54,
      55,
      56,
      57,
      58,
      59,
      60,
      61,
      62,
      63,
      64,
      65,
      66,
      67,
      68,
      69,
      70,
      71,
      72,
      73,
      74,
      75,
      76,
      77,
      78,
      79,
      80,
      81,
      82,
      83,
      84,
      85,
      86,
      87,
      88,
      89,
      90,
      91,
      92,
      93,
      94,
      95,
      96,
      97,
      98,
      99,
      100,
      101,
      102,
      103,
      104,
      105,
      106,
      107,
      108,
      109,
      110,
      111,
      112,
      113,
      114,
      115,
      116,
      117,
      118,
      119,
      120,
      121,
      122,
      123,
      124,
      125,
      126,
      127,
      128,
      129,
      130,
      131,
      132,
      133,
      134,
      135,
      136,
      137,
      138,
      139,
      140,
      141,
      142,
      143,
      144,
      145,
      146,
      147,
      148,
      149,
      150,
      151,
      152,
      153,
      154,
      155,
      156,
      157,
      158,
      159,
      160,
      161,
      162,
      163,
      164,
      165,
      166,
      167,
      168,
      169,
      170,
      171,
      172,
      173,
      174,
      175,
      176,
      177,
      178,
      179,
      180,
      181,
      182,
      183,
      184,
      185,
      186,
      187,
      188,
      189,
      190,
      191,
      192,
      193,
      194,
      195,
      196,
      197,
      198,
      199,
      200,
      201,
      202,
      203,
      204,
      205,
      206,
      207,
      208,
      209,
      210,
      211,
      212,
      213,
      214,
      215,
      216,
      217,
      218,
      219,
      220,
      221,
      222,
      223,
      224,
      225,
      226,
      227,
      228,
      229,
      230,
      231,
      232,
      233,
      234,
      235,
      236,
      237,
      238,
      239,
      240,
      241,
      242,
      243,
      244,
      245,
      246,
      247,
      248,
      249,
      250,
      251,
      252,
      253,
      254,
      255,
      256,
      257,
      258,
      259,
      260,
      261,
      262,
      263,
      264,
      265,
      266,
      267,
      268,
      269,
      270,
      271,
      272,
      273,
      274,
      275,
      276,
      277,
      278,
      279,
      280,
      281,
      282,
      283,
      284,
      285,
      286,
      287,
      288,
      289,
      290,
      291,
      292,
      293,
      294,
      295,
      296,
      297,
      298,
      299,
      300,
      301,
      302,
      303,
      304,
      305,
      306,
      307,
      308,
      309,
      310,
      311,
      312,
      313,
      314,
      315,
      316,
      317,
      318,
      319,
      320,
      321,
      322,
      323,
      324,
      325,
      326,
      327,
      328,
      329,
      330,
      331,
      332,
      333,
      334,
      335,
      336,
      337,
      338,
      339,
      340,
      341,
      342,
      343,
      344,
      345,
      346,
      347,
      348,
      349,
      350,
      351,
      352,
      353,
      354,
      355,
      356,
      357,
      358,
      359,
      360,
      361,
      362,
      363,
      364,
      365,
      366,
      367,
      368,
      369,
      370,
      371,
      372,
      373,
      374,
      375,
      376,
      377,
      378,
      379,
      380,
      381,
      382,
      383,
      384,
      385,
      386,
      387,
      388,
      389,
      390,
      391,
      392,
      393,
      394,
      395,
      396,
      397,
      398,
      399,
      400,
      401,
      402,
      403,
      404,
      405,
      406,
      407,
      408,
      409,
      410,
      411,
      412,
      413,
      414,
      415,
      416,
      417,
      418,
      419,
      420,
      421,
      422,
      423,
      424,
      425,
      426,
      427,
      428,
      429,
      430,
      431,
      432,
      433,
      434,
      435,
      436,
      437,
      438,
      439,
      440,
      441,
      442,
      443,
      444,
      445,
      446,
      447,
      448,
      449,
      450,
      451,
      452,
      453,
      454,
      455,
      456,
      457,
      458,
      459,
      460,
      461,
      462,
      463,
      464,
      465,
      466,
      467,
      468,
      469,
      470,
      471,
      472,
      473,
      474,
      475,
      476,
      477,
      478,
      479,
      480,
      481,
      482,
      483,
      484,
      485,
      486,
      487,
      488,
      489,
      490,
      491,
      492,
      493,
      494,
      495,
      496,
      497,
      498,
      499,
      500,
      501,
      502,
      503,
      504,
      505
    ]
  },
  "code/classix/classix_t_packed_adjbits.py": {
    "sha256": "09173bad948bc4826bda083814bdff7883eeb97bccc29273a8d9dbb19dec8ff7",
    "bytes": 15784,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/classix/classix_t_reference_adjbits.py": {
    "sha256": "e7c6972a5985677ad0ce2483a12844300e5652dfcc59a52b5512afc1238ae710",
    "bytes": 15547,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/classix/LICENSE": {
    "sha256": "9d739b5199e77aef4757893620fe6a0c0c5549ffddb13f9f98100fb003c93ddb",
    "bytes": 1069,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/classix/packed_adjacency.py": {
    "sha256": "b85b5425dac6ec625f8a5ade9d4de0a26cf649c50a4f54de389322ec0d103146",
    "bytes": 10045,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/classix/packed_kernel.c": {
    "sha256": "65bfeb98b9d18026fb9e6d2ebc3be5db094adf6ab99e6012ab8017b565c6048e",
    "bytes": 2652,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/classix/packed_native.py": {
    "sha256": "fb0815c9e57299994af4b62bab6052cb4906add82a971985b139d700575e586c",
    "bytes": 4010,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "code/classix/spsubmatxvec.c": {
    "sha256": "10c53fe53a9f1ef92d3495574715190cd1c65395eff50abc238b53cac7bcc2a0",
    "bytes": 606,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/bitbirch/__init__.py": {
    "sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "bytes": 0,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/bitbirch/bitbirch.py": {
    "sha256": "ff88e988306854ff5a45edcd9e656a6ae79f398dd61a4b60ade89f61d6d33b22",
    "bytes": 38882,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/bitbirch/cluster_analysis.py": {
    "sha256": "23b8aac5ca5403ab80ee29ef8a4359a448a311929711875ca669440e7c40d0dc",
    "bytes": 6409,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/bitbirch/cluster_control.py": {
    "sha256": "d681b94d75ecea6f9d064582a31361ef609613356f5c33974be4eb8eb6eb2c62",
    "bytes": 11423,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/bitbirch/plotting_utils.py": {
    "sha256": "d62bc65dad554d61a2f594fd276ab2ae0be2c1e347c9148e58985b574d9f6812",
    "bytes": 7297,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/bitbirch/pruning.py": {
    "sha256": "9eb65eb920112a10767d516abc5ca6064eee578b5969ff1489370ef1719ce0f5",
    "bytes": 5778,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/bitbirch/refinement.py": {
    "sha256": "ba79958e0e568dccc6b18d2294ea32e59497bd3e549f8e2095058e9ad4fca5b9",
    "bytes": 3061,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/LICENSE": {
    "sha256": "6557adc4bc1d59232e27567a185dd70cf2ecca3154018b0be61ea29cf9e814e2",
    "bytes": 7633,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/README.md": {
    "sha256": "1f5c9627b1d3e81767e91eefc54e3ad56542cd145eec1bb4dff57e88634735bb",
    "bytes": 578,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/pyproject.toml": {
    "sha256": "3ea9d413d619c538fd243fc384e17d7a3a31ea31854a74fb3deabf30f859cf68",
    "bytes": 885,
    "utf8_bom": false,
    "crlf_lines": []
  },
  "vendor/bitbirch/provenance.json": {
    "sha256": "a26a11618c627a6a91a58aeb0dafe6f22d006b20a0efc8d5441d04d5354a2bb1",
    "bytes": 3740,
    "utf8_bom": true,
    "crlf_lines": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48,
      49,
      50,
      51,
      52,
      53,
      54,
      55,
      56,
      57
    ]
  }
}''')

def source_bytes(name):
    text = SOURCES[name]
    metadata = SOURCE_METADATA[name]
    crlf_lines = set(metadata["crlf_lines"])
    lines = text.splitlines(keepends=True)
    for i in crlf_lines:
        assert lines[i].endswith("\n") and not lines[i].endswith("\r\n")
        lines[i] = lines[i][:-1] + "\r\n"
    payload = (b"\xef\xbb\xbf" if metadata["utf8_bom"] else b"") + "".join(lines).encode("utf-8")
    assert len(payload) == metadata["bytes"]
    assert hashlib.sha256(payload).hexdigest() == metadata["sha256"], name
    return payload

assert set(SOURCES) == set(SOURCE_METADATA)
for name in SOURCES:
    source_bytes(name)
PLAN = json.loads(SOURCES["plan.json"])
assert len(PLAN["cases"]) == 6 and PLAN["unique_datasets"] == 1
assert all(c["rows"] == 10000 and c["features"] == 1024 for c in PLAN["cases"])
print("Verified embedded sources/design:", len(SOURCES))


## Materialize sources and prepare Python

An existing work directory can be reused only when its materialized source and design bytes match this notebook. Otherwise choose a fresh `WORK_DIR`. Set `INSTALL_DEPENDENCIES=False` only when the selected interpreter already has the pinned dependencies. Source edits or package changes require a new run directory; checkpoints are never silently reused under a changed fit protocol.


In [ ]:
PINNED_REQUIREMENTS = r'''cloudpickle==3.1.2
colorama==0.4.6
contourpy==1.4.0
cycler==0.12.1
fonttools==4.65.0
joblib==1.6.0
kiwisolver==1.5.1
markdown-it-py==4.2.0
matplotlib==3.11.2
mdurl==0.1.2
narwhals==2.26.0
numpy==2.5.3
packaging==26.3
pillow==12.3.0
psutil==7.2.2
Pygments==2.21.0
pyparsing==3.3.3
python-dateutil==2.9.0.post0
rdkit==2026.3.6
rich==15.0.0
scikit-learn==1.9.1
scipy==1.18.1
six==1.17.0
threadpoolctl==3.7.0
tqdm==4.70.1
typing_extensions==4.16.0
'''

BBLEAN_WHEEL = json.loads(r'''{
  "filename": "bblean-0.10.3-cp312-cp312-win_amd64.whl",
  "url": "https://files.pythonhosted.org/packages/b7/2b/f7f088fe70597d72f3705ee050e71110ca3a56e8bb5c1f954025c6f9fac9/bblean-0.10.3-cp312-cp312-win_amd64.whl",
  "sha256": "36d10341de071024a0585020d5907d12950c93e251c03f74e92a14eee6a78fd9"
}''')

In [ ]:
assert os.name == "nt" and platform.machine().lower() in {"amd64", "x86_64"},     "The native kernels and memory protocol require Windows x64."
WORK_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_ROOT = WORK_DIR / "runtime"
RUNTIME_ROOT.mkdir(exist_ok=True)

def write_identical(path, payload):
    if path.exists():
        assert path.read_bytes() == payload, f"Source/design changed; choose a new WORK_DIR: {path}"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(payload)

for name in SOURCES:
    write_identical(RUNTIME_ROOT / name, source_bytes(name))
write_identical(RUNTIME_ROOT / "source_manifest.json", (json.dumps(SOURCE_METADATA, indent=2) + "\n").encode())
write_identical(RUNTIME_ROOT / "requirements.txt", PINNED_REQUIREMENTS.encode())

if PYTHON_FOR_RUNS is None:
    venv_dir = WORK_DIR / ".venv"
    python = venv_dir / "Scripts/python.exe"
    if not python.exists():
        base = [sys.executable] if sys.version_info[:2] == (3, 12) else ["py", "-3.12"]
        subprocess.run(base + ["-m", "venv", str(venv_dir)], check=True, creationflags=subprocess.CREATE_NO_WINDOW)
else:
    python = Path(PYTHON_FOR_RUNS).expanduser().resolve()
subprocess.run([str(python), "-c", "import sys; assert sys.version_info[:2] == (3,12) and sys.maxsize > 2**32"],
               check=True, creationflags=subprocess.CREATE_NO_WINDOW)
if INSTALL_DEPENDENCIES:
    import urllib.request
    subprocess.run([str(python), "-m", "pip", "install", "-r", str(RUNTIME_ROOT / "requirements.txt")],
                   check=True, creationflags=subprocess.CREATE_NO_WINDOW)
    download_dir = WORK_DIR / "downloads"
    download_dir.mkdir(exist_ok=True)
    wheel = download_dir / BBLEAN_WHEEL["filename"]
    if not wheel.is_file():
        urllib.request.urlretrieve(BBLEAN_WHEEL["url"], wheel)
    assert hashlib.sha256(wheel.read_bytes()).hexdigest() == BBLEAN_WHEEL["sha256"], "BBLean wheel hash mismatch"
    subprocess.run([str(python), "-m", "pip", "install", "--no-deps", str(wheel)],
                   check=True, creationflags=subprocess.CREATE_NO_WINDOW)
print("Runtime sources:", RUNTIME_ROOT)
print("Python:", python)


## Build the native CLASSIX kernels

The following cell compiles the included C sources locally with MSVC x64. It records their hashes, build flags and resulting DLL hashes and reuses only a matching previous build. Compilation and environment preparation are separate from fitting.


In [ ]:
import shutil
native_dir = RUNTIME_ROOT / "code/classix"
build_dir = WORK_DIR / "native_build"
build_dir.mkdir(exist_ok=True)
kernel_sources = [native_dir / "packed_kernel.c", native_dir / "spsubmatxvec.c"]
kernel_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in kernel_sources}
build_record = build_dir / "build.json"
if build_record.exists():
    build_info = json.loads(build_record.read_text())
    assert build_info["source_sha256"] == kernel_hashes
    for name, expected in build_info["dll_sha256"].items():
        assert hashlib.sha256((native_dir/name).read_bytes()).hexdigest() == expected
else:
    vswhere = Path(os.environ.get("ProgramFiles(x86)", "C:/Program Files (x86)")) / "Microsoft Visual Studio/Installer/vswhere.exe"
    assert vswhere.is_file(), "Install Visual Studio C++ Build Tools and Windows SDK first."
    install = subprocess.check_output([str(vswhere), "-latest", "-products", "*", "-requires",
        "Microsoft.VisualStudio.Component.VC.Tools.x86.x64", "-property", "installationPath"], text=True).strip()
    assert install, "The x64 C++ toolset was not found."
    vcvars = Path(install) / "VC/Auxiliary/Build/vcvars64.bat"
    assert vcvars.is_file()
    # A fixed local build script avoids carrying machine-specific binaries.
    commands = ["@echo off", f'call "{vcvars}"', "if errorlevel 1 exit /b 1",
        f'cl /nologo /O2 /Oi /MT /LD /W4 /TC "{kernel_sources[0]}" /Fopacked_kernel.obj /Fepacked_kernel.dll /link /INCREMENTAL:NO',
        "if errorlevel 1 exit /b 1",
        f'cl /nologo /O2 /Oi /MT /LD /W4 /TC "{kernel_sources[1]}" /Fooriginal_spmv.obj /Feoriginal_spmv.dll /link /EXPORT:multiply /INCREMENTAL:NO',
        "if errorlevel 1 exit /b 1"]
    script = build_dir / "build.cmd"
    script.write_text("\n".join(commands) + "\n", encoding="utf-8")
    with (build_dir/"build.log").open("w", encoding="utf-8") as log:
        subprocess.run(["cmd.exe", "/d", "/c", str(script)], cwd=build_dir, stdout=log,
                       stderr=subprocess.STDOUT, check=True, creationflags=subprocess.CREATE_NO_WINDOW)
    for name in ("packed_kernel.dll", "original_spmv.dll"):
        shutil.copy2(build_dir/name, native_dir/name)
    build_info = {"source_sha256": kernel_hashes,
        "dll_sha256": {name: hashlib.sha256((native_dir/name).read_bytes()).hexdigest()
                       for name in ("packed_kernel.dll", "original_spmv.dll")},
        "visual_studio_installation": install, "flags": "/O2 /Oi /MT /LD /W4 /TC",
        "compiler_output": "build.log"}
    build_record.write_text(json.dumps(build_info, indent=2) + "\n")
print("Compiled native kernels:", build_info["dll_sha256"])

## Run, pause or resume

`full` runs six radii across five methods. `pilot` runs the first radius on the first 256 shuffled rows for all five methods; the full data are generated before slicing, so pilot memory is diagnostic only. `check` only checks imports/design.

Create a file named `PAUSE` inside the printed run directory to stop after the current fit, or `STOP_CURRENT` to interrupt the current attempt and leave it retriable. Remove the control file yourself and execute this cell again to resume. Completed assignments are reused only after protocol and partition checks. Pilot/check runs cannot overwrite the full-study figures. A full re-run under identical source, environment and plan resumes completed fits rather than fitting them again.


In [ ]:
RUN_DIR = RUNTIME_ROOT / "runs" / ("pilot" if RUN_MODE == "pilot" else "reproduction")
command = [str(python), "-B", str(RUNTIME_ROOT / "code/reproduce.py"),
           "--check" if RUN_MODE == "check" else "--run", "--output-dir", str(RUN_DIR)]
if RUN_MODE == "pilot":
    command += ["--cases", PLAN["cases"][0]["case_id"], "--pilot-rows", str(PILOT_ROWS)]
print("Run directory:", RUN_DIR)
print("Command:", subprocess.list2cmdline(command))
subprocess.run(command, cwd=RUNTIME_ROOT, check=True, creationflags=subprocess.CREATE_NO_WINDOW)


## Score saved assignments and render ARI/AMI

This cell does not refit estimators. It independently recomputes ARI and AMI from newly generated ground truth and verified saved labels. It refuses to publish a full-study plot until all 30 fits have completed successfully; diagnostic pilot/check runs skip this figure. The plot uses plain method names, an upper-left legend, and no footer labels; data and parameter descriptions belong in the paper/caption.


In [ ]:
if RUN_MODE != "full":
    print("Diagnostic run finished; full-study ARI/AMI figure requires RUN_MODE='full'.")
else:
    import shutil
    assert not (RUN_DIR / "run.lock").exists(), "Wait for fit workers to finish."
    SCORE_DIR = RUNTIME_ROOT / "derived" / "radius_sensitivity"
    subprocess.run([str(python), "-B", str(RUNTIME_ROOT / "score_and_plot.py"),
                    "--run-dir", str(RUN_DIR), "--output-dir", str(SCORE_DIR)],
                   cwd=RUNTIME_ROOT, check=True, creationflags=subprocess.CREATE_NO_WINDOW)
    verification = json.loads((SCORE_DIR / "verification.json").read_text(encoding="utf-8"))
    assert verification["status"] == "passed" and verification["full_method_settings"] == 30
    assert verification["all_methods_complete"] and verification["scored_rows_per_fit"] == 10000
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    for artifact in verification["artifacts"]:
        generated = Path(artifact["path"])
        assert generated.parent.resolve() == (SCORE_DIR / "plots").resolve()
        assert hashlib.sha256(generated.read_bytes()).hexdigest() == artifact["sha256"]
        destination = PLOT_DIR / generated.name
        shutil.copy2(generated, destination)
        assert hashlib.sha256(destination.read_bytes()).hexdigest() == artifact["sha256"]
    print((SCORE_DIR / "scores.md").read_text(encoding="utf-8"))
    print("Full-study ARI/AMI plots:", PLOT_DIR / "radius_ari_ami.pdf")
    print("Generated scores, input checks and figure hashes:", SCORE_DIR)
    try:
        from IPython.display import Image, display
    except ImportError:
        pass
    else:
        display(Image(filename=str(PLOT_DIR / "radius_ari_ami.png")))


## Provenance and interpretation

The manuscript was preserved before this study at local Git commit `fd035db8694d9aec1e0c330588ddcfc53938079e`. The notebook's hashes identify the subsequent sensitivity implementation. Source-only reproduction creates all inputs and observations afresh and needs no files from that Git checkpoint or from scratch. Existing published figures are historical outputs and are not used to calculate new curves.

The six radius settings are tested on one fixed synthetic dataset and one row order. They describe parameter sensitivity in that controlled example, not variability over seeds or orderings. Matching a Tanimoto distance cutoff to `1 - similarity` aligns the numeric parameter scale; methods retain their own grouping/merging definitions. CLASSIX's `mergeScale` changes with radius, as requested.
